In [0]:
import json
from pyspark.sql import functions as F

CENTROS = ["D063", "D004", "D005", "D015", "D040", "D060", "D090"]
ANOS = ["2025", "2026"]
SOURCE_CANDIDATES = [
    "stage.analytics_madeira.tb_sim_tableau_full_madeira",
    "stage.analytics_madeira.tb_sim_tableau_full_madeira_hana",
    "stage.analytics_bkp.analytics_madeira_tb_sim_tableau_full_madeira",
]
RECEITA_COLS = [
    "vl_rec_liquida_brl_gerencial",
    "vl_rec_liquida_brl",
    "vl_rec_liquida",
]


def load_base():
    last_error = None
    for source in SOURCE_CANDIDATES:
        try:
            df = (
                spark.table(source)
                .filter(F.col("tipo") == "NF")
                .filter(F.col("centro").isin(CENTROS))
                .filter(F.col("ano_competencia").isin(ANOS))
            )
            df.limit(1).collect()
            return df, source, None
        except Exception as e:
            last_error = f"{source}: {str(e).splitlines()[0]}"
    return None, None, last_error


base, base_source, access_error = load_base()
receita_col = None
if base is not None:
    for candidate in RECEITA_COLS:
        if candidate in base.columns:
            receita_col = candidate
            break


def txt(v):
    return (v or "N/A").replace('"', "'")


if base is not None:
    vol_rows = (
        base.groupBy("centro", "ano_competencia", "mes_competencia", "mercado")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .collect()
    )
    inc_rows = (
        base.withColumn("ic", F.coalesce(F.col("incoterm"), F.lit("N/A")))
        .groupBy("centro", "ano_competencia", "mes_competencia", "ic")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .collect()
    )
    uf_rows = (
        base.filter(F.col("cli_uf").isNotNull())
        .filter(~F.col("cli_uf").isin("EX", "06", "CB"))
        .groupBy("centro", "ano_competencia", "mes_competencia", "cli_uf")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .collect()
    )
    cli_vol_rows = (
        base.filter(F.col("cli_razao_social").isNotNull())
        .groupBy("centro", "ano_competencia", "mes_competencia", "cli_razao_social")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .filter(F.col("v") > 0)
        .collect()
    )
    lin_rows = (
        base.withColumn(
            "mat2",
            F.coalesce(F.regexp_replace(F.trim(F.col("prd_dc_linha")), r"\s+", " "), F.lit("N/A")),
        )
        .groupBy("centro", "ano_competencia", "mes_competencia", "mat2")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .filter(F.col("v") != 0)
        .collect()
    )
    fat_dia_rows = (
        base.filter(F.col("dt_faturamento").isNotNull())
        .groupBy("centro", "ano_competencia", "mes_competencia", "dt_faturamento")
        .agg(F.sum("qt_vol_m3_liquido").alias("v"))
        .collect()
    )
    rec_rows = (
        base.groupBy("centro", "ano_competencia", "mes_competencia")
        .agg(F.sum(F.coalesce(F.col(receita_col), F.lit(0.0))).alias("v"))
        .collect()
        if receita_col
        else []
    )
else:
    vol_rows = []
    inc_rows = []
    uf_rows = []
    cli_vol_rows = []
    lin_rows = []
    fat_dia_rows = []
    rec_rows = []

payload = json.dumps(
    {
        "vol": [[r.centro, r.ano_competencia, r.mes_competencia, r.mercado, float(r.v or 0)] for r in vol_rows],
        "inc": [[r.centro, r.ano_competencia, r.mes_competencia, r.ic, float(r.v or 0)] for r in inc_rows],
        "uf": [[r.centro, r.ano_competencia, r.mes_competencia, r.cli_uf, float(r.v or 0)] for r in uf_rows],
        "cliVol": [[r.centro, r.ano_competencia, r.mes_competencia, txt(r.cli_razao_social), float(r.v or 0)] for r in cli_vol_rows],
        "lin": [[r.centro, r.ano_competencia, r.mes_competencia, r.mat2, float(r.v or 0)] for r in lin_rows],
        "fatDia": [[r.centro, r.ano_competencia, r.mes_competencia, r.dt_faturamento.isoformat(), float(r.v or 0)] for r in fat_dia_rows],
        "recLiq": [[r.centro, r.ano_competencia, r.mes_competencia, float(r.v or 0)] for r in rec_rows],
        "centros": CENTROS,
        "anos": ANOS,
        "source": base_source or "N/A",
        "accessError": access_error or "",
        "receitaCol": receita_col or "",
    },
    ensure_ascii=False,
)

html = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <style>
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Segoe UI',system-ui,-apple-system,sans-serif;background:#000;color:#e8e8e8}
    .hdr{background:linear-gradient(180deg,#0a0a0a 0%,#000 100%);border-bottom:1px solid #1a1a1a;padding:18px 32px 12px;display:flex;flex-direction:column;align-items:center;gap:6px}
    .hdr h1{text-align:center;font-size:32px;font-weight:900;color:#fff;letter-spacing:4px;font-family:'Segoe UI',system-ui,sans-serif}
    .hdr .logo{}
    .flt{background:#050505;border-bottom:1px solid #1a1a1a;padding:12px 32px;display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;justify-content:space-between}
    .fg{display:flex;flex-direction:column;gap:4px;align-items:center}
    .fg label{font-size:9px;color:#888;text-transform:uppercase;letter-spacing:1px;font-weight:600}
    .fg select{background:#111;color:#ccc;border:1px solid #2a2a2a;padding:6px 10px;border-radius:6px;font-size:12px;min-width:130px;transition:border-color .2s}
    .fg select:focus{border-color:#5b8def;outline:none}
    .chip-group{display:flex;flex-wrap:wrap;gap:6px;justify-content:center}
    .chip{background:#1a1a1a;color:#aaa;border:1px solid #333;padding:5px 12px;border-radius:16px;font-size:11px;cursor:pointer;transition:all .2s;font-weight:600;user-select:none}
    .chip:hover{background:#2a2a2a;color:#ddd}
    .chip.on{background:#fff;color:#111;border-color:#fff;font-weight:700}
    .d{padding:20px 32px 30px;display:grid;gap:18px;max-width:1800px;margin:0 auto}
    .row{display:grid;gap:18px}
    .r2{grid-template-columns:1fr 1fr}
    .r3{grid-template-columns:1fr 1fr 1fr}
    .rm{grid-template-columns:1fr 1fr}
    .c{background:#2d2d2d;border:1px solid #3a3a3a;border-radius:14px;padding:18px 20px;overflow:hidden;box-shadow:0 6px 18px rgba(0,0,0,.25)}
    .c h3{font-size:16px;color:#f2f2f2;text-transform:uppercase;letter-spacing:1.1px;margin-bottom:14px;border-bottom:1px solid #1f1f1f;padding-bottom:10px;text-align:center;font-weight:800}
    .kpi{text-align:center;padding:18px 0}
    .kpi .v{font-size:34px;font-weight:800;color:#fff;letter-spacing:-0.6px}
    .bc{display:flex;flex-direction:column;gap:12px}
    .br{display:flex;align-items:flex-start;gap:12px}
    .bl{font-size:12px;color:#d2d2d2;min-width:260px;width:260px;text-align:left;padding-right:0;white-space:normal;overflow:visible;text-overflow:clip;font-weight:600;line-height:1.35}
    .bw{flex:1;background:#1a1a1a;border-radius:6px;height:28px;overflow:hidden;margin-top:2px}
    .bf{height:100%;background:linear-gradient(90deg,#9e9e9e,#d6d6d6);border-radius:6px;transition:width .5s ease}
    .bv{font-size:12px;color:#efefef;min-width:125px;padding-left:8px;white-space:nowrap;font-weight:700;line-height:28px}
    .tc{overflow:auto;max-height:460px}
    table{width:100%;border-collapse:collapse;font-size:12px}
    th{background:#111;color:#bfbfbf;padding:10px 12px;text-align:left;position:sticky;top:0;border-bottom:1px solid #2a2a2a;text-transform:uppercase;font-size:11px;letter-spacing:.6px;font-weight:800}
    td{padding:9px 12px;border-bottom:1px solid #151515;color:#ddd}
    tr:hover td{background:#111;color:#fff}
    .pc{display:flex;align-items:center;justify-content:center;gap:20px;padding:8px 0}
    .pl{display:flex;flex-direction:column;gap:6px}
    .pi{display:flex;align-items:center;gap:6px;font-size:11px;color:#ccc}
    .pp{width:12px;height:12px;border-radius:3px}
    .nd{text-align:center;color:#555;padding:28px;font-size:12px;font-style:italic}
    #mapBR{width:100%;background:#050505;border-radius:8px;padding:10px}
    .map-legend{display:flex;gap:14px;margin-top:10px;justify-content:center;flex-wrap:wrap}
    .map-legend span{display:flex;align-items:center;gap:5px;font-size:10px;color:#999}
    .map-legend i{width:14px;height:14px;border-radius:3px;display:inline-block}
    .trend-wrap{display:flex;flex-direction:column;gap:14px}
    .trend-head{display:flex;justify-content:space-between;align-items:flex-end;gap:14px;flex-wrap:wrap;padding:6px 2px 0}
    .trend-meta{display:flex;flex-direction:column;gap:6px}
    .trend-label{font-size:12px;text-transform:uppercase;letter-spacing:1px;color:#8f8f8f;font-weight:700}
    .trend-value{font-size:34px;font-weight:800;color:#fff;letter-spacing:-0.5px}
    .trend-sub{font-size:13px;color:#9f9f9f}
    .trend-note{font-size:12px;color:#8d8d8d;text-align:right}
    .trend-svg{width:100%;height:auto;display:block}
    .ana-ref{font-size:13px;color:#9f9f9f;margin-bottom:16px;text-align:center}
    .ana-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}
    .ana-item{background:linear-gradient(180deg,#383838 0%,#2d2d2d 100%);border:1px solid #4a4a4a;border-radius:12px;padding:18px 18px;min-height:170px}
    .ana-lbl{font-size:12px;color:#a3a3a3;text-transform:uppercase;letter-spacing:.9px;margin-bottom:10px;font-weight:700}
    .ana-val{font-size:30px;font-weight:800;color:#fff;line-height:1.2}
    .ana-sub{font-size:13px;color:#a0a0a0;margin-top:8px;line-height:1.55}
    .cmp-wrap{display:flex;flex-direction:column;gap:14px}
    .cmp-legend{display:flex;align-items:center;justify-content:flex-end;gap:18px;flex-wrap:wrap}
    .cmp-legend-item{display:flex;align-items:center;gap:8px;font-size:12px;color:#d7d7d7;font-weight:700}
    .cmp-dot{width:12px;height:12px;border-radius:999px;display:inline-block}
    .pos{color:#78d08b}
    .neg{color:#ff7d7d}
    .neu{color:#d8d8d8}
    @media(max-width:1200px){.ana-grid{grid-template-columns:1fr}}
    @media(max-width:900px){.r2,.r3,.rm{grid-template-columns:1fr}.bl{width:100%;min-width:0}.br{flex-direction:column}.bv{line-height:1.4;padding-left:0}.cmp-legend{justify-content:flex-start}}
  </style>
</head>
<body>
  <div class="hdr">
    <h1>Relatório Faturamento - Unidade Painéis</h1>
    <div class="logo" style="background:#fff;padding:4px 14px;border-radius:6px;display:flex;align-items:center;justify-content:center">
      <img src="https://logodownload.org/wp-content/uploads/2022/05/dexco-logo-0.png" alt="DEXCO" style="height:22px;width:auto;max-width:120px;display:block;object-fit:contain" onerror="this.outerHTML='<svg viewBox=&quot;0 0 130 28&quot; height=&quot;22&quot;><text x=&quot;65&quot; y=&quot;22&quot; font-family=&quot;Arial Black,sans-serif&quot; font-size=&quot;24&quot; font-weight=&quot;900&quot; fill=&quot;%23000&quot; letter-spacing=&quot;5&quot; text-anchor=&quot;middle&quot;>DEXCO</text></svg>'">
    </div>
  </div>

  <div class="flt">
    <div class="fg"><label>Centro</label><div class="chip-group" id="fC"><span class="chip on" data-v="ALL" onclick="tc(this,'fC')">Todos</span><span class="chip" data-v="D063" onclick="tc(this,'fC')">D063</span><span class="chip" data-v="D004" onclick="tc(this,'fC')">D004</span><span class="chip" data-v="D005" onclick="tc(this,'fC')">D005</span><span class="chip" data-v="D015" onclick="tc(this,'fC')">D015</span><span class="chip" data-v="D040" onclick="tc(this,'fC')">D040</span><span class="chip" data-v="D060" onclick="tc(this,'fC')">D060</span><span class="chip" data-v="D090" onclick="tc(this,'fC')">D090</span></div></div>
    <div class="fg"><label>Mês</label><div class="chip-group" id="fM"><span class="chip on" data-v="ALL" onclick="tc(this,'fM')">Todos</span><span class="chip" data-v="JAN" onclick="tc(this,'fM')">Jan</span><span class="chip" data-v="FEV" onclick="tc(this,'fM')">Fev</span><span class="chip" data-v="MAR" onclick="tc(this,'fM')">Mar</span><span class="chip" data-v="ABR" onclick="tc(this,'fM')">Abr</span><span class="chip" data-v="MAI" onclick="tc(this,'fM')">Mai</span><span class="chip" data-v="JUN" onclick="tc(this,'fM')">Jun</span><span class="chip" data-v="JUL" onclick="tc(this,'fM')">Jul</span><span class="chip" data-v="AGO" onclick="tc(this,'fM')">Ago</span><span class="chip" data-v="SET" onclick="tc(this,'fM')">Set</span><span class="chip" data-v="OUT" onclick="tc(this,'fM')">Out</span><span class="chip" data-v="NOV" onclick="tc(this,'fM')">Nov</span><span class="chip" data-v="DEZ" onclick="tc(this,'fM')">Dez</span></div></div>
    <div class="fg"><label>Ano</label><div class="chip-group" id="fA"><span class="chip" data-v="ALL" onclick="tc(this,'fA')">Todos</span><span class="chip on" data-v="2026" onclick="tc(this,'fA')">2026</span><span class="chip" data-v="2025" onclick="tc(this,'fA')">2025</span></div></div>
  </div>

  <div class="d">
    <div class="row r3">
      <div class="c"><h3>Volume Total Faturado</h3><div class="kpi"><div class="v" id="k1">— m³</div></div></div>
      <div class="c"><h3>Volume Mercado Interno (MI)</h3><div class="kpi"><div class="v" id="k2">— m³</div></div></div>
      <div class="c"><h3>Volume Mercado Externo (ME)</h3><div class="kpi"><div class="v" id="k3">— m³</div></div></div>
    </div>

    <div class="row r2">
      <div class="c"><h3>Volume por Canal (MI / ME)</h3><div id="cC"></div></div>
      <div class="c"><h3>Volume por Tipo de Frete</h3><div id="cI"></div></div>
    </div>

    <div class="row rm">
      <div class="c">
        <h3>Volume Faturado por Estado (m³)</h3>
        <div id="mapBR"></div>
        <div class="map-legend">
          <span><i style="background:#2d6a4f"></i>Norte</span>
          <span><i style="background:#b5654a"></i>Nordeste</span>
          <span><i style="background:#3a6b7e"></i>Centro-Oeste</span>
          <span><i style="background:#5a8fb5"></i>Sudeste</span>
          <span><i style="background:#7a8590"></i>Sul</span>
        </div>

      </div>
      <div class="c"><h3>Top 10 Estados por Volume</h3><div id="cU"></div></div>
    </div>

    <div class="row r2">
      <div class="c"><h3>Top 10 Clientes por Volume (m³)</h3><div id="cK"></div></div>
      <div class="c"><h3>Volume por Linha de Produto</h3><div id="cL"></div></div>
    </div>

    <div class="row">
      <div class="c"><h3>Volume Mensal (m³)</h3><div id="cE"></div></div>
    </div>

    <div class="row">
      <div class="c"><h3>Receita Líquida Mensal (R$)</h3><div id="cR"></div></div>
    </div>

    <div class="row">
      <div class="c"><h3>Dados Analíticos de Faturamento</h3><div id="cA"></div></div>
    </div>

    <div class="row">
      <div class="c"><h3>Comparativo de Volume por Ano</h3><div id="cY"></div></div>
    </div>

    <div class="row">
      <div class="c"><h3>Faturamento Diário</h3><div id="cD"></div></div>
    </div>
  </div>

  <script>
    const DATA = __PAYLOAD__;
    const DV = DATA.vol, DI = DATA.inc, DU = DATA.uf, DC = DATA.cliVol, DL = DATA.lin, DFD = DATA.fatDia, DRL = DATA.recLiq;
    const MO = ["JAN","FEV","MAR","ABR","MAI","JUN","JUL","AGO","SET","OUT","NOV","DEZ"];
    const MO_FULL = {JAN:"Janeiro",FEV:"Fevereiro",MAR:"Março",ABR:"Abril",MAI:"Maio",JUN:"Junho",JUL:"Julho",AGO:"Agosto",SET:"Setembro",OUT:"Outubro",NOV:"Novembro",DEZ:"Dezembro"};
    const RC = {Norte:"#2d6a4f",Nordeste:"#b5654a","Centro-Oeste":"#3a6b7e",Sudeste:"#5a8fb5",Sul:"#7a8590"};
    const UR = {AC:"Norte",AM:"Norte",AP:"Norte",PA:"Norte",RO:"Norte",RR:"Norte",TO:"Norte",AL:"Nordeste",BA:"Nordeste",CE:"Nordeste",MA:"Nordeste",PB:"Nordeste",PE:"Nordeste",PI:"Nordeste",RN:"Nordeste",SE:"Nordeste",DF:"Centro-Oeste",GO:"Centro-Oeste",MS:"Centro-Oeste",MT:"Centro-Oeste",ES:"Sudeste",MG:"Sudeste",RJ:"Sudeste",SP:"Sudeste",PR:"Sul",RS:"Sul",SC:"Sul"};
    const UN = {AC:"Acre",AL:"Alagoas",AM:"Amazonas",AP:"Amapá",BA:"Bahia",CE:"Ceará",DF:"Distrito Federal",ES:"Espírito Santo",GO:"Goiás",MA:"Maranhão",MG:"Minas Gerais",MS:"Mato Grosso do Sul",MT:"Mato Grosso",PA:"Pará",PB:"Paraíba",PE:"Pernambuco",PI:"Piauí",PR:"Paraná",RJ:"Rio de Janeiro",RN:"Rio Grande do Norte",RO:"Rondônia",RR:"Roraima",RS:"Rio Grande do Sul",SC:"Santa Catarina",SE:"Sergipe",SP:"São Paulo",TO:"Tocantins"};
    const UL = {AC:[-9.02,-70.81],AL:[-9.57,-36.78],AM:[-3.47,-65.10],AP:[1.41,-51.77],BA:[-12.97,-41.68],CE:[-5.20,-39.53],DF:[-15.83,-47.86],ES:[-19.19,-40.34],GO:[-15.98,-49.86],MA:[-5.42,-45.44],MG:[-18.10,-44.38],MS:[-20.51,-54.54],MT:[-12.64,-55.42],PA:[-3.79,-52.48],PB:[-7.24,-36.72],PE:[-8.28,-35.97],PI:[-6.60,-42.28],PR:[-24.89,-51.55],RJ:[-22.25,-42.66],RN:[-5.81,-36.59],RO:[-10.83,-63.34],RR:[1.99,-61.33],RS:[-30.17,-53.50],SC:[-27.45,-50.95],SE:[-10.57,-37.45],SP:[-22.19,-48.79],TO:[-10.25,-48.25]};
    const NM = {"Acre":"AC","Alagoas":"AL","Amazonas":"AM","Amapá":"AP","Bahia":"BA","Ceará":"CE","Distrito Federal":"DF","Espírito Santo":"ES","Goiás":"GO","Maranhão":"MA","Minas Gerais":"MG","Mato Grosso do Sul":"MS","Mato Grosso":"MT","Pará":"PA","Paraíba":"PB","Pernambuco":"PE","Piauí":"PI","Paraná":"PR","Rio de Janeiro":"RJ","Rio Grande do Norte":"RN","Rondônia":"RO","Roraima":"RR","Rio Grande do Sul":"RS","Santa Catarina":"SC","Sergipe":"SE","São Paulo":"SP","Tocantins":"TO"};

    function ff(n){ return (n || 0).toLocaleString('pt-BR',{minimumFractionDigits:2,maximumFractionDigits:2}) + ' m³'; }
    function fv(n){ return (n || 0).toLocaleString('pt-BR',{minimumFractionDigits:2,maximumFractionDigits:2}); }
    function fm(n){ return (n || 0).toLocaleString('pt-BR',{style:'currency',currency:'BRL',minimumFractionDigits:2,maximumFractionDigits:2}); }
    function fp(n){ return (n >= 0 ? '+' : '') + (n || 0).toLocaleString('pt-BR',{minimumFractionDigits:1,maximumFractionDigits:1}) + '%'; }
    function fd(iso){ const p = iso.split('-'); return p.length === 3 ? `${p[2]}/${p[1]}/${p[0]}` : iso; }
    function emptyState(msg){ return `<div class="nd">${msg || 'Sem dados'}</div>`; }
    function tc(el,gid){
      const grp=document.getElementById(gid);
      if(el.dataset.v==='ALL'){
        grp.querySelectorAll('.chip').forEach(c=>c.classList.remove('on'));
        el.classList.add('on');
      } else {
        const ac=grp.querySelector('.chip[data-v="ALL"]');
        if(ac) ac.classList.remove('on');
        el.classList.toggle('on');
        if(![...grp.querySelectorAll('.chip:not([data-v="ALL"])')].some(c=>c.classList.contains('on'))){
          const ac2=grp.querySelector('.chip[data-v="ALL"]'); if(ac2) ac2.classList.add('on');
        }
      }
      go();
    }
    function gChips(gid){
      const grp=document.getElementById(gid);
      const ac=grp.querySelector('.chip[data-v="ALL"]');
      if(ac&&ac.classList.contains('on')) return null;
      return [...grp.querySelectorAll('.chip.on')].map(c=>c.dataset.v);
    }
    function gf(){
      const cc=gChips('fC'),aa=gChips('fA'),mm=gChips('fM');
      return {c:cc||DATA.centros, a:aa||DATA.anos, m:mm||MO};
    }

    function bar(id, items, mx, formatter){
      const e = document.getElementById(id);
      const fmt = formatter || ff;
      if(!items.length){ e.innerHTML = emptyState('Sem dados'); return; }
      const maxVal = mx || Math.max(...items.map(i => i[1]), 0);
      e.innerHTML = '<div class="bc">' + items.map(i => {
        const label = String(i[0] || 'N/A');
        const p = maxVal > 0 ? (i[1] / maxVal * 100) : 0;
        return '<div class="br"><div class="bl" title="' + label + '">' + label + '</div><div class="bw"><div class="bf" style="width:' + p + '%"></div></div><div class="bv">' + fmt(i[1]) + '</div></div>';
      }).join('') + '</div>';
    }

    function vbar(id,items,formatter){
      const e=document.getElementById(id);
      const fmt=formatter||ff;
      if(!items.length){e.innerHTML=emptyState('Sem dados');return;}
      const W=960,H=340,pt=28,pr=14,pb=56,pl=14;
      const mx=Math.max(...items.map(i=>i[1]),1);
      const gap=(W-pl-pr)/items.length;
      const bw=Math.min(54,gap*0.68);
      const grid=[0.25,0.5,0.75,1].map(fr=>{const y=pt+(H-pt-pb)*(1-fr);return `<line x1="${pl}" y1="${y.toFixed(1)}" x2="${W-pr}" y2="${y.toFixed(1)}" stroke="rgba(255,255,255,.06)" stroke-dasharray="4 6"/>`;}).join('');
      const bars=items.map((it,i)=>{
        const x=pl+gap*i+(gap-bw)/2;
        const bh=(it[1]/mx)*(H-pt-pb);
        const y=H-pb-bh;
        const lbl=String(it[0]||'N/A');
        return `<rect x="${x.toFixed(1)}" y="${y.toFixed(1)}" width="${bw.toFixed(1)}" height="${Math.max(bh,1).toFixed(1)}" fill="url(#vbG)" rx="4"/><text x="${(x+bw/2).toFixed(1)}" y="${(y-7).toFixed(1)}" fill="#d4d4d4" font-size="9" font-weight="700" text-anchor="middle">${fmt(it[1])}</text><text x="${(x+bw/2).toFixed(1)}" y="${(H-pb+18).toFixed(1)}" fill="#bbb" font-size="11" font-weight="700" text-anchor="middle">${lbl}</text>`;
      }).join('');
      e.innerHTML=`<svg class="trend-svg" viewBox="0 0 ${W} ${H}" preserveAspectRatio="none"><defs><linearGradient id="vbG" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#d6d6d6"/><stop offset="100%" stop-color="#9e9e9e"/></linearGradient></defs>${grid}${bars}</svg>`;
    }

    function line(id, items, cfg){
      const e = document.getElementById(id);
      if(!items.length){ e.innerHTML = emptyState('Sem dados'); return; }
      const config = cfg || {};
      const valueFormatter = config.valueFormatter || ff;
      const width = 960, height = config.height || 360;
      const pad = {t:config.showValues ? 48 : 28,r:56,b:64,l:56};
      const maxValue = Math.max(...items.map(i => i.value), 1);
      const span = Math.max(items.length - 1, 1);
      const pts = items.map((d, i) => {
        const x = items.length === 1 ? width / 2 : pad.l + ((width - pad.l - pad.r) * i / span);
        const y = pad.t + ((height - pad.t - pad.b) * (1 - (d.value / maxValue)));
        return {...d, x, y};
      });
      const linePath = pts.map((p, i) => `${i === 0 ? 'M' : 'L'} ${p.x.toFixed(1)} ${p.y.toFixed(1)}`).join(' ');
      const areaPath = `${linePath} L ${pts[pts.length - 1].x.toFixed(1)} ${(height - pad.b).toFixed(1)} L ${pts[0].x.toFixed(1)} ${(height - pad.b).toFixed(1)} Z`;
      const grid = [0.25,0.5,0.75,1].map(fr => {
        const y = pad.t + ((height - pad.t - pad.b) * (1 - fr));
        return `<line x1="${pad.l}" y1="${y.toFixed(1)}" x2="${width - pad.r}" y2="${y.toFixed(1)}" stroke="rgba(255,255,255,.07)" stroke-dasharray="4 6"/>`;
      }).join('');
      const labels = pts.map((p, i) => {
        const anchor = i === 0 ? 'start' : (i === pts.length - 1 ? 'end' : 'middle');
        return `<text x="${p.x.toFixed(1)}" y="${height - 16}" fill="#8f8f8f" font-size="12" font-weight="700" text-anchor="${anchor}">${p.label}</text>`;
      }).join('');
      const dots = pts.map((p, i) => `<circle cx="${p.x.toFixed(1)}" cy="${p.y.toFixed(1)}" r="${i === pts.length - 1 ? 5.5 : 4}" fill="${i === pts.length - 1 ? '#ffffff' : '#b6bec8'}" stroke="#0a0a0a" stroke-width="2"/>`).join('');
      const valLabels = config.showValues ? pts.map((p, i) => {
        const anchor = i === 0 ? 'start' : (i === pts.length - 1 ? 'end' : 'middle');
        const x = i === 0 ? p.x + 6 : (i === pts.length - 1 ? p.x - 6 : p.x);
        const y = Math.max(18, p.y - 12);
        return `<text x="${x.toFixed(1)}" y="${y.toFixed(1)}" fill="#d4d4d4" font-size="9" font-weight="700" text-anchor="${anchor}">${valueFormatter(p.value)}</text>`;
      }).join('') : '';
      const latest = pts[pts.length - 1];
      const head = config.headerValue ? `
        <div class="trend-head">
          <div class="trend-meta">
            <div class="trend-label">${config.headerLabel || 'Referência'}</div>
            <div class="trend-value">${config.headerValue}</div>
            <div class="trend-sub">${config.headerSub || ''}</div>
          </div>
          <div class="trend-note">Último ponto: <b style="color:#fff">${latest.label}</b></div>
        </div>` : '';
      e.innerHTML = `
        <div class="trend-wrap">
          ${head}
          <svg class="trend-svg" viewBox="0 0 ${width} ${height}" preserveAspectRatio="none">
            ${grid}
            <path d="${areaPath}" fill="${config.fill || 'rgba(255,255,255,.05)'}"></path>
            <path d="${linePath}" fill="none" stroke="${config.stroke || '#d9d9d9'}" stroke-width="3" stroke-linecap="round" stroke-linejoin="round"></path>
            ${dots}
            ${valLabels}
            ${labels}
          </svg>
        </div>`;
    }

    function multiLine(id, series, cfg){
      const e = document.getElementById(id);
      const config = cfg || {};
      const validSeries = (series || []).filter(s => s.values && s.values.length);
      if(!validSeries.length){ e.innerHTML = emptyState('Sem dados'); return; }
      const width = 960, height = config.height || 360;
      const pad = {t:48,r:56,b:64,l:56};
      const labels = validSeries[0].values.map(v => v.label);
      const maxValue = Math.max(...validSeries.flatMap(s => s.values.map(v => v.value)), 1);
      const span = Math.max(labels.length - 1, 1);
      const grid = [0.25,0.5,0.75,1].map(fr => {
        const y = pad.t + ((height - pad.t - pad.b) * (1 - fr));
        return `<line x1="${pad.l}" y1="${y.toFixed(1)}" x2="${width - pad.r}" y2="${y.toFixed(1)}" stroke="rgba(255,255,255,.07)" stroke-dasharray="4 6"/>`;
      }).join('');
      const xForIndex = i => labels.length === 1 ? width / 2 : pad.l + ((width - pad.l - pad.r) * i / span);
      const labelMarks = labels.map((label, i) => {
        const anchor = i === 0 ? 'start' : (i === labels.length - 1 ? 'end' : 'middle');
        return `<text x="${xForIndex(i).toFixed(1)}" y="${height - 16}" fill="#8f8f8f" font-size="12" font-weight="700" text-anchor="${anchor}">${label}</text>`;
      }).join('');
      const rendered = validSeries.map((s, si) => {
        const pts = s.values.map((d, i) => {
          const x = xForIndex(i);
          const y = pad.t + ((height - pad.t - pad.b) * (1 - (d.value / maxValue)));
          return {...d, x, y};
        });
        const path = pts.map((p, i) => `${i === 0 ? 'M' : 'L'} ${p.x.toFixed(1)} ${p.y.toFixed(1)}`).join(' ');
        const dots = pts.map(p => `<circle cx="${p.x.toFixed(1)}" cy="${p.y.toFixed(1)}" r="4.5" fill="${s.color}" stroke="#0a0a0a" stroke-width="2"/>`).join('');
        const vLbl = pts.map((p, i) => {
          const anchor = i === 0 ? 'start' : (i === labels.length - 1 ? 'end' : 'middle');
          const x = i === 0 ? p.x + 8 : (i === labels.length - 1 ? p.x - 8 : p.x);
          const y = Math.max(18, Math.min(height - pad.b - 8, p.y + (si === 0 ? -14 : 18)));
          return `<text x="${x.toFixed(1)}" y="${y.toFixed(1)}" fill="${s.color}" font-size="9" font-weight="700" text-anchor="${anchor}">${ff(p.value)}</text>`;
        }).join('');
        return `<path d="${path}" fill="none" stroke="${s.color}" stroke-width="3.2" stroke-linecap="round" stroke-linejoin="round"></path>${dots}${vLbl}`;
      }).join('');
      const legend = validSeries.map(s => `<div class="cmp-legend-item"><span class="cmp-dot" style="background:${s.color}"></span>${s.name}</div>`).join('');
      const latestLabel = labels.length ? labels[labels.length - 1] : '';
      e.innerHTML = `
        <div class="cmp-wrap">
          <div class="trend-head">
            <div class="trend-meta">
              <div class="trend-label">Comparativo mensal</div>
              <div class="trend-sub">Leitura mês a mês dos anos selecionados</div>
            </div>
            <div class="cmp-legend">${legend}</div>
          </div>
          <div class="trend-note">Última referência exibida: <b style="color:#fff">${latestLabel || 'N/A'}</b></div>
          <svg class="trend-svg" viewBox="0 0 ${width} ${height}" preserveAspectRatio="none">
            ${grid}
            ${rendered}
            ${labelMarks}
          </svg>
        </div>`;
    }

    function getSigla(f){ return (f.properties.SIGLA || f.properties.sigla || f.id || NM[f.properties.name] || NM[f.properties.NAME_1] || '??').toUpperCase(); }

    function monthMapForFilters(f){
      const mm = {}; MO.forEach(m => mm[m] = 0);
      DV.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1])){ mm[r[2]] = (mm[r[2]] || 0) + r[4]; } });
      return mm;
    }

    function previousMonthWithData(mm, currentMonth){
      let idx = MO.indexOf(currentMonth) - 1;
      while(idx >= 0){
        const m = MO[idx];
        if((mm[m] || 0) !== 0){ return m; }
        idx -= 1;
      }
      return null;
    }

    function currentMonthForAnalytics(f, mm){
      if(f.m.length < 12){ return f.m[f.m.length - 1]; }
      const months = MO.filter(m => (mm[m] || 0) !== 0);
      return months.length ? months[months.length - 1] : null;
    }

    function renderAnalytics(f){
      const el = document.getElementById('cA');
      const mm = monthMapForFilters(f);
      const targetMonth = currentMonthForAnalytics(f, mm);
      if(!targetMonth){ el.innerHTML = emptyState('Sem dados para análise'); return; }
      const prevMonth = previousMonthWithData(mm, targetMonth);
      const currentValue = mm[targetMonth] || 0;
      const prevValue = prevMonth ? (mm[prevMonth] || 0) : 0;
      const variation = prevValue > 0 ? ((currentValue - prevValue) / prevValue) * 100 : null;
      const varClass = variation === null ? 'neu' : (variation >= 0 ? 'pos' : 'neg');
      const varText = variation === null ? 'Sem base' : fp(variation);
      const varSub = variation === null
        ? `Não há mês anterior com faturamento para comparar em ${f.a.join('/')}`
        : `${variation >= 0 ? 'Variação positiva' : 'Variação negativa'} de ${MO_FULL[targetMonth]} contra ${MO_FULL[prevMonth]}`;

      const clientMonth = {};
      DC.forEach(r => {
        if(f.c.includes(r[0]) && f.a.includes(r[1]) && r[2] === targetMonth){
          clientMonth[r[3]] = (clientMonth[r[3]] || 0) + r[4];
        }
      });
      const top10Value = Object.values(clientMonth).sort((a,b) => b - a).slice(0, 10).reduce((acc, v) => acc + v, 0);
      const top10Pct = currentValue > 0 ? (top10Value / currentValue) * 100 : 0;

      const regionSet = new Set();
      DU.forEach(r => {
        if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2]) && r[4] > 0){
          const region = UR[r[3]];
          if(region){ regionSet.add(region); }
        }
      });

      el.innerHTML = `
        <div class="ana-ref">Mês de referência: <b style="color:#fff">${MO_FULL[targetMonth]}/${f.a.join('/')}</b></div>
        <div class="ana-grid">
          <div class="ana-item">
            <div class="ana-lbl">Variação do último mês vs anterior</div>
            <div class="ana-val ${varClass}">${varText}</div>
            <div class="ana-sub">${varSub}</div>
          </div>
          <div class="ana-item">
            <div class="ana-lbl">Participação do top 10 clientes</div>
            <div class="ana-val">${top10Pct.toLocaleString('pt-BR',{minimumFractionDigits:1,maximumFractionDigits:1})}%</div>
            <div class="ana-sub">Top 10 representam ${ff(top10Value)} de ${ff(currentValue)} no mês de referência</div>
          </div>
          <div class="ana-item">
            <div class="ana-lbl">Regiões Atendidas</div>
            <div class="ana-val">${regionSet.size}</div>
            <div class="ana-sub">Quantidade de regiões com faturamento dentro dos filtros aplicados</div>
          </div>
        </div>`;
    }

    const _ms = document.getElementById('mS');
    const brGeo = {"type":"FeatureCollection","features":[{"type":"Feature","id":"AC","properties":{"SIGLA":"AC","Estado":"Acre"},"geometry":{"type":"Polygon","coordinates":[[[-67.32595,-9.59237],[-66.61982,-9.89408],[-66.62746,-9.93578],[-66.87321,-10.08096],[-67.01619,-10.25825],[-67.16935,-10.33745],[-67.31591,-10.31974],[-67.3113,-10.37672],[-67.35088,-10.39241],[-67.40763,-10.37429],[-67.44408,-10.45534],[-67.57971,-10.50322],[-67.67678,-10.60526],[-67.70758,-10.71087],[-67.86433,-10.64109],[-68.03336,-10.65528],[-68.10582,-10.71731],[-68.1038,-10.77584],[-68.2379,-10.95656],[-68.39614,-11.05106],[-68.48784,-11.05101],[-68.54304,-11.11046],[-68.71623,-11.14526],[-68.76553,-11.06665],[-68.75814,-11.00122],[-68.91228,-11.02235],[-69.0728,-10.96636],[-69.42184,-10.92721],[-69.737,-10.97488],[-69.7695,-10.93015],[-69.93489,-10.92233],[-70.15916,-11.04138],[-70.30719,-11.07026],[-70.43722,-11.03966],[-70.5308,-10.93508],[-70.62149,-11.00025],[-70.6221,-9.82197],[-70.53526,-9.76366],[-70.52498,-9.71395],[-70.60087,-9.56328],[-70.55634,-9.57162],[-70.57067,-9.5307],[-70.50517,-9.504],[-70.49326,-9.42663],[-70.59425,-9.44106],[-70.96342,-9.74793],[-70.99577,-9.81714],[-71.16395,-9.87613],[-71.21179,-9.9672],[-71.37603,-10.00124],[-72.19939,-10.00056],[-72.14931,-9.79978],[-72.2696,-9.75016],[-72.24646,-9.6549],[-72.28363,-9.5403],[-72.75087,-9.41067],[-73.21276,-9.41176],[-73.07856,-9.23727],[-72.99665,-9.19893],[-73.0184,-9.17772],[-72.95334,-9.12909],[-72.94217,-8.98733],[-73.04582,-8.90316],[-73.15033,-8.6843],[-73.20948,-8.6861],[-73.28785,-8.61988],[-73.30591,-8.47237],[-73.38996,-8.46918],[-73.41327,-8.41139],[-73.53784,-8.34627],[-73.62464,-8.02923],[-73.77288,-7.90277],[-73.76203,-7.85843],[-73.70044,-7.86958],[-73.68469,-7.77684],[-73.82255,-7.71828],[-73.98421,-7.56697],[-73.92018,-7.46607],[-73.96704,-7.35239],[-73.87051,-7.37922],[-73.70068,-7.30469],[-73.80135,-7.11185],[-72.58459,-7.55203],[-70.05224,-7.84739],[-68.64708,-9.04944],[-67.5173,-9.5607],[-67.32595,-9.59237]]]}},{"type":"Feature","id":"AL","properties":{"SIGLA":"AL","Estado":"Alagoas"},"geometry":{"type":"Polygon","coordinates":[[[-35.15263,-8.91373],[-35.35413,-9.25547],[-35.65001,-9.56756],[-35.69487,-9.66481],[-35.81287,-9.73224],[-36.04637,-10.07569],[-36.2722,-10.2751],[-36.3854,-10.49936],[-36.4569,-10.40738],[-36.56307,-10.41973],[-36.56594,-10.32955],[-36.62545,-10.25641],[-36.68514,-10.27529],[-36.83569,-10.20411],[-36.9216,-10.12835],[-36.99492,-9.9754],[-37.04425,-9.98873],[-37.153,-9.90089],[-37.23558,-9.89607],[-37.41779,-9.76299],[-37.58173,-9.73412],[-37.7118,-9.63133],[-37.78512,-9.63866],[-37.89417,-9.53956],[-37.98936,-9.52739],[-38.02218,-9.47003],[-38.20504,-9.41676],[-38.23762,-9.32983],[-38.09232,-9.17276],[-37.97938,-9.1484],[-37.83913,-8.97842],[-37.82912,-8.89381],[-37.75887,-8.85659],[-37.69888,-8.9925],[-37.64392,-9.01838],[-37.58364,-8.9702],[-37.49037,-8.96594],[-37.4694,-9.02771],[-37.31555,-9.09431],[-37.23436,-9.24008],[-37.19835,-9.21552],[-37.16437,-9.27574],[-37.10631,-9.23982],[-36.95247,-9.38258],[-36.8713,-9.26812],[-36.69388,-9.27536],[-36.60509,-9.34154],[-36.43704,-9.21213],[-36.22429,-9.17132],[-36.26688,-9.10241],[-36.1207,-9.02748],[-36.12732,-8.95614],[-35.89624,-8.8543],[-35.79511,-8.84772],[-35.74844,-8.91718],[-35.47017,-8.81316],[-35.15263,-8.91373]]]}},{"type":"Feature","id":"AM","properties":{"SIGLA":"AM","Estado":"Amazonas"},"geometry":{"type":"Polygon","coordinates":[[[-67.34329,2.20552],[-67.2779,1.87579],[-67.15686,1.84889],[-67.09719,1.73264],[-67.08797,1.16694],[-66.85675,1.23025],[-66.31634,0.73617],[-66.2128,0.78069],[-66.15091,0.74451],[-66.08836,0.7591],[-66.06988,0.8051],[-65.96425,0.80951],[-65.87955,0.93273],[-65.73925,0.99963],[-65.58539,1.00897],[-65.49348,0.88214],[-65.5912,0.72165],[-65.54045,0.64894],[-65.42342,0.708],[-65.41286,0.81575],[-65.32897,0.9316],[-65.18231,0.92248],[-65.15489,1.12501],[-65.10292,1.15669],[-65.02208,1.11498],[-64.96064,1.23021],[-64.86841,1.2319],[-64.81036,1.31442],[-64.74648,1.22548],[-64.3973,1.52687],[-64.3487,1.50211],[-64.39981,1.3949],[-64.33786,1.36355],[-64.30203,1.46831],[-64.09508,1.61713],[-64.06175,1.93077],[-63.99536,1.97992],[-63.84592,1.96408],[-63.70314,2.04629],[-63.66786,2.01691],[-63.6173,2.10658],[-63.42092,2.13134],[-63.37193,2.21188],[-63.28263,2.1539],[-63.14136,2.17246],[-63.05341,2.0289],[-62.83868,2.01648],[-62.70505,1.93658],[-62.72358,1.71241],[-62.80513,1.59136],[-62.63982,1.43825],[-62.54571,1.12376],[-62.47148,1.08629],[-62.44344,0.96825],[-62.48593,0.86654],[-62.4576,0.7852],[-62.53526,0.69037],[-62.53287,0.50882],[-62.44623,0.37912],[-62.46815,0.27576],[-62.42426,0.09158],[-62.34651,0.02961],[-62.3196,-0.10252],[-62.24491,-0.17471],[-62.24824,-0.30193],[-62.18824,-0.33082],[-62.30907,-0.51415],[-62.2963,-0.65221],[-62.40678,-0.72729],[-62.4865,-0.68146],[-62.51011,-0.75896],[-62.01734,-1.14174],[-61.8965,-1.39535],[-61.79009,-1.37897],[-61.63539,-1.43397],[-61.47456,-1.57897],[-61.53817,-1.43341],[-61.619,-1.3948],[-61.58205,-1.35341],[-61.62873,-1.30147],[-61.544,-1.0623],[-61.58511,-0.93702],[-61.54205,-0.76369],[-61.46455,-0.6648],[-61.224,-0.55953],[-61.21678,-0.50009],[-60.92067,-0.55537],[-60.90817,-0.62398],[-60.76095,-0.76148],[-60.75261,-0.8612],[-60.6679,-0.89453],[-60.59123,-0.85342],[-60.53095,-0.87509],[-60.47901,-0.77093],[-60.30956,-0.72482],[-60.31345,-0.64148],[-60.38317,-0.59009],[-60.39984,-0.5101],[-60.28983,-0.29316],[-60.29928,-0.21176],[-60.22428,-0.14288],[-60.20316,-0.03361],[-60.13845,0.02875],[-60.11616,0.13321],[-60.05897,0.16297],[-60.03786,0.26352],[-58.89536,0.26347],[-58.87197,-0.34303],[-58.72967,-0.43532],[-58.70508,-0.67901],[-58.43596,-0.8835],[-58.42999,-1.02715],[-58.32319,-1.14333],[-58.25468,-1.13262],[-58.16255,-1.22969],[-58.08177,-1.12652],[-58.01745,-1.10619],[-57.97193,-1.1626],[-58.00085,-1.33021],[-57.96026,-1.40161],[-57.82395,-1.43979],[-57.7836,-1.50804],[-57.71308,-1.50418],[-57.67859,-1.59437],[-57.6025,-1.57033],[-57.5287,-1.65276],[-57.40338,-1.67556],[-57.39261,-1.72311],[-57.25381,-1.69633],[-57.16482,-1.72091],[-57.16313,-1.76862],[-57.06382,-1.80853],[-57.03713,-1.91159],[-56.98827,-1.9072],[-56.84988,-2.02021],[-56.73347,-2.02283],[-56.76846,-2.16579],[-56.679,-2.2126],[-56.61284,-2.21021],[-56.52699,-2.13923],[-56.41381,-2.17599],[-56.2283,-2.05626],[-56.09858,-2.02698],[-56.22022,-2.19588],[-56.38424,-2.27051],[-56.46552,-2.42315],[-56.40209,-2.45665],[-58.25518,-6.45413],[-58.47839,-6.69962],[-58.43448,-6.90882],[-58.20867,-7.13636],[-58.13698,-7.35613],[-58.21298,-7.45854],[-58.20209,-7.62086],[-58.38466,-7.844],[-58.28629,-8.08849],[-58.32239,-8.17005],[-58.31482,-8.32307],[-58.41698,-8.49197],[-58.38944,-8.5948],[-58.43722,-8.70336],[-58.32615,-8.72004],[-58.38793,-8.7748],[-58.50028,-8.80283],[-61.58305,-8.7987],[-61.63047,-8.72197],[-61.71319,-8.68787],[-61.77452,-8.74824],[-61.83642,-8.73274],[-61.85997,-8.8531],[-61.98514,-8.87862],[-62.03098,-8.79973],[-62.12489,-8.80169],[-62.20044,-8.64431],[-62.1883,-8.59057],[-62.2688,-8.57577],[-62.28122,-8.63861],[-62.33569,-8.6095],[-62.36499,-8.39104],[-62.46539,-8.33975],[-62.52605,-8.38307],[-62.56183,-8.28402],[-62.68657,-8.17379],[-62.69219,-8.09292],[-62.84496,-7.98654],[-63.62069,-7.96931],[-63.7815,-8.32926],[-63.86739,-8.28933],[-63.94403,-8.3312],[-63.9778,-8.42015],[-63.94075,-8.45228],[-63.97441,-8.48726],[-63.92239,-8.56739],[-63.94586,-8.6095],[-63.98612,-8.58565],[-64.02778,-8.71568],[-64.14298,-8.74305],[-64.14936,-8.95896],[-64.31123,-8.99649],[-64.32331,-8.92888],[-64.61098,-9.01647],[-64.83923,-8.99413],[-64.91718,-9.04316],[-64.94059,-9.26243],[-65.09776,-9.43254],[-65.18453,-9.42702],[-65.20793,-9.28566],[-65.24657,-9.25769],[-65.38512,-9.33806],[-65.44693,-9.3164],[-65.43463,-9.46636],[-65.59668,-9.41389],[-65.67934,-9.45142],[-65.72346,-9.55908],[-65.79143,-9.58567],[-65.97021,-9.41319],[-66.3074,-9.43687],[-66.40876,-9.40697],[-66.39232,-9.50051],[-66.50052,-9.63359],[-66.77527,-9.75312],[-66.81016,-9.81808],[-67.32595,-9.59237],[-67.5173,-9.5607],[-68.64708,-9.04944],[-70.05224,-7.84739],[-72.58459,-7.55203],[-73.80135,-7.11185],[-73.72658,-7.02255],[-73.75268,-6.9331],[-73.6426,-6.76218],[-73.35429,-6.59494],[-73.21276,-6.57786],[-73.13822,-6.498],[-73.10878,-6.41058],[-73.25004,-6.14497],[-73.23726,-6.03222],[-73.18882,-5.99926],[-73.15676,-5.86944],[-72.96157,-5.6546],[-72.9567,-5.46543],[-72.86409,-5.2733],[-72.88486,-5.16684],[-72.81437,-5.10976],[-72.62931,-5.05204],[-72.60647,-4.9956],[-72.38548,-4.8723],[-72.37145,-4.80738],[-72.12694,-4.72336],[-72.03913,-4.62326],[-71.94526,-4.60492],[-71.8827,-4.51619],[-71.76104,-4.50361],[-71.74837,-4.4726],[-71.71476,-4.51227],[-71.65376,-4.50827],[-71.6192,-4.47026],[-71.63401,-4.5192],[-71.60409,-4.53369],[-71.53737,-4.46739],[-71.49468,-4.48742],[-71.50928,-4.44836],[-71.41765,-4.46625],[-71.35376,-4.42799],[-71.31446,-4.45858],[-71.32378,-4.42166],[-71.26641,-4.42874],[-71.27792,-4.38805],[-71.23037,-4.40779],[-71.20693,-4.37987],[-71.20123,-4.42607],[-71.15363,-4.38314],[-71.11748,-4.41055],[-71.10035,-4.3774],[-70.99552,-4.38451],[-70.99663,-4.34522],[-70.94288,-4.38587],[-70.80799,-4.18354],[-70.75882,-4.15998],[-70.68085,-4.1998],[-70.65332,-4.1272],[-70.62377,-4.13178],[-70.62477,-4.1928],[-70.54915,-4.13719],[-70.50327,-4.19714],[-70.43932,-4.13415],[-70.34063,-4.18683],[-70.32585,-4.1463],[-70.29219,-4.17652],[-70.29818,-4.29075],[-70.25544,-4.27769],[-70.26666,-4.31202],[-70.21638,-4.3191],[-70.19891,-4.36642],[-70.10769,-4.26446],[-70.03261,-4.35517],[-69.96364,-4.30065],[-69.45731,-1.49346],[-69.3998,-1.37071],[-69.43107,-1.23303],[-69.39552,-1.13258],[-69.44321,-1.03658],[-69.42147,-1.00041],[-69.52335,-0.92972],[-69.62618,-0.74973],[-69.5641,-0.63979],[-69.61527,-0.50628],[-69.84468,-0.34578],[-69.92295,-0.3315],[-70.05722,-0.18676],[-70.04335,0.55885],[-69.80922,0.57222],[-69.67959,0.6706],[-69.60659,0.62923],[-69.48127,0.73563],[-69.35884,0.61187],[-69.30153,0.64925],[-69.29806,0.60341],[-69.20349,0.60538],[-69.19302,0.65155],[-69.11496,0.64987],[-69.18599,0.73321],[-69.13563,0.87721],[-69.18491,0.90283],[-69.16319,0.946],[-69.26515,1.06439],[-69.37934,1.09535],[-69.4163,1.05238],[-69.6067,1.1052],[-69.68455,1.0765],[-69.71037,1.12653],[-69.84416,1.0854],[-69.84233,1.7208],[-69.63625,1.7368],[-69.55213,1.79177],[-69.39303,1.7241],[-68.15661,1.73167],[-68.2667,1.82737],[-68.20848,1.96162],[-68.1405,1.98482],[-68.08733,1.90135],[-67.9411,1.83077],[-67.76878,2.03936],[-67.61964,2.02372],[-67.53991,2.15593],[-67.40639,2.24628],[-67.34329,2.20552]]]}},{"type":"Feature","id":"AP","properties":{"SIGLA":"AP","Estado":"Amapá"},"geometry":{"type":"Polygon","coordinates":[[[-50.09389,0.70205],[-50.31462,0.67797],[-50.40984,0.62329],[-50.55612,0.30667],[-50.67564,0.17913],[-50.93414,0.06963],[-51.08489,-0.08759],[-51.21633,-0.11866],[-51.45776,-0.51328],[-51.66545,-0.76194],[-51.70039,-1.06326],[-51.80906,-1.15715],[-51.92592,-1.13236],[-51.95584,-1.16972],[-51.98646,-1.12199],[-51.99109,-1.17186],[-52.06394,-1.18598],[-52.05461,-1.23426],[-52.10204,-1.22601],[-52.12023,-1.14624],[-52.33295,-1.11594],[-52.35812,-1.06263],[-52.427,-1.05084],[-52.43946,-0.9356],[-52.40312,-0.8772],[-52.45548,-0.83032],[-52.53862,-0.85486],[-52.49784,-0.7308],[-52.52314,-0.58913],[-52.63997,-0.58496],[-52.70264,-0.45929],[-52.68878,-0.30419],[-52.85552,-0.14811],[-52.9327,-0.14237],[-52.97725,0.02373],[-53.174,0.37424],[-53.10625,0.68391],[-53.41144,0.92927],[-53.45911,1.1336],[-53.39983,1.15833],[-53.42647,1.24264],[-53.53894,1.21235],[-53.54211,1.34476],[-53.65091,1.33614],[-53.64678,1.40754],[-53.6705,1.37191],[-53.71713,1.43168],[-53.73227,1.38621],[-53.80394,1.42344],[-53.85029,1.39176],[-53.89594,1.45504],[-53.93825,1.43538],[-53.92871,1.4691],[-53.99741,1.47569],[-54.00899,1.51958],[-54.08641,1.48821],[-54.14361,1.64053],[-54.18216,1.65799],[-54.19105,1.62482],[-54.30925,1.74096],[-54.74472,1.77571],[-54.75438,1.97051],[-54.81277,2.06272],[-54.76305,2.20223],[-54.87618,2.4266],[-54.74397,2.47139],[-54.68485,2.44673],[-54.66219,2.32689],[-54.60163,2.33707],[-54.43652,2.20979],[-54.18902,2.17874],[-53.9423,2.2423],[-53.93625,2.29689],[-53.90775,2.27391],[-53.88386,2.31624],[-53.8148,2.31693],[-53.83494,2.33957],[-53.76716,2.37883],[-53.72341,2.35398],[-53.74872,2.31272],[-53.54986,2.25823],[-53.45494,2.26418],[-53.4696,2.28874],[-53.33813,2.35351],[-53.22946,2.26229],[-53.27975,2.22325],[-53.27913,2.1859],[-53.0995,2.21419],[-52.98445,2.15709],[-52.90594,2.18544],[-52.85733,2.27734],[-52.71616,2.33552],[-52.5548,2.51489],[-52.52755,2.57673],[-52.55855,2.6339],[-52.38163,2.91339],[-52.33321,3.17215],[-52.19207,3.30126],[-51.92429,3.78334],[-51.80198,3.88176],[-51.75572,3.98801],[-51.64648,4.04445],[-51.55409,4.41464],[-51.51398,4.43666],[-51.25196,4.19133],[-51.17896,3.94476],[-51.07556,3.89053],[-51.09054,3.44675],[-51.02202,3.20801],[-51.03577,3.13706],[-50.94222,2.8033],[-50.90194,2.79454],[-50.84099,2.63865],[-50.85491,2.49065],[-50.80608,2.50919],[-50.7648,2.448],[-50.68233,2.17296],[-50.7025,2.13814],[-50.52197,2.20832],[-50.44611,2.19918],[-50.35241,2.08675],[-50.23811,1.80313],[-50.09172,1.79136],[-49.91503,1.69547],[-49.87619,1.48077],[-49.89364,1.19297],[-49.94581,1.1082],[-49.94076,0.9935],[-50.09306,0.78214],[-50.09389,0.70205]]]}},{"type":"Feature","id":"BA","properties":{"SIGLA":"BA","Estado":"Bahia"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-38.65357,-18.02441],[-38.65822,-18.04327],[-38.66784,-18.03251],[-38.65357,-18.02441]]],[[[-38.69387,-18.0172],[-38.67764,-18.01098],[-38.6797,-18.02293],[-38.69387,-18.0172]]],[[[-38.6604,-18.00544],[-38.64744,-17.99552],[-38.6484,-18.00758],[-38.6604,-18.00544]]],[[[-38.66236,-17.98697],[-38.65332,-17.99127],[-38.65738,-17.99128],[-38.66236,-17.98697]]],[[[-38.64276,-17.97207],[-38.63407,-17.98375],[-38.64885,-17.97839],[-38.64276,-17.97207]]],[[[-38.66597,-17.97134],[-38.65242,-17.97636],[-38.66335,-17.97721],[-38.66597,-17.97134]]],[[[-38.65182,-17.92264],[-38.65299,-17.94834],[-38.66102,-17.94609],[-38.65182,-17.92264]]],[[[-39.31248,-8.55835],[-39.22855,-8.71035],[-39.16959,-8.69208],[-39.04283,-8.73339],[-38.95397,-8.8041],[-38.8041,-8.78922],[-38.70657,-8.8623],[-38.66578,-8.97817],[-38.62413,-8.98008],[-38.60385,-8.88063],[-38.54117,-8.82352],[-38.46983,-8.86578],[-38.51984,-8.94762],[-38.46243,-9.01728],[-38.40354,-9.03727],[-38.32078,-8.99078],[-38.29643,-9.02289],[-38.31952,-9.13827],[-38.20504,-9.41676],[-37.99798,-9.49932],[-38.04054,-9.60262],[-37.98791,-9.64743],[-38.04339,-9.71227],[-38.02823,-9.77611],[-37.9667,-9.81609],[-37.99769,-9.91675],[-37.9575,-9.97132],[-37.83325,-9.99838],[-37.73491,-10.33284],[-37.83902,-10.37925],[-37.85893,-10.43024],[-37.81309,-10.50852],[-37.84061,-10.5496],[-37.81397,-10.69109],[-37.98522,-10.75911],[-38.04712,-10.69354],[-38.21083,-10.71727],[-38.24508,-10.82301],[-38.22638,-10.91661],[-38.10176,-11.01509],[-38.06177,-11.17213],[-37.97414,-11.19536],[-38.01493,-11.2857],[-37.97796,-11.39359],[-37.86795,-11.42629],[-37.81234,-11.51607],[-37.67356,-11.56868],[-37.65174,-11.51832],[-37.51993,-11.54888],[-37.34118,-11.44237],[-37.62252,-11.99579],[-38.04958,-12.63499],[-38.34712,-12.95078],[-38.48904,-13.01468],[-38.58324,-12.99977],[-38.56219,-12.92],[-38.61628,-12.93133],[-38.65535,-13.02952],[-38.93254,-13.21792],[-38.96438,-13.28093],[-38.96113,-13.36747],[-38.90627,-13.37796],[-38.88854,-13.44466],[-38.93537,-13.56007],[-38.89088,-13.65648],[-38.96682,-13.67066],[-38.99793,-13.74457],[-38.92867,-13.93706],[-39.06571,-14.70542],[-39.05357,-14.77826],[-39.02665,-14.77122],[-38.93312,-15.67605],[-38.8567,-15.86056],[-38.94907,-16.15633],[-39.02126,-16.26028],[-39.0085,-16.37549],[-39.05862,-16.42757],[-39.14142,-16.7569],[-39.11196,-16.8957],[-39.21306,-17.1688],[-39.19492,-17.57958],[-39.13581,-17.68823],[-39.27131,-17.87175],[-39.44314,-17.94489],[-39.54989,-18.09579],[-39.55503,-18.08542],[-39.58267,-18.08341],[-39.55131,-18.10025],[-39.66917,-18.34943],[-40.22198,-17.98068],[-40.26356,-17.92205],[-40.17439,-17.85188],[-40.22396,-17.73409],[-40.28038,-17.72281],[-40.34743,-17.6132],[-40.35467,-17.6443],[-40.41448,-17.61012],[-40.41193,-17.56775],[-40.46043,-17.56853],[-40.51969,-17.44684],[-40.62348,-17.40601],[-40.60904,-17.327],[-40.54776,-17.28374],[-40.5708,-17.06236],[-40.49122,-16.88452],[-40.28165,-16.90122],[-40.25782,-16.80658],[-40.34551,-16.7873],[-40.27575,-16.57406],[-40.15987,-16.58036],[-40.17123,-16.52418],[-40.09966,-16.42325],[-40.06581,-16.45756],[-39.99172,-16.31374],[-39.9167,-16.28388],[-39.93654,-16.24582],[-39.85685,-16.11386],[-39.9357,-16.02379],[-39.91516,-16.00003],[-40.00445,-16.00188],[-40.08501,-15.89716],[-40.17073,-15.90757],[-40.23113,-15.80389],[-40.37687,-15.82356],[-40.46144,-15.75327],[-40.56348,-15.80294],[-40.70065,-15.66803],[-40.76767,-15.71412],[-40.81685,-15.64761],[-40.88157,-15.69451],[-40.96313,-15.64844],[-41.00391,-15.71189],[-41.14435,-15.77193],[-41.33121,-15.74471],[-41.35805,-15.49884],[-41.80071,-15.10114],[-41.93254,-15.17454],[-42.09162,-15.18664],[-42.17332,-15.08599],[-42.26479,-15.12522],[-42.44293,-15.06136],[-42.4368,-15.02275],[-42.72335,-14.88565],[-42.93887,-14.70807],[-43.17634,-14.65058],[-43.43488,-14.72307],[-43.53138,-14.81533],[-43.5806,-14.75085],[-43.88349,-14.65326],[-43.87482,-14.52458],[-43.78317,-14.33927],[-43.99763,-14.26544],[-44.1356,-14.28222],[-44.21532,-14.23321],[-44.56555,-14.34009],[-44.83322,-14.49985],[-44.84337,-14.58086],[-45.0403,-14.68051],[-45.08275,-14.74879],[-45.20555,-14.74476],[-45.45511,-14.95374],[-45.56754,-14.94585],[-45.7216,-15.11225],[-45.82666,-15.11697],[-45.87346,-15.15814],[-45.95372,-15.13955],[-46.07705,-15.2647],[-46.11922,-15.19224],[-46.04775,-15.16375],[-45.96605,-14.96566],[-46.0528,-14.83139],[-46.00797,-14.7941],[-46.02955,-14.68342],[-45.98972,-14.65921],[-45.97706,-14.53886],[-46.01731,-14.42072],[-45.93532,-14.41536],[-45.90727,-14.35325],[-46.061,-14.23254],[-46.06113,-14.19439],[-46.26575,-14.09798],[-46.20998,-14.01287],[-46.26811,-13.94378],[-46.23363,-13.84563],[-46.28242,-13.79651],[-46.2243,-13.7654],[-46.26686,-13.74034],[-46.23549,-13.70818],[-46.26151,-13.68906],[-46.16194,-13.59095],[-46.23547,-13.56293],[-46.20639,-13.46565],[-46.24329,-13.43028],[-46.0411,-13.27764],[-46.26647,-13.34837],[-46.31533,-13.30492],[-46.29161,-13.14361],[-46.32264,-13.0983],[-46.27335,-13.01497],[-46.12582,-12.9615],[-46.11388,-12.91827],[-46.30476,-12.94989],[-46.26228,-12.83169],[-46.27992,-12.58492],[-46.16062,-12.52859],[-46.15386,-12.48337],[-46.25478,-12.49433],[-46.25638,-12.42434],[-46.35237,-12.33742],[-46.3234,-12.30356],[-46.37435,-12.28983],[-46.39784,-12.04036],[-46.32568,-11.95851],[-46.17231,-11.89792],[-46.37456,-11.86862],[-46.3227,-11.77076],[-46.37345,-11.75115],[-46.29404,-11.69278],[-46.31151,-11.62724],[-46.08317,-11.63615],[-46.31343,-11.53951],[-46.47929,-11.51641],[-46.61707,-11.28947],[-46.47234,-11.1912],[-46.39892,-10.99441],[-46.2834,-10.90677],[-46.28392,-10.83208],[-46.22978,-10.7986],[-46.25141,-10.75709],[-46.19254,-10.71974],[-46.21095,-10.64969],[-46.06448,-10.60278],[-46.02501,-10.5428],[-45.83075,-10.43876],[-45.82737,-10.36826],[-45.74306,-10.34676],[-45.69751,-10.26359],[-45.72308,-10.15518],[-45.60571,-10.10878],[-45.39598,-10.44505],[-45.44724,-10.50745],[-45.42978,-10.63411],[-45.24804,-10.82199],[-45.07322,-10.84045],[-44.93121,-10.92878],[-44.84429,-10.90006],[-44.75122,-10.78217],[-44.66677,-10.75891],[-44.66661,-10.68237],[-44.57777,-10.6268],[-44.50451,-10.65077],[-44.33309,-10.5488],[-44.26401,-10.6247],[-44.13131,-10.63492],[-44.07282,-10.58109],[-44.02332,-10.4081],[-43.91622,-10.42675],[-43.78481,-10.19171],[-43.76858,-10.0857],[-43.69291,-10.07743],[-43.66228,-10.00407],[-43.70903,-9.91325],[-43.65324,-9.83909],[-43.78492,-9.76242],[-43.84969,-9.54808],[-43.76973,-9.44208],[-43.48546,-9.26532],[-43.42783,-9.27262],[-43.27877,-9.42461],[-43.11962,-9.37193],[-43.05695,-9.41917],[-42.97026,-9.40955],[-42.94625,-9.51845],[-42.83714,-9.55041],[-42.82056,-9.61926],[-42.76517,-9.61678],[-42.72087,-9.53134],[-42.62173,-9.54137],[-42.58649,-9.49681],[-42.49112,-9.49315],[-42.31258,-9.31619],[-41.8383,-9.24243],[-41.7352,-9.09474],[-41.72362,-9.01358],[-41.54453,-8.96058],[-41.38127,-8.70731],[-41.28155,-8.73614],[-41.11373,-8.70383],[-41.12458,-8.75271],[-41.03625,-8.78591],[-41.02126,-8.8436],[-40.92135,-8.83547],[-40.80412,-9.09837],[-40.66741,-9.1591],[-40.68725,-9.27837],[-40.75561,-9.31358],[-40.77677,-9.45449],[-40.62318,-9.48294],[-40.52913,-9.41394],[-40.46777,-9.42061],[-40.42091,-9.35261],[-40.37143,-9.37875],[-40.33506,-9.35394],[-40.25581,-9.06326],[-40.12944,-9.11025],[-39.95806,-9.04863],[-39.87348,-8.93408],[-39.89298,-8.82883],[-39.67682,-8.78855],[-39.692,-8.66426],[-39.61005,-8.65725],[-39.40765,-8.53839],[-39.31248,-8.55835]]]]}},{"type":"Feature","id":"CE","properties":{"SIGLA":"CE","Estado":"Ceará"},"geometry":{"type":"Polygon","coordinates":[[[-37.25302,-4.83177],[-37.64037,-4.92646],[-37.90228,-5.50071],[-38.08291,-5.67259],[-38.0872,-5.73057],[-38.04704,-5.73031],[-38.16482,-5.94658],[-38.30483,-6.08701],[-38.36527,-6.0885],[-38.38198,-6.05283],[-38.44734,-6.08532],[-38.57874,-6.28035],[-38.56333,-6.35607],[-38.60207,-6.38966],[-38.53086,-6.39317],[-38.51794,-6.40877],[-38.61231,-6.51323],[-38.67318,-6.69726],[-38.61395,-6.78248],[-38.76524,-6.91095],[-38.76502,-6.99393],[-38.66987,-7.04774],[-38.68786,-7.1902],[-38.62483,-7.19136],[-38.53461,-7.29383],[-38.58428,-7.43322],[-38.64494,-7.46006],[-38.65535,-7.56558],[-38.71105,-7.58667],[-38.74283,-7.66036],[-38.81972,-7.66486],[-38.82813,-7.71965],[-38.9396,-7.75881],[-38.96214,-7.84421],[-39.01807,-7.8127],[-39.09132,-7.85822],[-39.13398,-7.72531],[-39.25259,-7.70497],[-39.27243,-7.66017],[-39.47248,-7.57522],[-39.45852,-7.47425],[-39.55285,-7.48069],[-39.71552,-7.33496],[-39.85147,-7.32525],[-40.12777,-7.38195],[-40.27902,-7.29943],[-40.43361,-7.36721],[-40.52374,-7.31848],[-40.48976,-7.11872],[-40.40627,-7.00536],[-40.42854,-6.86429],[-40.37053,-6.8031],[-40.43242,-6.81286],[-40.47391,-6.73491],[-40.73226,-6.65362],[-40.79214,-6.51289],[-40.78203,-6.3166],[-40.85224,-6.22425],[-40.87286,-6.05666],[-40.90756,-6.04469],[-40.87365,-5.96679],[-40.93879,-5.67337],[-40.9013,-5.6119],[-40.94248,-5.41974],[-40.91157,-5.40741],[-40.92531,-5.18213],[-41.13189,-5.04809],[-41.12262,-5.0081],[-41.22138,-4.93826],[-41.24959,-4.86972],[-41.20803,-4.7768],[-41.24955,-4.75608],[-41.17392,-4.66858],[-41.24255,-4.57155],[-41.12019,-4.3306],[-41.13592,-4.23355],[-41.0909,-4.17037],[-41.14319,-4.12428],[-41.1143,-4.0406],[-41.25674,-4.03488],[-41.22025,-3.94154],[-41.30083,-3.82633],[-41.23935,-3.71268],[-41.34176,-3.68081],[-41.37126,-3.56783],[-41.29923,-3.49106],[-41.42353,-3.36804],[-41.25666,-3.08805],[-41.25714,-3.00468],[-41.32269,-2.92136],[-41.25914,-2.88437],[-41.10173,-2.90099],[-40.59225,-2.84382],[-40.49967,-2.78445],[-39.94269,-2.85825],[-39.46482,-3.1422],[-39.25255,-3.22217],[-39.07961,-3.3967],[-38.99497,-3.39751],[-38.91074,-3.50374],[-38.81224,-3.54122],[-38.66685,-3.67471],[-38.4993,-3.7253],[-38.47769,-3.70125],[-38.36428,-3.88191],[-38.0123,-4.24762],[-37.86265,-4.3726],[-37.77013,-4.40133],[-37.72941,-4.49725],[-37.59769,-4.62292],[-37.32522,-4.70179],[-37.25302,-4.83177]]]}},{"type":"Feature","id":"DF","properties":{"SIGLA":"DF","Estado":"Distrito Federal"},"geometry":{"type":"Polygon","coordinates":[[[-48.20051,-15.50018],[-47.41731,-15.50032],[-47.4177,-15.54684],[-47.31732,-15.58857],[-47.31263,-15.7404],[-47.36592,-15.8282],[-47.37778,-15.97383],[-47.30878,-16.05054],[-48.27909,-16.05156],[-48.2511,-15.94645],[-48.28635,-15.83672],[-48.20687,-15.74617],[-48.24264,-15.68916],[-48.20028,-15.62187],[-48.20051,-15.50018]]]}},{"type":"Feature","id":"ES","properties":{"SIGLA":"ES","Estado":"Espírito Santo"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-40.39568,-20.62177],[-40.39649,-20.6222],[-40.39649,-20.62184],[-40.39568,-20.62177]]],[[[-40.37974,-20.61491],[-40.38291,-20.61557],[-40.38275,-20.61371],[-40.37974,-20.61491]]],[[[-40.37588,-20.61222],[-40.37809,-20.61432],[-40.378,-20.61246],[-40.37588,-20.61222]]],[[[-40.37291,-20.61197],[-40.37461,-20.61339],[-40.375,-20.61221],[-40.37291,-20.61197]]],[[[-40.38121,-20.61152],[-40.38254,-20.61301],[-40.38287,-20.61195],[-40.38121,-20.61152]]],[[[-40.3876,-20.61021],[-40.38857,-20.61086],[-40.38871,-20.61031],[-40.3876,-20.61021]]],[[[-40.37395,-20.60861],[-40.37359,-20.61181],[-40.37593,-20.61098],[-40.37395,-20.60861]]],[[[-40.39207,-20.59747],[-40.39055,-20.5983],[-40.39218,-20.59839],[-40.39207,-20.59747]]],[[[-40.35676,-20.5241],[-40.35795,-20.52464],[-40.35809,-20.52416],[-40.35676,-20.5241]]],[[[-40.31057,-20.42066],[-40.31228,-20.42166],[-40.3122,-20.4208],[-40.31057,-20.42066]]],[[[-40.29537,-20.38143],[-40.2983,-20.3819],[-40.29742,-20.37992],[-40.29537,-20.38143]]],[[[-40.27791,-20.36406],[-40.27944,-20.36532],[-40.27967,-20.36415],[-40.27791,-20.36406]]],[[[-40.27968,-20.36258],[-40.28083,-20.36354],[-40.28123,-20.36258],[-40.27968,-20.36258]]],[[[-40.25322,-20.35265],[-40.25414,-20.35399],[-40.25463,-20.35288],[-40.25322,-20.35265]]],[[[-40.28079,-20.35379],[-40.28224,-20.35491],[-40.28201,-20.35386],[-40.28079,-20.35379]]],[[[-40.25063,-20.35218],[-40.25232,-20.35248],[-40.25206,-20.35141],[-40.25063,-20.35218]]],[[[-40.27164,-20.33384],[-40.27251,-20.33569],[-40.27284,-20.3342],[-40.27164,-20.33384]]],[[[-40.27139,-20.33222],[-40.27153,-20.33339],[-40.27217,-20.33258],[-40.27139,-20.33222]]],[[[-40.26817,-20.33095],[-40.26965,-20.33269],[-40.26983,-20.33124],[-40.26817,-20.33095]]],[[[-40.27137,-20.32038],[-40.27272,-20.32239],[-40.27324,-20.32077],[-40.27137,-20.32038]]],[[[-40.26912,-20.31985],[-40.26976,-20.32173],[-40.27061,-20.32053],[-40.26912,-20.31985]]],[[[-40.2431,-20.29439],[-40.24236,-20.29505],[-40.24331,-20.295],[-40.2431,-20.29439]]],[[[-40.51806,-17.89398],[-40.22198,-17.98068],[-39.66603,-18.33324],[-39.72858,-18.5004],[-39.74699,-18.70618],[-39.69654,-19.36767],[-39.80743,-19.64823],[-40.00432,-19.75784],[-40.13968,-19.94916],[-40.19075,-20.05656],[-40.19781,-20.21683],[-40.2375,-20.28948],[-40.29205,-20.28573],[-40.27086,-20.33038],[-40.30987,-20.38145],[-40.42293,-20.63542],[-40.52705,-20.65562],[-40.54472,-20.67356],[-40.5297,-20.67839],[-40.53716,-20.68566],[-40.52272,-20.69139],[-40.49121,-20.66777],[-40.52097,-20.73068],[-40.62816,-20.84175],[-40.64927,-20.80736],[-40.75878,-20.86407],[-40.95792,-21.30175],[-41.09245,-21.21863],[-41.19091,-21.24847],[-41.56573,-21.18271],[-41.73565,-21.10334],[-41.71241,-20.87155],[-41.74186,-20.8728],[-41.75568,-20.80753],[-41.8798,-20.75955],[-41.8088,-20.64421],[-41.8564,-20.61718],[-41.80493,-20.54468],[-41.79823,-20.43501],[-41.85905,-20.37367],[-41.84839,-20.33017],[-41.78214,-20.29447],[-41.75676,-20.20686],[-41.38212,-20.1888],[-41.30792,-19.94834],[-41.18489,-19.88896],[-41.16848,-19.67222],[-41.03626,-19.56853],[-41.04576,-19.4879],[-40.98829,-19.50794],[-40.94934,-19.47316],[-40.96798,-19.42534],[-40.90829,-19.30652],[-40.94451,-19.27955],[-40.91716,-19.25686],[-40.94401,-19.14401],[-41.06556,-19.05134],[-41.01803,-18.97352],[-41.16607,-18.85837],[-41.24297,-18.85455],[-41.23255,-18.79729],[-41.13414,-18.79605],[-41.09748,-18.84232],[-40.91678,-18.81556],[-40.9415,-18.68931],[-41.05311,-18.62874],[-41.02042,-18.45911],[-41.1821,-18.43927],[-41.14426,-18.40526],[-41.15881,-18.30877],[-41.05609,-18.16664],[-40.9884,-18.16982],[-40.89312,-18.1079],[-40.7715,-18.15594],[-40.77331,-18.10758],[-40.90281,-17.9873],[-40.88272,-17.97057],[-40.80925,-17.95234],[-40.70369,-18.02375],[-40.51806,-17.89398]]]]}},{"type":"Feature","id":"GO","properties":{"SIGLA":"GO","Estado":"Goiás"},"geometry":{"type":"Polygon","coordinates":[[[-50.15818,-12.41237],[-50.22935,-12.53391],[-50.19313,-12.56431],[-50.30038,-12.68226],[-50.29261,-12.83992],[-49.91143,-12.96677],[-49.36949,-13.27463],[-49.34029,-13.2539],[-49.33743,-13.0663],[-49.23737,-12.88408],[-49.12099,-12.7904],[-49.06748,-12.84734],[-49.0777,-12.90417],[-48.97559,-12.9572],[-48.84673,-12.80981],[-48.73654,-12.92103],[-48.73071,-12.9898],[-48.60137,-13.06089],[-48.58637,-13.31762],[-48.50873,-13.12892],[-48.4697,-13.14959],[-48.44167,-13.29238],[-48.14662,-13.15202],[-48.16537,-13.3058],[-48.08298,-13.28879],[-48.06242,-13.23541],[-47.96629,-13.31527],[-47.82396,-13.31188],[-47.67895,-13.46772],[-47.6227,-13.36768],[-47.66886,-13.20942],[-47.63441,-13.10321],[-47.56131,-13.12469],[-47.56304,-13.18424],[-47.47809,-13.18769],[-47.42565,-13.28975],[-47.37931,-13.23112],[-47.28212,-13.26492],[-47.22589,-13.19351],[-47.15381,-13.20991],[-46.97884,-13.13159],[-46.75058,-12.96917],[-46.45462,-12.97117],[-46.4176,-12.82348],[-46.36617,-12.86487],[-46.36425,-12.99138],[-46.11388,-12.91827],[-46.12582,-12.9615],[-46.27335,-13.01497],[-46.32264,-13.0983],[-46.29161,-13.14361],[-46.33112,-13.24956],[-46.29792,-13.25588],[-46.31533,-13.30492],[-46.27936,-13.34792],[-46.0411,-13.27764],[-46.24329,-13.43028],[-46.20639,-13.46565],[-46.23547,-13.56293],[-46.16194,-13.59095],[-46.26151,-13.68906],[-46.23549,-13.70818],[-46.26686,-13.74034],[-46.2243,-13.7654],[-46.28242,-13.79651],[-46.23363,-13.84563],[-46.26811,-13.94378],[-46.20998,-14.01287],[-46.26575,-14.09798],[-46.06113,-14.19439],[-46.061,-14.23254],[-45.90694,-14.35807],[-45.93532,-14.41536],[-46.01731,-14.42072],[-45.97706,-14.53886],[-45.98972,-14.65921],[-46.02955,-14.68342],[-46.00797,-14.7941],[-46.08811,-14.93641],[-46.16139,-14.90619],[-46.17552,-14.94944],[-46.28683,-14.92816],[-46.32217,-14.8146],[-46.50341,-14.70409],[-46.56586,-14.78639],[-46.52152,-14.8512],[-46.5465,-14.92546],[-46.50257,-15.05227],[-46.64099,-15.0876],[-46.82984,-15.00928],[-46.92503,-15.05752],[-46.88897,-15.11113],[-46.94083,-15.23071],[-46.89145,-15.23527],[-46.83586,-15.32655],[-46.92928,-15.43995],[-46.94865,-15.55786],[-46.85485,-15.61831],[-46.81169,-15.8861],[-47.08905,-15.96173],[-47.14192,-15.92665],[-47.21949,-16.01425],[-47.31909,-16.0367],[-47.37581,-15.98663],[-47.37936,-15.88178],[-47.31412,-15.74724],[-47.31722,-15.58931],[-47.4177,-15.54684],[-47.41731,-15.50032],[-48.20051,-15.50018],[-48.20028,-15.62187],[-48.24264,-15.68916],[-48.20603,-15.74146],[-48.28706,-15.84314],[-48.25268,-15.94276],[-48.27909,-16.05156],[-47.3035,-16.06019],[-47.3527,-16.13284],[-47.32796,-16.24923],[-47.46003,-16.5051],[-47.41284,-16.57587],[-47.25076,-16.66601],[-47.19706,-16.91084],[-47.1259,-16.98177],[-47.18206,-17.06396],[-47.35195,-17.1665],[-47.36261,-17.22271],[-47.43442,-17.27622],[-47.44153,-17.34839],[-47.51118,-17.32558],[-47.54115,-17.45431],[-47.45706,-17.5384],[-47.33311,-17.52345],[-47.26576,-17.61045],[-47.37297,-17.831],[-47.28317,-18.05824],[-47.5337,-18.23071],[-47.5362,-18.19488],[-47.56753,-18.20386],[-47.63873,-18.28768],[-47.62615,-18.3343],[-47.74053,-18.36384],[-47.74945,-18.41255],[-47.81054,-18.40011],[-47.83454,-18.45127],[-47.9549,-18.50028],[-47.98087,-18.44286],[-48.11782,-18.42499],[-48.26237,-18.33188],[-48.31359,-18.38484],[-48.40568,-18.35634],[-48.47902,-18.38012],[-48.56137,-18.32388],[-48.81632,-18.37966],[-48.93673,-18.3062],[-49.077,-18.41678],[-49.12624,-18.3825],[-49.20563,-18.41179],[-49.19254,-18.44972],[-49.24877,-18.52291],[-49.39152,-18.6466],[-49.48179,-18.56103],[-49.49719,-18.4926],[-49.53387,-18.49251],[-49.55892,-18.54676],[-49.65325,-18.56382],[-49.64265,-18.60133],[-49.76849,-18.61098],[-49.78364,-18.64133],[-50.01386,-18.59948],[-50.08,-18.67224],[-50.30622,-18.69547],[-50.50878,-18.93573],[-50.4987,-19.03402],[-50.54335,-19.10592],[-50.67436,-19.13619],[-50.81467,-19.28578],[-50.87587,-19.41991],[-50.8249,-19.47503],[-50.84213,-19.49914],[-51.03068,-19.40143],[-51.08792,-19.30832],[-51.25217,-19.27407],[-51.42043,-19.16653],[-51.6579,-19.13791],[-51.8549,-19.05147],[-51.94038,-18.96786],[-52.08132,-18.9509],[-52.18143,-18.84791],[-52.33407,-18.82824],[-52.44907,-18.69096],[-52.55299,-18.65698],[-52.59282,-18.68989],[-52.74743,-18.69202],[-52.91824,-18.63794],[-52.9623,-18.54082],[-52.75905,-18.35057],[-52.93518,-18.29655],[-53.02764,-18.35147],[-53.1011,-18.31083],[-53.14319,-18.08184],[-53.07566,-18.05079],[-53.06999,-17.98591],[-53.13139,-17.9169],[-53.16857,-17.76626],[-53.24683,-17.69068],[-53.24615,-17.52424],[-53.19688,-17.38045],[-53.21912,-17.29961],[-53.11907,-17.11322],[-53.05596,-17.07037],[-53.01127,-16.85802],[-52.95054,-16.86041],[-52.93259,-16.80287],[-52.78593,-16.74119],[-52.74681,-16.63071],[-52.71565,-16.63926],[-52.74026,-16.59042],[-52.6261,-16.53345],[-52.60434,-16.46466],[-52.68834,-16.3963],[-52.6821,-16.30407],[-52.54763,-16.26238],[-52.56965,-16.22885],[-52.52548,-16.14056],[-52.32803,-16.0706],[-52.25258,-15.89266],[-52.01126,-15.88623],[-51.94878,-15.81039],[-51.87956,-15.82539],[-51.75776,-15.63595],[-51.7775,-15.539],[-51.72767,-15.55079],[-51.69874,-15.48496],[-51.65205,-15.17987],[-51.53591,-15.06969],[-51.33775,-14.97337],[-51.29044,-15.03433],[-51.24255,-15.0354],[-51.08401,-14.91645],[-51.08939,-14.84036],[-50.9628,-14.52702],[-50.99902,-14.41923],[-50.97522,-14.29085],[-50.91744,-14.11482],[-50.8336,-14.08985],[-50.87174,-13.73309],[-50.80391,-13.69097],[-50.76341,-13.53001],[-50.66541,-13.44228],[-50.6656,-13.37614],[-50.60716,-13.31086],[-50.61166,-13.06436],[-50.56913,-13.03913],[-50.59296,-13.00286],[-50.52588,-12.97642],[-50.47802,-12.70995],[-50.32513,-12.50881],[-50.15818,-12.41237]]]}},{"type":"Feature","id":"MA","properties":{"SIGLA":"MA","Estado":"Maranhão"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-43.99713,-2.40015],[-44.01675,-2.40745],[-44.00038,-2.39088],[-43.99713,-2.40015]]],[[[-44.45676,-2.13103],[-44.45162,-2.13423],[-44.45826,-2.13417],[-44.45676,-2.13103]]],[[[-44.48615,-2.11212],[-44.46377,-2.10118],[-44.47106,-2.12572],[-44.48615,-2.11212]]],[[[-44.44443,-2.05782],[-44.4713,-2.07705],[-44.48228,-2.07368],[-44.44443,-2.05782]]],[[[-44.43886,-2.01879],[-44.45751,-2.03016],[-44.44622,-2.00794],[-44.43886,-2.01879]]],[[[-44.56844,-1.91628],[-44.56694,-1.91794],[-44.56929,-1.91786],[-44.56844,-1.91628]]],[[[-44.63335,-1.85392],[-44.59614,-1.85087],[-44.63346,-1.8589],[-44.63335,-1.85392]]],[[[-44.60806,-1.82099],[-44.60179,-1.84076],[-44.61611,-1.82115],[-44.60806,-1.82099]]],[[[-44.59264,-1.81133],[-44.56071,-1.83967],[-44.58361,-1.85887],[-44.59264,-1.81133]]],[[[-44.62264,-1.79821],[-44.62312,-1.81874],[-44.63081,-1.78846],[-44.62264,-1.79821]]],[[[-44.60333,-1.77546],[-44.61707,-1.78523],[-44.61191,-1.77731],[-44.60333,-1.77546]]],[[[-44.60479,-1.81732],[-44.61786,-1.79186],[-44.59352,-1.76877],[-44.57627,-1.77971],[-44.60479,-1.81732]]],[[[-44.73525,-1.68251],[-44.74122,-1.70138],[-44.75226,-1.69418],[-44.73525,-1.68251]]],[[[-44.74162,-1.67963],[-44.74121,-1.68461],[-44.74528,-1.68301],[-44.74162,-1.67963]]],[[[-44.72322,-1.66223],[-44.72474,-1.67741],[-44.73808,-1.67422],[-44.72322,-1.66223]]],[[[-44.75163,-1.59125],[-44.75817,-1.59301],[-44.75596,-1.5871],[-44.75163,-1.59125]]],[[[-44.69222,-1.58435],[-44.69296,-1.5954],[-44.69782,-1.58889],[-44.69222,-1.58435]]],[[[-44.77489,-1.57635],[-44.77731,-1.58071],[-44.77819,-1.57748],[-44.77489,-1.57635]]],[[[-44.77275,-1.56253],[-44.78109,-1.57287],[-44.78374,-1.56892],[-44.77275,-1.56253]]],[[[-44.82019,-1.56193],[-44.82436,-1.57045],[-44.83156,-1.56052],[-44.82019,-1.56193]]],[[[-44.72922,-1.56245],[-44.75593,-1.58327],[-44.74302,-1.53796],[-44.72922,-1.56245]]],[[[-45.13262,-1.48869],[-45.1352,-1.49439],[-45.13332,-1.48556],[-45.13262,-1.48869]]],[[[-44.8098,-1.53174],[-44.79677,-1.56352],[-44.82869,-1.5413],[-44.76377,-1.48426],[-44.8098,-1.53174]]],[[[-45.13478,-1.46975],[-45.13736,-1.47664],[-45.13844,-1.46632],[-45.13478,-1.46975]]],[[[-44.73713,-1.51194],[-44.7854,-1.52849],[-44.76142,-1.4889],[-44.77231,-1.4623],[-44.75343,-1.45813],[-44.73713,-1.51194]]],[[[-45.09129,-1.45177],[-45.07461,-1.4754],[-45.09583,-1.47169],[-45.09129,-1.45177]]],[[[-45.29875,-1.42446],[-45.30193,-1.42467],[-45.30011,-1.42264],[-45.29875,-1.42446]]],[[[-45.07951,-1.41931],[-45.08259,-1.44172],[-45.08959,-1.43204],[-45.07951,-1.41931]]],[[[-45.10453,-1.40269],[-45.10668,-1.40637],[-45.10946,-1.40123],[-45.10453,-1.40269]]],[[[-45.00082,-1.40238],[-45.00451,-1.41125],[-45.0168,-1.39673],[-45.00082,-1.40238]]],[[[-45.40352,-1.39416],[-45.40677,-1.41864],[-45.41288,-1.41056],[-45.40352,-1.39416]]],[[[-45.10744,-1.39256],[-45.10868,-1.39705],[-45.1121,-1.39197],[-45.10744,-1.39256]]],[[[-45.51421,-1.40115],[-45.51845,-1.4006],[-45.50899,-1.39041],[-45.51421,-1.40115]]],[[[-45.11058,-1.38555],[-45.11404,-1.38581],[-45.11254,-1.38207],[-45.11058,-1.38555]]],[[[-45.06865,-1.36774],[-45.09346,-1.43188],[-45.07852,-1.37056],[-45.06865,-1.36774]]],[[[-45.49114,-1.37903],[-45.49738,-1.3897],[-45.48038,-1.3592],[-45.48325,-1.37089],[-45.49114,-1.37903]]],[[[-45.64348,-1.35871],[-45.64727,-1.36458],[-45.65318,-1.36266],[-45.64348,-1.35871]]],[[[-45.35726,-1.34771],[-45.3562,-1.3507],[-45.35973,-1.34983],[-45.35726,-1.34771]]],[[[-45.04952,-1.34375],[-45.04853,-1.34773],[-45.05256,-1.34628],[-45.04952,-1.34375]]],[[[-45.05058,-1.35092],[-45.03317,-1.3868],[-45.04009,-1.39876],[-45.05058,-1.35092]]],[[[-44.87594,-1.33975],[-44.87734,-1.34551],[-44.88186,-1.33886],[-44.87594,-1.33975]]],[[[-45.47041,-1.35028],[-45.48646,-1.35999],[-45.47673,-1.33732],[-45.47041,-1.35028]]],[[[-45.04805,-1.32744],[-45.05243,-1.34238],[-45.05526,-1.33192],[-45.04805,-1.32744]]],[[[-44.972,-1.33581],[-44.97929,-1.35455],[-45.00101,-1.34165],[-45.02843,-1.33891],[-44.972,-1.33581]]],[[[-45.62412,-1.32754],[-45.62853,-1.33975],[-45.63121,-1.31409],[-45.62412,-1.32754]]],[[[-45.63669,-1.30012],[-45.63632,-1.32019],[-45.63967,-1.30867],[-45.63669,-1.30012]]],[[[-45.54536,-1.28883],[-45.55157,-1.30781],[-45.55159,-1.29107],[-45.54536,-1.28883]]],[[[-45.67809,-1.30034],[-45.69394,-1.30423],[-45.691,-1.28835],[-45.67809,-1.30034]]],[[[-44.92604,-1.27823],[-44.84273,-1.33293],[-44.87587,-1.35566],[-44.86997,-1.32682],[-44.90861,-1.32689],[-44.89923,-1.31205],[-44.92604,-1.27823]]],[[[-44.98568,-1.3033],[-44.93071,-1.28569],[-44.91011,-1.32943],[-44.88898,-1.33626],[-44.92515,-1.35055],[-44.88756,-1.34609],[-44.98319,-1.40965],[-45.02006,-1.3579],[-45.00268,-1.34504],[-44.98263,-1.35655],[-44.97223,-1.38684],[-44.96536,-1.38705],[-44.97114,-1.33275],[-44.94035,-1.34302],[-44.98568,-1.3033]]],[[[-45.64806,-1.26759],[-45.64618,-1.27092],[-45.64982,-1.26949],[-45.64806,-1.26759]]],[[[-45.53531,-1.26535],[-45.53618,-1.28864],[-45.55981,-1.28377],[-45.53531,-1.26535]]],[[[-45.55513,-1.26014],[-45.56816,-1.2735],[-45.55704,-1.25702],[-45.55513,-1.26014]]],[[[-45.81304,-1.26114],[-45.81823,-1.26203],[-45.81298,-1.25564],[-45.81304,-1.26114]]],[[[-45.85601,-1.25295],[-45.85551,-1.26025],[-45.86248,-1.25959],[-45.85601,-1.25295]]],[[[-45.64649,-1.24246],[-45.64619,-1.25069],[-45.64965,-1.24563],[-45.64649,-1.24246]]],[[[-45.80213,-1.24203],[-45.79122,-1.24847],[-45.79822,-1.24776],[-45.80213,-1.24203]]],[[[-45.57042,-1.1901],[-45.53148,-1.23135],[-45.5933,-1.24667],[-45.57042,-1.1901]]],[[[-45.68043,-1.18771],[-45.65988,-1.2436],[-45.68341,-1.26203],[-45.68043,-1.18771]]],[[[-45.79873,-1.19611],[-45.77592,-1.21051],[-45.78626,-1.22795],[-45.79873,-1.19611]]],[[[-45.76221,-1.1859],[-45.76123,-1.1896],[-45.7649,-1.18735],[-45.76221,-1.1859]]],[[[-45.81949,-1.15511],[-45.81995,-1.15938],[-45.82051,-1.15705],[-45.81949,-1.15511]]],[[[-45.81577,-1.15427],[-45.81754,-1.15527],[-45.81722,-1.15313],[-45.81577,-1.15427]]],[[[-45.60388,-1.1271],[-45.63084,-1.21021],[-45.64143,-1.31011],[-45.63285,-1.3487],[-45.69647,-1.36213],[-45.67313,-1.34582],[-45.68741,-1.33104],[-45.67302,-1.3115],[-45.68462,-1.26982],[-45.65475,-1.25743],[-45.65243,-1.2697],[-45.6426,-1.2772],[-45.65996,-1.29217],[-45.65188,-1.31083],[-45.63839,-1.27444],[-45.62917,-1.12718],[-45.60388,-1.1271]]],[[[-45.79351,-1.1194],[-45.81654,-1.14883],[-45.81706,-1.12171],[-45.79351,-1.1194]]],[[[-45.95639,-1.10145],[-45.95411,-1.10666],[-45.95792,-1.10508],[-45.95639,-1.10145]]],[[[-45.84144,-1.04463],[-45.85194,-1.12578],[-45.88075,-1.13406],[-45.91407,-1.18188],[-45.86561,-1.14695],[-45.86573,-1.20198],[-45.91226,-1.23568],[-45.90347,-1.25999],[-45.84665,-1.21029],[-45.88024,-1.26011],[-45.86019,-1.28398],[-45.79169,-1.16298],[-45.84055,-1.28355],[-45.80117,-1.34445],[-45.83853,-1.28385],[-45.8064,-1.25928],[-45.80338,-1.25036],[-45.78062,-1.26511],[-45.79718,-1.27666],[-45.81028,-1.27173],[-45.81493,-1.2762],[-45.79433,-1.27899],[-45.78235,-1.27008],[-45.77745,-1.2751],[-45.73507,-1.13267],[-45.67823,-1.14279],[-45.74599,-1.23735],[-45.69479,-1.2696],[-45.69955,-1.30876],[-45.71893,-1.31382],[-45.73003,-1.34624],[-45.69803,-1.31321],[-45.69052,-1.33731],[-45.67799,-1.34478],[-45.74911,-1.36875],[-45.70041,-1.35823],[-45.72572,-1.41737],[-45.69473,-1.3711],[-45.62786,-1.3641],[-45.614,-1.27843],[-45.58017,-1.25727],[-45.6082,-1.34213],[-45.57655,-1.35239],[-45.58818,-1.31223],[-45.58099,-1.29578],[-45.55237,-1.27415],[-45.56232,-1.28291],[-45.5548,-1.31885],[-45.57434,-1.33163],[-45.57398,-1.34014],[-45.55744,-1.32799],[-45.54764,-1.30851],[-45.52985,-1.31659],[-45.5586,-1.33051],[-45.56965,-1.37513],[-45.50138,-1.29204],[-45.54462,-1.37346],[-45.51166,-1.36116],[-45.52717,-1.40438],[-45.51892,-1.41211],[-45.49196,-1.38954],[-45.4663,-1.34999],[-45.46562,-1.31977],[-45.41016,-1.28965],[-45.40829,-1.33904],[-45.48848,-1.4316],[-45.45028,-1.5175],[-45.48766,-1.53033],[-45.44359,-1.54309],[-45.44847,-1.4489],[-45.41705,-1.41547],[-45.42871,-1.46541],[-45.40522,-1.48727],[-45.40229,-1.42213],[-45.36898,-1.43673],[-45.39103,-1.41254],[-45.34838,-1.39406],[-45.37164,-1.3629],[-45.32981,-1.3613],[-45.35547,-1.31452],[-45.30092,-1.34336],[-45.29299,-1.42162],[-45.30452,-1.40633],[-45.31588,-1.41178],[-45.38826,-1.48176],[-45.3793,-1.51656],[-45.37284,-1.51601],[-45.38457,-1.48494],[-45.3457,-1.4699],[-45.37295,-1.54501],[-45.38418,-1.53744],[-45.39171,-1.54204],[-45.39878,-1.57489],[-45.3696,-1.5477],[-45.33708,-1.47449],[-45.32746,-1.51709],[-45.32223,-1.47361],[-45.29807,-1.49453],[-45.35513,-1.66726],[-45.38979,-1.68033],[-45.40273,-1.70602],[-45.35756,-1.67963],[-45.35281,-1.73663],[-45.30944,-1.60062],[-45.24798,-1.6227],[-45.2342,-1.54892],[-45.1512,-1.46096],[-45.14569,-1.52291],[-45.12638,-1.51162],[-45.12993,-1.52969],[-45.12724,-1.544],[-45.12308,-1.54762],[-45.10752,-1.54881],[-45.13011,-1.45943],[-45.10228,-1.40667],[-45.10124,-1.36041],[-45.09492,-1.45734],[-45.1093,-1.49614],[-45.09717,-1.5136],[-45.06912,-1.47651],[-45.06846,-1.45793],[-45.079,-1.4399],[-45.07079,-1.42437],[-45.06189,-1.43993],[-45.07646,-1.51725],[-45.05411,-1.47827],[-44.95861,-1.5152],[-44.9616,-1.49313],[-44.94577,-1.47673],[-44.93525,-1.48723],[-44.9502,-1.4918],[-44.95321,-1.49841],[-44.94299,-1.51734],[-44.87157,-1.42577],[-44.81751,-1.41724],[-44.82856,-1.48872],[-44.84074,-1.45819],[-44.88689,-1.4815],[-44.85217,-1.49252],[-44.90702,-1.54805],[-44.89799,-1.61375],[-44.84587,-1.6243],[-44.82237,-1.57487],[-44.77762,-1.62499],[-44.79906,-1.66002],[-44.77303,-1.63416],[-44.78475,-1.58282],[-44.77429,-1.58204],[-44.7678,-1.57714],[-44.75943,-1.5623],[-44.77012,-1.61129],[-44.73814,-1.61209],[-44.69835,-1.55328],[-44.73312,-1.62283],[-44.69221,-1.59985],[-44.68185,-1.56505],[-44.64275,-1.62427],[-44.65731,-1.66632],[-44.70031,-1.64104],[-44.78426,-1.67094],[-44.80478,-1.70467],[-44.75504,-1.73498],[-44.78855,-1.74374],[-44.78905,-1.74967],[-44.75588,-1.74384],[-44.75188,-1.73238],[-44.77287,-1.71107],[-44.69716,-1.738],[-44.84608,-1.82872],[-44.69923,-1.80393],[-44.66,-1.74375],[-44.696,-1.82215],[-44.64824,-1.76494],[-44.65197,-1.78769],[-44.648,-1.79107],[-44.63846,-1.73569],[-44.65335,-1.71996],[-44.59306,-1.74579],[-44.63448,-1.78422],[-44.63318,-1.82913],[-44.60769,-1.8347],[-44.6395,-1.85776],[-44.61971,-1.86866],[-44.56957,-1.8593],[-44.5471,-1.82209],[-44.52987,-1.83824],[-44.54806,-1.87057],[-44.60113,-1.89683],[-44.60521,-1.90602],[-44.56141,-1.9231],[-44.52319,-1.88238],[-44.48868,-1.94297],[-44.48788,-1.96699],[-44.50414,-1.9493],[-44.51506,-1.94735],[-44.52285,-1.95791],[-44.46596,-2.0161],[-44.50035,-2.05376],[-44.4996,-2.14178],[-44.39596,-2.21356],[-44.35585,-2.33715],[-44.41193,-2.41399],[-44.32599,-2.50004],[-44.02285,-2.39827],[-44.02229,-2.45028],[-44.05338,-2.43077],[-44.11882,-2.47289],[-43.97716,-2.46697],[-43.98055,-2.57312],[-43.83104,-2.36397],[-43.78533,-2.34024],[-43.7478,-2.37335],[-43.72855,-2.28798],[-43.61517,-2.21938],[-43.49279,-2.37317],[-43.38014,-2.32517],[-43.18517,-2.37354],[-42.79317,-2.56003],[-42.75045,-2.5447],[-42.69317,-2.65054],[-42.69348,-2.5866],[-42.74162,-2.55581],[-42.71082,-2.55731],[-42.47929,-2.71246],[-42.2282,-2.66713],[-42.2207,-2.66759],[-42.22476,-2.68918],[-42.21287,-2.70167],[-42.05031,-2.6892],[-42.02542,-2.7587],[-42.01595,-2.71673],[-41.82384,-2.71932],[-41.86497,-2.87804],[-41.79662,-2.95985],[-41.92276,-3.10796],[-41.93906,-3.18739],[-42.13164,-3.27537],[-42.09587,-3.30293],[-42.20438,-3.4358],[-42.33082,-3.43389],[-42.4587,-3.48569],[-42.50448,-3.45621],[-42.67576,-3.67562],[-42.66871,-3.79093],[-42.72621,-3.91003],[-42.82181,-3.99058],[-42.90307,-4.12437],[-42.88819,-4.15563],[-42.9892,-4.23408],[-42.96332,-4.37644],[-42.89246,-4.40675],[-42.84937,-4.48262],[-42.95164,-4.68062],[-42.92055,-4.73728],[-42.95033,-4.78978],[-42.85523,-4.93533],[-42.79796,-5.18324],[-42.8267,-5.34805],[-42.96879,-5.45203],[-43.03456,-5.59222],[-43.09901,-5.63321],[-43.09943,-5.9066],[-43.04863,-6.00057],[-43.07501,-6.05473],[-42.86256,-6.23947],[-42.8602,-6.33057],[-42.82934,-6.33745],[-42.87776,-6.41789],[-42.91978,-6.67034],[-43.01733,-6.76094],[-43.2447,-6.76638],[-43.42005,-6.84386],[-43.715,-6.69876],[-43.93048,-6.77108],[-43.96121,-6.74192],[-44.0334,-6.76043],[-44.0697,-6.8214],[-44.11628,-6.80581],[-44.10441,-6.85657],[-44.12513,-6.83561],[-44.26198,-6.99875],[-44.3062,-7.11719],[-44.38004,-7.11905],[-44.56424,-7.22758],[-44.68798,-7.39455],[-44.81673,-7.36123],[-44.92444,-7.47031],[-45.33976,-7.58027],[-45.45599,-7.67069],[-45.54037,-7.86472],[-45.58146,-8.15606],[-45.66155,-8.25004],[-45.79377,-8.58442],[-45.76563,-8.6096],[-45.93499,-8.77912],[-45.99428,-8.92689],[-45.89706,-9.18654],[-45.89366,-9.3422],[-45.82415,-9.37497],[-45.78313,-9.4798],[-45.84141,-9.56232],[-45.82141,-9.77315],[-45.86598,-9.87167],[-45.84227,-9.94158],[-45.87853,-10.11032],[-45.9456,-10.19646],[-45.9463,-10.25885],[-46.00403,-10.26168],[-46.02826,-10.17689],[-46.16568,-10.21316],[-46.21085,-10.16976],[-46.36787,-10.16883],[-46.4574,-10.0473],[-46.50943,-9.80139],[-46.57699,-9.8052],[-46.57507,-9.75859],[-46.64639,-9.73372],[-46.59237,-9.58722],[-46.53674,-9.55713],[-46.56096,-9.48405],[-46.63999,-9.45774],[-46.66596,-9.39194],[-46.76109,-9.41018],[-46.84704,-9.29244],[-46.82779,-9.19429],[-46.89354,-9.12232],[-46.93178,-9.12808],[-46.92332,-9.06568],[-47.06871,-9.06389],[-47.04346,-9.02656],[-47.06848,-9.01204],[-46.97027,-8.91451],[-46.99357,-8.88026],[-46.91343,-8.84784],[-46.93123,-8.7266],[-46.88693,-8.68651],[-46.91435,-8.58713],[-46.84601,-8.5336],[-46.80682,-8.39899],[-46.64129,-8.32152],[-46.54365,-8.31894],[-46.50758,-8.27079],[-46.51462,-8.16662],[-46.46692,-8.07432],[-46.4903,-7.98326],[-46.59702,-7.89936],[-46.66055,-7.90058],[-46.87274,-7.95558],[-46.99741,-8.06724],[-47.04349,-8.0536],[-47.17453,-7.85701],[-47.24499,-7.81552],[-47.24622,-7.75853],[-47.31241,-7.73158],[-47.31419,-7.66064],[-47.3598,-7.61616],[-47.38189,-7.65407],[-47.42763,-7.52578],[-47.4752,-7.53338],[-47.50469,-7.43665],[-47.59092,-7.43953],[-47.48311,-7.36123],[-47.50383,-7.29011],[-47.58752,-7.26373],[-47.64769,-7.30414],[-47.75043,-7.18952],[-47.70472,-7.13587],[-47.64444,-7.14363],[-47.5283,-6.97342],[-47.49863,-6.67125],[-47.37722,-6.26678],[-47.43309,-6.11215],[-47.43105,-5.86922],[-47.49336,-5.73054],[-47.50011,-5.52562],[-47.7443,-5.38015],[-47.84375,-5.37622],[-47.88556,-5.26071],[-48.00405,-5.23463],[-48.07925,-5.27508],[-48.17853,-5.26079],[-48.36389,-5.16839],[-48.52298,-5.19334],[-48.60612,-5.33651],[-48.67942,-5.30583],[-48.75513,-5.34918],[-47.79256,-4.58643],[-47.67997,-4.60865],[-47.61193,-4.55737],[-47.47473,-4.31506],[-47.37176,-4.2456],[-47.31942,-4.04779],[-47.19025,-3.98372],[-47.17314,-3.92384],[-47.08874,-3.8616],[-47.03746,-3.56135],[-46.94797,-3.47646],[-46.94463,-3.3782],[-46.81337,-3.30956],[-46.80618,-3.24449],[-46.75479,-3.21573],[-46.77188,-3.17724],[-46.71964,-3.15394],[-46.74782,-3.12089],[-46.6758,-3.09367],[-46.64803,-2.96171],[-46.67965,-2.88175],[-46.63759,-2.88323],[-46.61394,-2.82897],[-46.57581,-2.8414],[-46.63664,-2.79369],[-46.66395,-2.69421],[-46.62192,-2.64454],[-46.59375,-2.66539],[-46.505,-2.61493],[-46.49433,-2.53652],[-46.4187,-2.52797],[-46.44013,-2.41282],[-46.40752,-2.3687],[-46.46591,-2.36937],[-46.42821,-2.23975],[-46.37783,-2.2535],[-46.27591,-2.15872],[-46.20862,-1.8309],[-46.31113,-1.8039],[-46.32398,-1.76556],[-46.15367,-1.67764],[-46.13128,-1.57857],[-46.16954,-1.47037],[-46.09527,-1.32714],[-46.14019,-1.29934],[-46.14592,-1.22715],[-46.07708,-1.19626],[-46.05635,-1.1316],[-46.04022,-1.17398],[-46.04606,-1.12116],[-45.98403,-1.04514],[-45.9574,-1.07998],[-45.98205,-1.12247],[-45.9536,-1.12627],[-45.9902,-1.1381],[-45.99436,-1.16243],[-45.94268,-1.13823],[-45.94848,-1.23864],[-45.84144,-1.04463]]]]}},{"type":"Feature","id":"MG","properties":{"SIGLA":"MG","Estado":"Minas Gerais"},"geometry":{"type":"Polygon","coordinates":[[[-44.20903,-14.23726],[-44.1356,-14.28222],[-43.99763,-14.26543],[-43.78317,-14.33927],[-43.87482,-14.52458],[-43.85919,-14.67433],[-43.5806,-14.75085],[-43.53138,-14.81533],[-43.43488,-14.72307],[-43.17634,-14.65058],[-42.93887,-14.70806],[-42.72335,-14.88565],[-42.4368,-15.02275],[-42.44293,-15.06136],[-42.26479,-15.12522],[-42.17332,-15.08599],[-42.09162,-15.18664],[-41.93254,-15.17454],[-41.80071,-15.10114],[-41.35805,-15.49884],[-41.33121,-15.74471],[-41.14435,-15.77193],[-41.00391,-15.71189],[-40.96313,-15.64844],[-40.88157,-15.69451],[-40.81685,-15.64761],[-40.76767,-15.71412],[-40.70065,-15.66803],[-40.56348,-15.80294],[-40.46144,-15.75327],[-40.37687,-15.82356],[-40.23113,-15.80389],[-40.17073,-15.90757],[-40.08501,-15.89716],[-40.00445,-16.00188],[-39.91516,-16.00003],[-39.9357,-16.02379],[-39.85685,-16.11387],[-39.93654,-16.24582],[-39.9167,-16.28388],[-39.99172,-16.31374],[-40.06581,-16.45756],[-40.09966,-16.42325],[-40.17123,-16.52418],[-40.15987,-16.58036],[-40.27575,-16.57406],[-40.34551,-16.7873],[-40.25782,-16.80658],[-40.28165,-16.90122],[-40.48018,-16.87684],[-40.5708,-17.06236],[-40.54776,-17.28374],[-40.60904,-17.327],[-40.62348,-17.40601],[-40.51969,-17.44684],[-40.46043,-17.56853],[-40.41193,-17.56775],[-40.41448,-17.61012],[-40.35467,-17.6443],[-40.34743,-17.6132],[-40.28038,-17.72281],[-40.22396,-17.73409],[-40.17439,-17.85188],[-40.26356,-17.92205],[-40.22198,-17.98068],[-40.52705,-17.89197],[-40.70369,-18.02375],[-40.80925,-17.95234],[-40.88272,-17.97057],[-40.90281,-17.9873],[-40.77331,-18.10758],[-40.7715,-18.15594],[-40.89312,-18.1079],[-40.9884,-18.16982],[-41.05609,-18.16664],[-41.15881,-18.30877],[-41.14426,-18.40526],[-41.1821,-18.43927],[-41.02042,-18.45911],[-41.05311,-18.62874],[-40.9415,-18.68931],[-40.91678,-18.81556],[-41.09748,-18.84232],[-41.13414,-18.79605],[-41.23255,-18.79729],[-41.24297,-18.85455],[-41.16607,-18.85837],[-41.01803,-18.97352],[-41.0718,-19.02386],[-40.92651,-19.18855],[-40.94451,-19.27955],[-40.90829,-19.30652],[-40.96798,-19.42534],[-40.94488,-19.46012],[-40.97135,-19.50382],[-41.04576,-19.4879],[-41.03626,-19.56853],[-41.16848,-19.67222],[-41.18489,-19.88896],[-41.30792,-19.94834],[-41.38212,-20.1888],[-41.75676,-20.20686],[-41.78214,-20.29447],[-41.84839,-20.33017],[-41.85905,-20.37367],[-41.79823,-20.43501],[-41.80493,-20.54468],[-41.8564,-20.61718],[-41.8088,-20.64421],[-41.87881,-20.77871],[-41.92814,-20.79498],[-41.97667,-20.93549],[-42.15168,-20.97436],[-42.0805,-21.03609],[-42.2085,-21.1782],[-42.1897,-21.24988],[-42.29262,-21.46002],[-42.25298,-21.49367],[-42.36914,-21.61994],[-42.26736,-21.71419],[-42.88251,-21.96009],[-43.07413,-22.09332],[-43.13654,-22.10964],[-43.13113,-22.02966],[-43.24654,-22.00701],[-43.34953,-22.00337],[-43.56313,-22.08822],[-43.58608,-22.052],[-43.7346,-22.09723],[-43.76599,-22.06289],[-44.10007,-22.17328],[-44.23547,-22.26618],[-44.45644,-22.25722],[-44.66069,-22.38049],[-44.72397,-22.35992],[-44.89783,-22.45288],[-45.09111,-22.48307],[-45.27585,-22.61693],[-45.39997,-22.65365],[-45.47384,-22.58886],[-45.53989,-22.65255],[-45.58435,-22.65166],[-45.5765,-22.60167],[-45.66453,-22.65034],[-45.65986,-22.58005],[-45.71682,-22.57767],[-45.73741,-22.60851],[-45.69397,-22.65168],[-45.81797,-22.71134],[-45.72813,-22.72365],[-45.71219,-22.76981],[-45.71356,-22.81475],[-45.7549,-22.79288],[-45.78187,-22.85522],[-45.84172,-22.83315],[-45.8889,-22.87578],[-45.91265,-22.81656],[-46.00842,-22.88919],[-46.13907,-22.92272],[-46.14409,-22.85797],[-46.34462,-22.9045],[-46.38417,-22.86993],[-46.33452,-22.76027],[-46.47817,-22.69954],[-46.39304,-22.66294],[-46.43373,-22.57385],[-46.40693,-22.5399],[-46.64701,-22.42881],[-46.72324,-22.3066],[-46.67428,-22.17628],[-46.59851,-22.13636],[-46.72342,-22.0768],[-46.66476,-22.06884],[-46.68088,-22.02644],[-46.61244,-22.00484],[-46.69069,-21.83761],[-46.626,-21.7662],[-46.62118,-21.67553],[-46.56585,-21.6793],[-46.5182,-21.6128],[-46.50923,-21.46988],[-46.61523,-21.44145],[-46.6667,-21.36127],[-46.70594,-21.40482],[-46.76468,-21.36004],[-46.92075,-21.42711],[-47.01121,-21.42225],[-46.99856,-21.34907],[-47.07726,-21.24021],[-47.06173,-21.21084],[-47.1185,-21.18591],[-47.14371,-20.98237],[-47.24001,-20.88527],[-47.18622,-20.73143],[-47.11527,-20.70719],[-47.09687,-20.64451],[-47.15449,-20.51954],[-47.29273,-20.44772],[-47.29818,-20.34842],[-47.23104,-20.21914],[-47.25701,-20.16619],[-47.44026,-20.04882],[-47.42509,-20.01333],[-47.46826,-19.96141],[-47.63473,-20.04924],[-47.7042,-19.97989],[-47.85253,-19.98982],[-47.89937,-20.12566],[-47.97593,-20.03536],[-48.01982,-20.12014],[-48.11296,-20.14375],[-48.24085,-20.02927],[-48.2413,-20.14017],[-48.40574,-20.1135],[-48.64689,-20.16699],[-48.82571,-20.16195],[-48.88507,-20.26644],[-48.86702,-20.39885],[-48.89982,-20.44132],[-48.96886,-20.39373],[-48.99221,-20.16537],[-49.06744,-20.15473],[-49.12068,-20.27052],[-49.17252,-20.31252],[-49.22863,-20.30334],[-49.3085,-20.10388],[-49.24961,-19.96954],[-49.44448,-19.9805],[-49.55144,-19.90583],[-49.89117,-19.94357],[-50.1031,-19.87472],[-50.33739,-19.86947],[-50.47188,-19.77965],[-50.5745,-19.81449],[-50.65806,-19.90714],[-50.90809,-19.99425],[-51.00043,-20.08537],[-51.0459,-19.72904],[-50.98751,-19.5895],[-50.92309,-19.57605],[-50.96323,-19.48363],[-50.82665,-19.48746],[-50.87587,-19.41991],[-50.81467,-19.28578],[-50.67436,-19.13619],[-50.54335,-19.10592],[-50.4987,-19.03402],[-50.50878,-18.93573],[-50.30622,-18.69547],[-50.08,-18.67224],[-50.01386,-18.59947],[-49.78364,-18.64133],[-49.76849,-18.61098],[-49.64265,-18.60133],[-49.65325,-18.56382],[-49.55892,-18.54676],[-49.53387,-18.49251],[-49.49719,-18.4926],[-49.48179,-18.56102],[-49.39152,-18.6466],[-49.24877,-18.52291],[-49.19254,-18.44972],[-49.20563,-18.41179],[-49.12624,-18.3825],[-49.077,-18.41678],[-48.93673,-18.3062],[-48.81632,-18.37966],[-48.56137,-18.32388],[-48.47902,-18.38012],[-48.40568,-18.35634],[-48.31359,-18.38484],[-48.26237,-18.33188],[-48.11782,-18.42499],[-47.98087,-18.44286],[-47.9549,-18.50028],[-47.83454,-18.45127],[-47.81054,-18.40011],[-47.74945,-18.41255],[-47.74053,-18.36384],[-47.62615,-18.3343],[-47.63873,-18.28768],[-47.56753,-18.20386],[-47.5362,-18.19487],[-47.5337,-18.23071],[-47.28317,-18.05824],[-47.37297,-17.831],[-47.26576,-17.61045],[-47.33311,-17.52345],[-47.49776,-17.52465],[-47.53879,-17.3888],[-47.51118,-17.32558],[-47.44153,-17.34839],[-47.43442,-17.27622],[-47.36261,-17.22271],[-47.35195,-17.1665],[-47.18206,-17.06396],[-47.1259,-16.98177],[-47.19706,-16.91084],[-47.25076,-16.66601],[-47.41284,-16.57587],[-47.46003,-16.5051],[-47.32796,-16.24923],[-47.3527,-16.13284],[-47.30047,-16.01706],[-47.21949,-16.01425],[-47.14192,-15.92665],[-47.08905,-15.96173],[-46.80609,-15.87201],[-46.85485,-15.61831],[-46.94865,-15.55786],[-46.92928,-15.43995],[-46.83586,-15.32655],[-46.89145,-15.23527],[-46.94083,-15.23071],[-46.88897,-15.11113],[-46.91802,-15.04903],[-46.82984,-15.00928],[-46.64099,-15.0876],[-46.51384,-15.06132],[-46.5465,-14.92546],[-46.52152,-14.85119],[-46.56586,-14.78639],[-46.47653,-14.70511],[-46.32217,-14.8146],[-46.28683,-14.92816],[-46.17552,-14.94944],[-46.16139,-14.90619],[-46.09819,-14.939],[-46.03755,-14.87467],[-46.00385,-14.90231],[-45.97547,-15.0389],[-46.05425,-15.17199],[-46.11922,-15.19224],[-46.05808,-15.26276],[-45.95372,-15.13955],[-45.87346,-15.15814],[-45.82666,-15.11697],[-45.7216,-15.11225],[-45.56754,-14.94585],[-45.45511,-14.95374],[-45.20555,-14.74476],[-45.08275,-14.74879],[-45.0403,-14.68051],[-44.84337,-14.58086],[-44.83322,-14.49985],[-44.56555,-14.34009],[-44.31647,-14.24038],[-44.20903,-14.23726]]]}},{"type":"Feature","id":"MS","properties":{"SIGLA":"MS","Estado":"Mato Grosso do Sul"},"geometry":{"type":"Polygon","coordinates":[[[-56.44407,-17.33163],[-56.04568,-17.17097],[-55.93815,-17.27853],[-55.88648,-17.26391],[-55.70129,-17.35414],[-55.64302,-17.33938],[-55.52404,-17.48119],[-55.29261,-17.54256],[-55.1272,-17.65273],[-54.86081,-17.62386],[-54.75873,-17.5657],[-54.75031,-17.52344],[-54.58161,-17.46801],[-54.48126,-17.48767],[-54.33553,-17.66173],[-54.17878,-17.60285],[-54.08464,-17.61904],[-54.03759,-17.48645],[-53.95219,-17.45934],[-53.83127,-17.35502],[-53.82002,-17.29462],[-53.70509,-17.22816],[-53.68002,-17.25357],[-53.70456,-17.66351],[-53.85586,-17.70272],[-53.95497,-17.87125],[-53.94886,-17.92343],[-53.86,-17.92116],[-53.77417,-18.00077],[-53.69214,-18.01368],[-53.61608,-17.97619],[-53.48602,-18.0404],[-53.43897,-17.98393],[-53.07223,-18.03397],[-53.14319,-18.08184],[-53.06989,-18.34288],[-52.93518,-18.29655],[-52.75905,-18.35057],[-52.9623,-18.54082],[-52.91272,-18.64198],[-52.74743,-18.69202],[-52.59282,-18.68989],[-52.55299,-18.65698],[-52.44907,-18.69096],[-52.33407,-18.82824],[-52.18143,-18.84791],[-52.08132,-18.9509],[-51.94038,-18.96786],[-51.8549,-19.05147],[-51.6579,-19.13791],[-51.42043,-19.16653],[-51.25217,-19.27407],[-51.08792,-19.30832],[-51.03068,-19.40143],[-50.93586,-19.46064],[-50.9629,-19.50057],[-50.92309,-19.57605],[-50.98751,-19.5895],[-51.0459,-19.72904],[-51.00043,-20.08537],[-51.05822,-20.23318],[-51.13237,-20.29552],[-51.34129,-20.35368],[-51.5941,-20.64383],[-51.63486,-20.75484],[-51.62354,-20.94442],[-51.72293,-20.97742],[-51.78963,-21.1021],[-51.87582,-21.13613],[-51.86102,-21.33666],[-51.96786,-21.50175],[-52.06641,-21.50458],[-52.09592,-21.54205],[-52.04948,-21.64307],[-52.07645,-21.71486],[-52.16122,-21.77076],[-52.4898,-22.21828],[-52.99854,-22.49316],[-53.19896,-22.72831],[-53.35199,-22.77566],[-53.60738,-22.95131],[-53.63655,-23.12501],[-53.7308,-23.31979],[-53.77107,-23.37211],[-53.9815,-23.45683],[-54.10173,-23.95194],[-54.28929,-24.06835],[-54.43754,-23.90664],[-54.67143,-23.81219],[-54.70571,-23.86336],[-54.88388,-23.89598],[-54.92477,-23.95952],[-55.06049,-23.99276],[-55.10708,-23.96127],[-55.2283,-24.01355],[-55.30451,-23.96383],[-55.34366,-23.99595],[-55.41928,-23.95946],[-55.43486,-23.71779],[-55.48205,-23.64114],[-55.53744,-23.62579],[-55.5621,-23.48289],[-55.50574,-23.37748],[-55.55647,-23.31919],[-55.52394,-23.19704],[-55.54257,-23.15588],[-55.59744,-23.15266],[-55.66621,-22.85214],[-55.61737,-22.76813],[-55.61368,-22.65739],[-55.7219,-22.55198],[-55.74029,-22.39508],[-55.78885,-22.3883],[-55.85343,-22.28035],[-56.21074,-22.27725],[-56.36353,-22.17044],[-56.39291,-22.07485],[-56.50649,-22.09676],[-56.63559,-22.26294],[-56.70347,-22.21822],[-56.72072,-22.2635],[-56.79408,-22.24256],[-56.84219,-22.30255],[-56.88497,-22.24692],[-56.99681,-22.22328],[-57.08073,-22.25067],[-57.20142,-22.21057],[-57.24112,-22.24983],[-57.37603,-22.23195],[-57.57829,-22.17659],[-57.61309,-22.09502],[-57.70668,-22.09021],[-57.80368,-22.1509],[-57.99204,-22.09084],[-58.00588,-22.0504],[-57.91281,-21.87961],[-57.97048,-21.84641],[-57.91386,-21.79192],[-57.94609,-21.74293],[-57.88253,-21.68849],[-57.93356,-21.65151],[-57.91303,-21.59043],[-57.96684,-21.52555],[-57.85453,-21.33879],[-57.92173,-21.2811],[-57.85017,-21.21985],[-57.86792,-21.04141],[-57.81944,-20.94265],[-57.92849,-20.89821],[-57.85972,-20.82695],[-57.96108,-20.78932],[-57.93336,-20.746],[-57.86322,-20.7418],[-57.92113,-20.66358],[-57.98547,-20.70177],[-58.01888,-20.51391],[-57.99545,-20.44073],[-58.08436,-20.37272],[-58.09477,-20.2556],[-58.16156,-20.2625],[-58.12239,-20.1952],[-58.16753,-20.17186],[-57.95922,-20.02227],[-57.9023,-20.04213],[-57.85947,-19.97037],[-58.13177,-19.75852],[-57.78421,-19.03353],[-57.69433,-19.01098],[-57.71979,-18.89908],[-57.76663,-18.8993],[-57.55778,-18.24037],[-57.45387,-18.23121],[-57.57481,-18.13191],[-57.72061,-17.82788],[-57.68317,-17.80802],[-57.79562,-17.56003],[-57.71209,-17.54315],[-57.68456,-17.7165],[-57.64025,-17.71738],[-57.60583,-17.80236],[-57.45242,-17.90266],[-57.3774,-17.8262],[-57.11555,-17.7811],[-57.04486,-17.73034],[-56.95794,-17.60805],[-56.9823,-17.58025],[-56.87591,-17.53299],[-56.82866,-17.38998],[-56.78593,-17.38754],[-56.7338,-17.30988],[-56.64541,-17.33936],[-56.50171,-17.28634],[-56.44407,-17.33163]]]}},{"type":"Feature","id":"MT","properties":{"SIGLA":"MT","Estado":"Mato Grosso"},"geometry":{"type":"Polygon","coordinates":[[[-57.68687,-8.41626],[-57.59287,-8.75649],[-57.42751,-8.78325],[-57.38431,-8.87957],[-57.18879,-8.93316],[-57.03917,-9.0983],[-57.06006,-9.18198],[-56.99548,-9.23385],[-56.82004,-9.24635],[-56.7542,-9.40637],[-56.67181,-9.3674],[-50.22477,-9.84116],[-50.3026,-10.03542],[-50.39211,-10.13404],[-50.41824,-10.35629],[-50.47368,-10.40491],[-50.53738,-10.60522],[-50.60335,-10.66001],[-50.57054,-10.75251],[-50.62196,-10.83942],[-50.60941,-11.06745],[-50.65861,-11.12882],[-50.65599,-11.24024],[-50.73921,-11.43539],[-50.74094,-11.53724],[-50.65743,-11.59281],[-50.66366,-11.67753],[-50.722,-11.7397],[-50.68626,-11.859],[-50.63924,-11.88457],[-50.68219,-11.99089],[-50.68682,-12.20216],[-50.64366,-12.22297],[-50.61821,-12.429],[-50.68643,-12.56921],[-50.66793,-12.59601],[-50.7061,-12.61352],[-50.69964,-12.64778],[-50.64688,-12.65112],[-50.62268,-12.81974],[-50.58449,-12.79978],[-50.5978,-12.82678],[-50.52029,-12.8364],[-50.50182,-12.8827],[-50.52588,-12.97642],[-50.59296,-13.00286],[-50.56913,-13.03913],[-50.61166,-13.06436],[-50.60716,-13.31086],[-50.6656,-13.37614],[-50.66541,-13.44228],[-50.76341,-13.53001],[-50.80391,-13.69097],[-50.87174,-13.73309],[-50.8336,-14.08985],[-50.91744,-14.11482],[-50.97522,-14.29085],[-50.99902,-14.41923],[-50.9628,-14.52702],[-51.08939,-14.84036],[-51.08401,-14.91645],[-51.24255,-15.0354],[-51.29044,-15.03433],[-51.33775,-14.97337],[-51.53591,-15.06969],[-51.65205,-15.17987],[-51.69874,-15.48496],[-51.72767,-15.55079],[-51.7775,-15.539],[-51.75776,-15.63595],[-51.87956,-15.82539],[-51.94878,-15.81039],[-52.01126,-15.88623],[-52.25258,-15.89266],[-52.32803,-16.0706],[-52.52548,-16.14056],[-52.56965,-16.22885],[-52.54763,-16.26238],[-52.6821,-16.30407],[-52.68834,-16.3963],[-52.60434,-16.46466],[-52.6357,-16.55163],[-52.74026,-16.59042],[-52.71565,-16.63926],[-52.74681,-16.63071],[-52.78593,-16.74119],[-52.93259,-16.80287],[-52.95054,-16.86041],[-53.01127,-16.85802],[-53.05596,-17.07037],[-53.11907,-17.11322],[-53.21912,-17.29961],[-53.19688,-17.38045],[-53.23338,-17.44093],[-53.24683,-17.69068],[-53.16857,-17.76626],[-53.07223,-18.03397],[-53.43897,-17.98393],[-53.48602,-18.0404],[-53.61608,-17.97619],[-53.69214,-18.01368],[-53.77417,-18.00077],[-53.86,-17.92116],[-53.94886,-17.92343],[-53.95497,-17.87125],[-53.85586,-17.70272],[-53.70456,-17.66351],[-53.68002,-17.25357],[-53.70509,-17.22816],[-53.82002,-17.29462],[-53.83127,-17.35502],[-53.95219,-17.45934],[-54.03759,-17.48645],[-54.08464,-17.61904],[-54.17878,-17.60285],[-54.33553,-17.66173],[-54.48126,-17.48767],[-54.58161,-17.46801],[-54.75031,-17.52344],[-54.75873,-17.5657],[-54.86081,-17.62386],[-55.1272,-17.65273],[-55.29261,-17.54256],[-55.52404,-17.48119],[-55.64302,-17.33938],[-55.70129,-17.35414],[-55.88648,-17.26391],[-55.93815,-17.27853],[-56.04568,-17.17097],[-56.1139,-17.16664],[-56.44407,-17.33163],[-56.50171,-17.28634],[-56.64541,-17.33936],[-56.7338,-17.30988],[-56.78593,-17.38754],[-56.82866,-17.38998],[-56.87591,-17.53299],[-56.9823,-17.58025],[-56.95794,-17.60805],[-57.04486,-17.73034],[-57.11555,-17.7811],[-57.3774,-17.8262],[-57.45242,-17.90266],[-57.60583,-17.80236],[-57.64025,-17.71738],[-57.68456,-17.7165],[-57.66878,-17.67193],[-57.728,-17.52977],[-57.75224,-17.56448],[-57.88339,-17.44956],[-57.99631,-17.51563],[-58.04333,-17.49269],[-58.15153,-17.38467],[-58.25464,-17.35257],[-58.39695,-17.18158],[-58.42298,-16.98931],[-58.47436,-16.93498],[-58.47053,-16.70305],[-58.43603,-16.59238],[-58.33369,-16.49004],[-58.3562,-16.42807],[-58.30709,-16.37021],[-58.32103,-16.26477],[-58.38864,-16.26103],[-58.43042,-16.32132],[-60.17169,-16.26584],[-60.23855,-15.47389],[-60.56496,-15.10872],[-60.24426,-15.09684],[-60.27209,-14.62033],[-60.32107,-14.60891],[-60.49237,-14.18826],[-60.47987,-14.0978],[-60.38203,-13.98749],[-60.45087,-13.93759],[-60.45675,-13.85391],[-60.49273,-13.85634],[-60.46723,-13.79576],[-60.71696,-13.68515],[-60.63214,-13.57173],[-60.38786,-13.45472],[-60.36491,-13.29997],[-60.26845,-13.145],[-60.28293,-13.08058],[-60.18922,-12.97117],[-60.11711,-12.95958],[-60.07905,-12.88117],[-60.06867,-12.61602],[-59.93516,-12.48711],[-59.85579,-12.47863],[-59.77429,-12.34098],[-59.89227,-12.24548],[-59.89974,-12.11725],[-59.97929,-12.02957],[-59.98573,-11.9121],[-60.10871,-11.8396],[-60.11482,-11.59148],[-59.92584,-11.4155],[-59.91712,-11.33846],[-59.98196,-11.23352],[-59.98383,-11.1163],[-60.19893,-11.11411],[-60.30118,-11.05717],[-60.3479,-11.10906],[-60.44653,-11.04217],[-60.45998,-10.98988],[-61.5502,-10.98612],[-61.51102,-10.79484],[-61.47377,-10.79754],[-61.49875,-10.69996],[-61.46172,-10.41993],[-61.57666,-10.2486],[-61.5586,-10.17652],[-61.60151,-10.15667],[-61.5084,-9.87299],[-61.53143,-9.73993],[-61.57457,-9.71781],[-61.5052,-9.6914],[-61.47673,-9.62874],[-61.54948,-9.51524],[-61.52573,-9.50182],[-61.5769,-9.4745],[-61.55243,-9.38835],[-61.62585,-9.36303],[-61.63327,-9.27062],[-61.52478,-9.24287],[-61.55567,-9.09242],[-61.46842,-8.91738],[-61.5235,-8.81861],[-61.58305,-8.7987],[-58.4158,-8.79285],[-58.32615,-8.72004],[-58.43723,-8.70336],[-58.38944,-8.5948],[-58.41698,-8.49197],[-58.31482,-8.32307],[-58.32239,-8.17005],[-58.28709,-8.12855],[-58.38466,-7.844],[-58.20209,-7.62086],[-58.21298,-7.45854],[-58.13836,-7.34904],[-58.06033,-7.39703],[-57.97251,-7.5346],[-57.93948,-7.65314],[-57.89887,-7.6719],[-57.82876,-7.9729],[-57.64314,-8.21218],[-57.68687,-8.41626]]]}},{"type":"Feature","id":"PA","properties":{"SIGLA":"PA","Estado":"Pará"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-46.07252,-1.06159],[-46.06666,-1.1052],[-46.08833,-1.06131],[-46.07252,-1.06159]]],[[[-46.43767,-1.00146],[-46.44647,-1.01037],[-46.45094,-1.00944],[-46.43767,-1.00146]]],[[[-46.3708,-0.99143],[-46.3848,-1.00036],[-46.37456,-0.98756],[-46.3708,-0.99143]]],[[[-46.46841,-0.96433],[-46.47541,-0.96457],[-46.46925,-0.95806],[-46.46841,-0.96433]]],[[[-46.3613,-0.95143],[-46.36704,-0.96174],[-46.36652,-0.95042],[-46.3613,-0.95143]]],[[[-46.31696,-0.94647],[-46.32709,-0.95064],[-46.32305,-0.94529],[-46.31696,-0.94647]]],[[[-46.1983,-0.94007],[-46.2068,-0.94114],[-46.20313,-0.93671],[-46.1983,-0.94007]]],[[[-46.26619,-0.93323],[-46.26752,-0.93685],[-46.27056,-0.93385],[-46.26619,-0.93323]]],[[[-46.41641,-0.93259],[-46.41222,-0.97096],[-46.43747,-0.98659],[-46.41641,-0.93259]]],[[[-46.60687,-0.85365],[-46.60171,-0.85974],[-46.60857,-0.85925],[-46.60687,-0.85365]]],[[[-46.60648,-0.82557],[-46.60133,-0.83166],[-46.60819,-0.83116],[-46.60648,-0.82557]]],[[[-47.03264,-0.71834],[-47.04339,-0.72237],[-47.04195,-0.718],[-47.03264,-0.71834]]],[[[-47.01725,-0.71351],[-47.01919,-0.71792],[-47.02219,-0.71581],[-47.01725,-0.71351]]],[[[-47.03391,-0.7083],[-47.02355,-0.69704],[-47.00544,-0.7007],[-47.03391,-0.7083]]],[[[-47.83175,-0.65931],[-47.84167,-0.65805],[-47.83347,-0.65412],[-47.83175,-0.65931]]],[[[-54.95424,2.5836],[-54.76305,2.20223],[-54.81277,2.06272],[-54.75438,1.97051],[-54.74472,1.77571],[-54.30925,1.74096],[-54.19105,1.62482],[-54.18216,1.65799],[-54.14361,1.64053],[-54.08641,1.48821],[-54.00899,1.51958],[-53.99741,1.47569],[-53.92871,1.4691],[-53.93825,1.43538],[-53.89594,1.45504],[-53.85029,1.39176],[-53.80394,1.42344],[-53.73227,1.38621],[-53.71713,1.43168],[-53.6705,1.37191],[-53.64678,1.40754],[-53.65091,1.33614],[-53.54211,1.34476],[-53.53894,1.21235],[-53.42647,1.24264],[-53.39983,1.15833],[-53.45911,1.1336],[-53.41144,0.92927],[-53.10625,0.68391],[-53.174,0.37424],[-52.97725,0.02373],[-52.9327,-0.14237],[-52.85552,-0.14811],[-52.68878,-0.30419],[-52.70264,-0.45929],[-52.63997,-0.58496],[-52.52314,-0.58913],[-52.49784,-0.7308],[-52.53862,-0.85486],[-52.45548,-0.83032],[-52.40312,-0.8772],[-52.43946,-0.9356],[-52.427,-1.05084],[-52.35812,-1.06263],[-52.33295,-1.11594],[-52.12023,-1.14624],[-52.10204,-1.22601],[-52.05461,-1.23426],[-52.06394,-1.18598],[-51.99109,-1.17186],[-51.98646,-1.12199],[-51.95584,-1.16972],[-51.92592,-1.13236],[-51.80906,-1.15715],[-51.70039,-1.06326],[-51.66545,-0.76194],[-51.45776,-0.51328],[-51.21633,-0.11866],[-51.08489,-0.08759],[-50.93414,0.06963],[-50.67564,0.17913],[-50.55612,0.30667],[-50.43652,0.60006],[-50.34721,0.6668],[-50.15695,0.70513],[-50.09389,0.70205],[-50.04078,0.57349],[-50.06123,0.33839],[-49.90331,0.31096],[-49.76314,0.34903],[-49.73896,0.32251],[-49.67761,0.36593],[-49.39695,0.01572],[-48.9307,-0.22595],[-48.41225,-0.25726],[-48.39048,-0.3633],[-48.47259,-0.49909],[-48.00041,-0.66431],[-47.99697,-0.70479],[-47.89759,-0.55283],[-47.84209,-0.58667],[-47.87119,-0.67747],[-47.84097,-0.68024],[-47.82225,-0.66508],[-47.80859,-0.55288],[-47.76648,-0.63827],[-47.76373,-0.56549],[-47.70391,-0.53527],[-47.69875,-0.59312],[-47.63651,-0.60373],[-47.62645,-0.70143],[-47.57944,-0.58138],[-47.52544,-0.62566],[-47.41919,-0.59205],[-47.40694,-0.65747],[-47.3225,-0.59419],[-47.28286,-0.60703],[-47.29322,-0.64582],[-47.21253,-0.63756],[-47.24374,-0.70727],[-47.15875,-0.67001],[-47.17778,-0.74394],[-47.17011,-0.77653],[-47.14131,-0.68776],[-47.08869,-0.66266],[-47.08967,-0.74928],[-47.06669,-0.77652],[-47.04992,-0.71154],[-47.05772,-0.80628],[-47.02842,-0.80287],[-47.04111,-0.72933],[-46.9577,-0.71342],[-46.97959,-0.78262],[-46.93527,-0.79028],[-46.93544,-0.8766],[-46.85778,-0.73628],[-46.82069,-0.90291],[-46.80177,-0.86226],[-46.78889,-0.90039],[-46.76636,-0.8175],[-46.74269,-0.92098],[-46.7216,-0.8271],[-46.67405,-0.86164],[-46.68571,-0.80685],[-46.63811,-0.78829],[-46.61056,-0.84971],[-46.6452,-0.86885],[-46.63469,-0.92861],[-46.68372,-0.95226],[-46.67533,-0.97625],[-46.60178,-0.95135],[-46.58516,-0.97663],[-46.55103,-0.90396],[-46.53716,-0.97765],[-46.511,-0.88428],[-46.42699,-0.85793],[-46.50028,-0.97339],[-46.46436,-0.97609],[-46.48666,-1.00035],[-46.46867,-1.05334],[-46.46122,-1.01754],[-46.41289,-1.0198],[-46.41258,-1.03984],[-46.43025,-1.0532],[-46.42875,-1.06541],[-46.40661,-1.04408],[-46.41033,-1.01781],[-46.42413,-1.00638],[-46.39044,-0.98716],[-46.37535,-1.05231],[-46.34466,-0.99661],[-46.34857,-1.07169],[-46.30555,-1.08436],[-46.3173,-1.01436],[-46.30105,-1.0552],[-46.2838,-1.04793],[-46.30727,-1.00944],[-46.26772,-0.99617],[-46.26507,-0.91784],[-46.20466,-0.88637],[-46.1871,-0.93329],[-46.21013,-0.93018],[-46.20155,-0.96492],[-46.25244,-0.99771],[-46.21505,-0.97969],[-46.20905,-1.05867],[-46.2618,-1.1231],[-46.27283,-1.172],[-46.17158,-0.99285],[-46.15611,-1.04972],[-46.20538,-1.13158],[-46.17271,-1.15547],[-46.16577,-1.08347],[-46.14886,-1.13524],[-46.15108,-1.07432],[-46.07293,-1.01858],[-46.10072,-1.07097],[-46.0733,-1.1109],[-46.09817,-1.19608],[-46.15393,-1.26951],[-46.09527,-1.32714],[-46.16954,-1.47037],[-46.13128,-1.57857],[-46.15367,-1.67764],[-46.32398,-1.76556],[-46.31113,-1.8039],[-46.20862,-1.8309],[-46.27591,-2.15872],[-46.37783,-2.2535],[-46.42821,-2.23975],[-46.46591,-2.36937],[-46.40752,-2.3687],[-46.44013,-2.41282],[-46.4187,-2.52797],[-46.49433,-2.53652],[-46.505,-2.61493],[-46.59375,-2.66539],[-46.62192,-2.64454],[-46.66395,-2.69421],[-46.63664,-2.79369],[-46.57581,-2.8414],[-46.61394,-2.82897],[-46.63759,-2.88323],[-46.67965,-2.88175],[-46.64803,-2.96171],[-46.6758,-3.09367],[-46.74782,-3.12089],[-46.71964,-3.15394],[-46.77188,-3.17724],[-46.75479,-3.21573],[-46.80618,-3.24449],[-46.81337,-3.30956],[-46.94463,-3.3782],[-46.94797,-3.47646],[-47.03746,-3.56135],[-47.08874,-3.8616],[-47.17314,-3.92384],[-47.19025,-3.98372],[-47.31942,-4.04779],[-47.37176,-4.2456],[-47.47473,-4.31506],[-47.61193,-4.55737],[-47.67995,-4.60863],[-47.79256,-4.58643],[-48.75513,-5.34918],[-48.59672,-5.42256],[-48.38159,-5.396],[-48.30069,-5.52302],[-48.13814,-5.6027],[-48.17358,-5.70996],[-48.2935,-5.75121],[-48.23161,-5.94615],[-48.3347,-6.00429],[-48.28577,-6.07041],[-48.30384,-6.1169],[-48.43192,-6.17731],[-48.3813,-6.37564],[-48.5002,-6.35094],[-48.61313,-6.45374],[-48.66483,-6.66278],[-48.77798,-6.7363],[-49.01886,-6.78422],[-49.2095,-6.92543],[-49.18526,-7.23518],[-49.38393,-7.54617],[-49.32805,-7.6778],[-49.14841,-7.80732],[-49.20955,-8.1755],[-49.28329,-8.37957],[-49.37395,-8.45123],[-49.40986,-8.58001],[-49.59238,-8.8395],[-49.74505,-8.90617],[-49.90766,-9.17383],[-50.03775,-9.28941],[-50.1053,-9.47542],[-50.08952,-9.5457],[-50.22477,-9.84116],[-56.67181,-9.3674],[-56.76104,-9.40501],[-56.82004,-9.24635],[-56.99548,-9.23385],[-57.06006,-9.18197],[-57.03917,-9.0983],[-57.2037,-8.92083],[-57.38431,-8.87957],[-57.42751,-8.78325],[-57.59287,-8.75649],[-57.68687,-8.41626],[-57.64314,-8.21218],[-57.82876,-7.9729],[-57.89887,-7.6719],[-57.93948,-7.65314],[-58.06033,-7.39703],[-58.16993,-7.31323],[-58.18221,-7.17982],[-58.46281,-6.83729],[-58.46511,-6.67005],[-58.25518,-6.45413],[-56.40209,-2.45665],[-56.46552,-2.42315],[-56.38424,-2.27051],[-56.22022,-2.19588],[-56.09749,-2.03717],[-56.2283,-2.05626],[-56.41381,-2.17599],[-56.52699,-2.13923],[-56.61284,-2.21021],[-56.679,-2.2126],[-56.76846,-2.16579],[-56.73347,-2.02283],[-56.84988,-2.02021],[-56.98827,-1.9072],[-57.03713,-1.91159],[-57.06382,-1.80853],[-57.16313,-1.76862],[-57.16482,-1.72091],[-57.25381,-1.69633],[-57.39261,-1.72311],[-57.40338,-1.67556],[-57.5287,-1.65276],[-57.6025,-1.57033],[-57.67859,-1.59437],[-57.71308,-1.50418],[-57.7836,-1.50804],[-57.82395,-1.43979],[-57.96026,-1.40161],[-58.00085,-1.33021],[-57.97193,-1.1626],[-58.01745,-1.10619],[-58.08177,-1.12652],[-58.16255,-1.22969],[-58.25468,-1.13262],[-58.32319,-1.14333],[-58.42999,-1.02715],[-58.43596,-0.8835],[-58.70508,-0.67901],[-58.72512,-0.44226],[-58.87197,-0.34303],[-58.89539,1.22873],[-58.82123,1.171],[-58.73956,1.1999],[-58.69456,1.29732],[-58.49623,1.26796],[-58.45787,1.37145],[-58.50512,1.40317],[-58.50873,1.46295],[-58.38559,1.46999],[-58.39472,1.52651],[-58.32237,1.59701],[-58.236,1.54669],[-58.16064,1.56011],[-58.12943,1.49889],[-58.00423,1.50303],[-57.99009,1.65844],[-57.85206,1.66782],[-57.77431,1.72973],[-57.70509,1.73093],[-57.65042,1.68237],[-57.55323,1.69244],[-57.43776,1.82681],[-57.43343,1.90598],[-57.30712,1.99665],[-57.22923,1.93759],[-57.08668,2.02644],[-57.01421,1.91489],[-56.91971,1.93036],[-56.79793,1.85336],[-56.72096,1.92581],[-56.62145,1.94588],[-56.57976,1.90588],[-56.45126,1.95614],[-55.98321,1.83552],[-55.90385,1.88803],[-55.93635,1.98647],[-55.9031,2.04108],[-56.05505,2.18464],[-56.04288,2.22778],[-56.13887,2.26575],[-56.09012,2.37228],[-56.02181,2.34247],[-55.97824,2.52733],[-55.76663,2.45524],[-55.71788,2.40201],[-55.49971,2.44324],[-55.38533,2.41836],[-55.32018,2.51537],[-55.23474,2.50338],[-55.1234,2.56762],[-55.10302,2.52564],[-54.95424,2.5836]]]]}},{"type":"Feature","id":"PB","properties":{"SIGLA":"PB","Estado":"Paraíba"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-34.86565,-7.00475],[-34.84777,-6.98209],[-34.85921,-7.01566],[-34.86565,-7.00475]]],[[[-37.23189,-6.02764],[-37.17397,-6.04806],[-37.15696,-6.15254],[-37.37736,-6.34444],[-37.39726,-6.51464],[-37.48316,-6.55045],[-37.48473,-6.71008],[-37.28375,-6.69396],[-37.23463,-6.82468],[-37.00363,-6.70966],[-36.95717,-6.79052],[-36.83507,-6.73111],[-36.7695,-6.83879],[-36.73023,-6.83644],[-36.76658,-6.93591],[-36.71859,-6.98278],[-36.68725,-6.92601],[-36.60983,-6.93688],[-36.56983,-6.84805],[-36.50553,-6.80887],[-36.50479,-6.75398],[-36.55892,-6.70915],[-36.52195,-6.59713],[-36.4359,-6.62592],[-36.53101,-6.45212],[-36.49989,-6.35821],[-36.43035,-6.3623],[-36.3946,-6.29402],[-36.29502,-6.29216],[-36.29929,-6.3821],[-36.24981,-6.43741],[-36.08006,-6.40524],[-35.97769,-6.48895],[-35.65731,-6.44583],[-35.56484,-6.49183],[-35.4008,-6.49165],[-35.33082,-6.54439],[-35.22351,-6.51628],[-35.17035,-6.55844],[-35.0956,-6.49773],[-35.05779,-6.52784],[-34.97163,-6.48529],[-34.93733,-6.77081],[-34.85367,-6.9004],[-34.86838,-7.00696],[-34.85518,-7.02756],[-34.83027,-6.96666],[-34.84373,-7.05587],[-34.79371,-7.15454],[-34.8262,-7.54782],[-34.96083,-7.5388],[-34.98546,-7.46205],[-35.07907,-7.39635],[-35.24611,-7.37465],[-35.4224,-7.47521],[-35.4754,-7.44494],[-35.53285,-7.65485],[-35.6097,-7.65382],[-35.99826,-7.81379],[-36.21667,-7.76414],[-36.25333,-7.82999],[-36.42408,-7.81581],[-36.44634,-7.91642],[-36.55205,-7.89875],[-36.57682,-7.95968],[-36.61191,-7.94793],[-36.65787,-8.07617],[-36.62865,-8.11109],[-36.99119,-8.30299],[-37.08124,-8.26641],[-37.06745,-8.23014],[-37.16026,-8.17044],[-37.14969,-8.02285],[-37.19271,-7.96063],[-37.3559,-7.9759],[-37.24412,-7.87388],[-37.24369,-7.81904],[-37.15198,-7.77988],[-37.19735,-7.64547],[-37.17326,-7.58794],[-36.98426,-7.4823],[-37.02796,-7.3859],[-37.23364,-7.27444],[-37.33263,-7.28977],[-37.38574,-7.35673],[-37.49807,-7.36688],[-37.5331,-7.47299],[-37.70654,-7.55062],[-37.76727,-7.65752],[-37.79084,-7.62598],[-37.85719,-7.65201],[-37.97136,-7.7776],[-38.03961,-7.74816],[-38.0771,-7.83062],[-38.1627,-7.77999],[-38.28682,-7.83026],[-38.35752,-7.6772],[-38.40953,-7.73053],[-38.59365,-7.75439],[-38.71832,-7.60939],[-38.65535,-7.56558],[-38.64494,-7.46006],[-38.58428,-7.43322],[-38.53461,-7.29383],[-38.62483,-7.19136],[-38.68786,-7.1902],[-38.66987,-7.04774],[-38.76502,-6.99393],[-38.76537,-6.94798],[-38.61395,-6.78248],[-38.67318,-6.69726],[-38.61231,-6.51323],[-38.51794,-6.40877],[-38.60207,-6.38966],[-38.45841,-6.33026],[-38.48626,-6.39879],[-38.40177,-6.4096],[-38.28875,-6.50566],[-38.23926,-6.48034],[-38.11709,-6.52271],[-38.05148,-6.44245],[-38.01298,-6.4689],[-38.01372,-6.42747],[-37.95542,-6.39825],[-37.92815,-6.42164],[-37.81045,-6.2917],[-37.75769,-6.29082],[-37.77565,-6.23139],[-37.7424,-6.18902],[-37.45898,-6.14505],[-37.23189,-6.02764]]]]}},{"type":"Feature","id":"PE","properties":{"SIGLA":"PE","Estado":"Pernambuco"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-35.01194,-8.43733],[-35.02727,-8.45583],[-35.03145,-8.44045],[-35.01194,-8.43733]]],[[[-40.26302,-7.30106],[-40.08747,-7.38313],[-39.85147,-7.32525],[-39.69753,-7.3422],[-39.55285,-7.48069],[-39.45852,-7.47425],[-39.47248,-7.57522],[-39.27243,-7.66017],[-39.25259,-7.70497],[-39.13398,-7.72531],[-39.09132,-7.85822],[-39.01807,-7.8127],[-38.96214,-7.84421],[-38.9396,-7.75881],[-38.71514,-7.62196],[-38.58998,-7.75572],[-38.40953,-7.73053],[-38.35752,-7.6772],[-38.28682,-7.83026],[-38.1627,-7.77999],[-38.0771,-7.83062],[-38.03961,-7.74816],[-37.97136,-7.7776],[-37.85719,-7.65201],[-37.79084,-7.62598],[-37.76727,-7.65752],[-37.70654,-7.55062],[-37.5331,-7.47299],[-37.49807,-7.36688],[-37.38574,-7.35673],[-37.33263,-7.28977],[-37.23364,-7.27444],[-37.02796,-7.3859],[-36.98426,-7.4823],[-37.17326,-7.58794],[-37.19735,-7.64547],[-37.15198,-7.77988],[-37.24369,-7.81904],[-37.24412,-7.87388],[-37.3559,-7.9759],[-37.19271,-7.96063],[-37.14969,-8.02285],[-37.16026,-8.17044],[-37.06745,-8.23014],[-37.08124,-8.26641],[-36.99119,-8.30299],[-36.62865,-8.11109],[-36.65787,-8.07617],[-36.61191,-7.94793],[-36.57682,-7.95968],[-36.55205,-7.89875],[-36.44634,-7.91642],[-36.42408,-7.81581],[-36.25333,-7.82999],[-36.21667,-7.76414],[-35.99826,-7.81379],[-35.6097,-7.65382],[-35.53285,-7.65485],[-35.50334,-7.45612],[-35.4224,-7.47521],[-35.24611,-7.37465],[-35.07039,-7.39892],[-34.98546,-7.46205],[-34.96083,-7.5388],[-34.83166,-7.55196],[-34.81969,-7.9178],[-34.9402,-8.25454],[-34.94064,-8.35181],[-35.05739,-8.46564],[-35.02217,-8.50837],[-34.98577,-8.46075],[-35.00478,-8.56601],[-35.07116,-8.65549],[-35.15263,-8.91373],[-35.47017,-8.81316],[-35.74844,-8.91718],[-35.79511,-8.84772],[-35.89624,-8.8543],[-36.12732,-8.95614],[-36.1207,-9.02748],[-36.26688,-9.10241],[-36.22429,-9.17132],[-36.43704,-9.21213],[-36.57188,-9.29261],[-36.5863,-9.33812],[-36.69388,-9.27536],[-36.8713,-9.26812],[-36.95247,-9.38258],[-37.10631,-9.23982],[-37.16437,-9.27574],[-37.19835,-9.21552],[-37.23436,-9.24008],[-37.31555,-9.09431],[-37.4694,-9.02771],[-37.49037,-8.96594],[-37.58364,-8.9702],[-37.64392,-9.01838],[-37.69888,-8.9925],[-37.75887,-8.85659],[-37.82912,-8.89381],[-37.83913,-8.97842],[-37.97938,-9.1484],[-38.09232,-9.17276],[-38.23762,-9.32983],[-38.31952,-9.13827],[-38.29643,-9.02289],[-38.34027,-8.99044],[-38.41548,-9.03861],[-38.50354,-8.98145],[-38.47986,-8.85013],[-38.57287,-8.83239],[-38.64074,-8.9873],[-38.70657,-8.8623],[-38.8041,-8.78922],[-38.95397,-8.8041],[-39.04283,-8.73339],[-39.16959,-8.69208],[-39.22855,-8.71035],[-39.28645,-8.5638],[-39.38639,-8.53353],[-39.61005,-8.65725],[-39.692,-8.66426],[-39.67682,-8.78855],[-39.89298,-8.82883],[-39.87348,-8.93408],[-39.95806,-9.04863],[-40.12944,-9.11025],[-40.25581,-9.06326],[-40.35642,-9.37703],[-40.42091,-9.35261],[-40.46777,-9.42061],[-40.52913,-9.41394],[-40.62318,-9.48294],[-40.77677,-9.45449],[-40.75561,-9.31358],[-40.68725,-9.27837],[-40.66741,-9.1591],[-40.80412,-9.09837],[-40.92135,-8.83547],[-41.02126,-8.8436],[-41.03625,-8.78591],[-41.12458,-8.75271],[-41.11373,-8.70383],[-41.28155,-8.73614],[-41.35835,-8.70766],[-41.21781,-8.64606],[-41.00069,-8.40076],[-40.92687,-8.44638],[-40.89318,-8.3574],[-40.83658,-8.38083],[-40.7825,-8.25334],[-40.58952,-8.13784],[-40.54204,-7.83868],[-40.67209,-7.76486],[-40.62204,-7.65922],[-40.70465,-7.4586],[-40.54848,-7.39273],[-40.52374,-7.31848],[-40.3939,-7.3682],[-40.26302,-7.30106]]],[[[-35.5584,-8.35427],[-35.56367,-8.35579],[-35.54939,-8.35168],[-35.5584,-8.35427]]],[[[-35.82448,-8.37871],[-35.8223,-8.38291],[-35.82448,-8.37871],[-35.82448,-8.37871]]],[[[-35.81992,-8.38393],[-35.81371,-8.38659],[-35.81992,-8.38393],[-35.81992,-8.38393]]],[[[-35.48859,-8.49594],[-35.48633,-8.49864],[-35.48859,-8.49594],[-35.48859,-8.49594]]],[[[-35.82045,-8.52681],[-35.82045,-8.52682],[-35.82106,-8.52431],[-35.82045,-8.52681]]],[[[-35.64129,-8.53683],[-35.64129,-8.53683],[-35.6413,-8.53496],[-35.64129,-8.53683]]],[[[-35.8037,-8.55521],[-35.80703,-8.56514],[-35.80912,-8.57135],[-35.80471,-8.55822],[-35.80346,-8.55452],[-35.8037,-8.55521]]],[[[-35.67913,-8.61135],[-35.67913,-8.61135],[-35.67497,-8.60407],[-35.67913,-8.61135]]],[[[-35.67945,-8.61191],[-35.68699,-8.62511],[-35.67944,-8.6119],[-35.67945,-8.61191]]],[[[-35.56773,-7.94291],[-35.55921,-7.94061],[-35.56773,-7.94291],[-35.56773,-7.94291]]],[[[-35.51618,-7.92772],[-35.51765,-7.92019],[-35.51678,-7.92465],[-35.51618,-7.92772]]],[[[-35.2257,-7.82861],[-35.22619,-7.82135],[-35.2215,-7.8906],[-35.2257,-7.82861]]],[[[-35.52511,-7.88282],[-35.52732,-7.88069],[-35.52732,-7.88069],[-35.52511,-7.88282]]],[[[-35.69875,-7.8416],[-35.69985,-7.83868],[-35.69985,-7.83868],[-35.69875,-7.8416]]],[[[-35.70057,-7.83677],[-35.70143,-7.83448],[-35.70057,-7.83677],[-35.70057,-7.83677]]],[[[-35.70317,-7.83369],[-35.70788,-7.83155],[-35.70317,-7.83369],[-35.70317,-7.83369]]],[[[-35.74517,-7.81547],[-35.74751,-7.81486],[-35.74751,-7.81486],[-35.74517,-7.81547]]],[[[-35.36431,-7.76787],[-35.37025,-7.7661],[-35.37025,-7.7661],[-35.36431,-7.76787]]],[[[-35.37551,-7.76453],[-35.3757,-7.76448],[-35.37552,-7.76453],[-35.37551,-7.76453]]],[[[-35.38339,-7.76219],[-35.38651,-7.76185],[-35.38651,-7.76185],[-35.38339,-7.76219]]],[[[-32.39684,-3.85248],[-32.42506,-3.87937],[-32.43575,-3.86987],[-32.45773,-3.88163],[-32.47672,-3.87503],[-32.39684,-3.85248]]]]}},{"type":"Feature","id":"PI","properties":{"SIGLA":"PI","Estado":"Piauí"},"geometry":{"type":"Polygon","coordinates":[[[-41.81412,-2.73934],[-41.5935,-2.90553],[-41.32269,-2.92136],[-41.25714,-3.00468],[-41.25666,-3.08805],[-41.42353,-3.36804],[-41.29923,-3.49106],[-41.37126,-3.56783],[-41.34176,-3.68081],[-41.23935,-3.71268],[-41.30083,-3.82633],[-41.22025,-3.94154],[-41.25674,-4.03488],[-41.1143,-4.0406],[-41.14319,-4.12428],[-41.0909,-4.17037],[-41.13592,-4.23355],[-41.12019,-4.3306],[-41.24255,-4.57155],[-41.17392,-4.66858],[-41.24955,-4.75608],[-41.20803,-4.7768],[-41.24959,-4.86972],[-41.22138,-4.93826],[-41.12262,-5.0081],[-41.13189,-5.04809],[-40.92531,-5.18213],[-40.91157,-5.40741],[-40.94248,-5.41974],[-40.9013,-5.6119],[-40.93879,-5.67337],[-40.87365,-5.96679],[-40.90756,-6.04469],[-40.87286,-6.05666],[-40.85224,-6.22425],[-40.78203,-6.3166],[-40.79214,-6.51289],[-40.73226,-6.65362],[-40.47391,-6.73491],[-40.43242,-6.81286],[-40.37053,-6.8031],[-40.42854,-6.86429],[-40.40627,-7.00536],[-40.48976,-7.11872],[-40.54848,-7.39273],[-40.71495,-7.48055],[-40.62204,-7.65922],[-40.67209,-7.76486],[-40.54204,-7.83868],[-40.58952,-8.13784],[-40.7825,-8.25334],[-40.81934,-8.36239],[-40.89318,-8.3574],[-40.92687,-8.44638],[-41.00069,-8.40076],[-41.21781,-8.64606],[-41.38127,-8.70731],[-41.54453,-8.96058],[-41.72362,-9.01358],[-41.7352,-9.09474],[-41.8383,-9.24243],[-42.31258,-9.31619],[-42.49112,-9.49315],[-42.58649,-9.49681],[-42.62173,-9.54137],[-42.72087,-9.53134],[-42.76517,-9.61678],[-42.82056,-9.61926],[-42.83714,-9.55041],[-42.94625,-9.51845],[-42.97026,-9.40955],[-43.05695,-9.41917],[-43.11962,-9.37193],[-43.27877,-9.42461],[-43.42783,-9.27262],[-43.48546,-9.26532],[-43.76973,-9.44208],[-43.84969,-9.54808],[-43.78492,-9.76242],[-43.65324,-9.83909],[-43.70903,-9.91325],[-43.66228,-10.00407],[-43.69291,-10.07743],[-43.76858,-10.0857],[-43.78481,-10.19171],[-43.91622,-10.42675],[-44.02332,-10.4081],[-44.07282,-10.58109],[-44.13131,-10.63492],[-44.26401,-10.6247],[-44.33309,-10.5488],[-44.50451,-10.65077],[-44.57777,-10.6268],[-44.66661,-10.68237],[-44.66677,-10.75891],[-44.75122,-10.78217],[-44.84429,-10.90006],[-44.93121,-10.92878],[-45.07322,-10.84045],[-45.24804,-10.82199],[-45.42978,-10.63411],[-45.44724,-10.50745],[-45.39598,-10.44505],[-45.5673,-10.20151],[-45.58388,-10.11701],[-45.72662,-10.15688],[-45.73384,-10.22862],[-45.79373,-10.26769],[-45.9463,-10.25885],[-45.95528,-10.21846],[-45.87055,-10.08021],[-45.84227,-9.94158],[-45.86598,-9.87167],[-45.82141,-9.77315],[-45.84141,-9.56232],[-45.78313,-9.4798],[-45.82415,-9.37497],[-45.89366,-9.3422],[-45.89706,-9.18654],[-45.99428,-8.92689],[-45.93499,-8.77912],[-45.76563,-8.6096],[-45.79377,-8.58442],[-45.73107,-8.41274],[-45.58146,-8.15606],[-45.48363,-7.71947],[-45.33976,-7.58027],[-44.92444,-7.47031],[-44.81673,-7.36123],[-44.68798,-7.39455],[-44.56424,-7.22758],[-44.38004,-7.11905],[-44.3062,-7.11719],[-44.26198,-6.99875],[-44.05338,-6.76833],[-43.96121,-6.74192],[-43.93048,-6.77108],[-43.715,-6.69876],[-43.42005,-6.84386],[-43.2447,-6.76638],[-43.01733,-6.76094],[-42.91978,-6.67034],[-42.87776,-6.41789],[-42.82934,-6.33745],[-42.8602,-6.33057],[-42.86256,-6.23947],[-43.07501,-6.05473],[-43.04863,-6.00057],[-43.09943,-5.9066],[-43.09901,-5.63321],[-43.03456,-5.59222],[-42.96879,-5.45203],[-42.8267,-5.34805],[-42.79796,-5.18324],[-42.85523,-4.93533],[-42.95033,-4.78978],[-42.92055,-4.73728],[-42.95164,-4.68062],[-42.84937,-4.48262],[-42.89246,-4.40675],[-42.96332,-4.37644],[-42.9892,-4.23408],[-42.88819,-4.15563],[-42.90307,-4.12437],[-42.82181,-3.99058],[-42.72621,-3.91003],[-42.66871,-3.79093],[-42.67576,-3.67562],[-42.50448,-3.45621],[-42.4587,-3.48569],[-42.33082,-3.43389],[-42.20438,-3.4358],[-42.09587,-3.30293],[-42.13164,-3.27537],[-42.00023,-3.24112],[-41.82428,-3.02453],[-41.79662,-2.95985],[-41.86497,-2.87804],[-41.84906,-2.77355],[-41.81412,-2.73934]]]}},{"type":"Feature","id":"RJ","properties":{"SIGLA":"RJ","Estado":"Rio de Janeiro"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-44.71963,-23.35865],[-44.7201,-23.35935],[-44.7206,-23.35859],[-44.71963,-23.35865]]],[[[-44.67158,-23.35619],[-44.6735,-23.35679],[-44.67186,-23.35534],[-44.67158,-23.35619]]],[[[-44.57013,-23.34929],[-44.56722,-23.3497],[-44.57114,-23.35349],[-44.57013,-23.34929]]],[[[-44.57044,-23.26777],[-44.57072,-23.26598],[-44.56772,-23.26499],[-44.57044,-23.26777]]],[[[-44.66855,-23.23926],[-44.67119,-23.24071],[-44.67158,-23.23985],[-44.66855,-23.23926]]],[[[-44.63402,-23.22581],[-44.64953,-23.23122],[-44.64678,-23.22724],[-44.63402,-23.22581]]],[[[-44.55435,-23.222],[-44.55691,-23.22167],[-44.55442,-23.2207],[-44.55435,-23.222]]],[[[-44.58488,-23.21133],[-44.58338,-23.21246],[-44.58497,-23.21236],[-44.58488,-23.21133]]],[[[-44.68983,-23.21486],[-44.69125,-23.21851],[-44.69342,-23.21663],[-44.68983,-23.21486]]],[[[-44.62041,-23.22222],[-44.58639,-23.21009],[-44.61875,-23.22476],[-44.62041,-23.22222]]],[[[-44.70944,-23.20451],[-44.70958,-23.20566],[-44.71064,-23.20477],[-44.70944,-23.20451]]],[[[-44.58089,-23.19898],[-44.57947,-23.2035],[-44.58425,-23.20104],[-44.58089,-23.19898]]],[[[-44.57325,-23.18941],[-44.57697,-23.19034],[-44.57349,-23.18397],[-44.57325,-23.18941]]],[[[-44.65805,-23.18177],[-44.65739,-23.18844],[-44.65983,-23.18582],[-44.65805,-23.18177]]],[[[-44.68253,-23.17748],[-44.68469,-23.17944],[-44.68407,-23.17717],[-44.68253,-23.17748]]],[[[-44.66199,-23.16017],[-44.66627,-23.16151],[-44.66399,-23.15899],[-44.66199,-23.16017]]],[[[-44.68194,-23.15759],[-44.69227,-23.16993],[-44.69469,-23.15647],[-44.68194,-23.15759]]],[[[-43.19907,-23.07341],[-43.19963,-23.07489],[-43.20004,-23.07343],[-43.19907,-23.07341]]],[[[-43.19429,-23.06772],[-43.19313,-23.07428],[-43.19893,-23.07164],[-43.19429,-23.06772]]],[[[-44.67536,-23.13198],[-44.67592,-23.13355],[-44.67678,-23.1331],[-44.67536,-23.13198]]],[[[-43.1496,-23.05975],[-43.1426,-23.0659],[-43.14746,-23.06549],[-43.1496,-23.05975]]],[[[-44.6848,-23.13016],[-44.68811,-23.13033],[-44.68483,-23.12863],[-44.6848,-23.13016]]],[[[-43.56043,-23.08136],[-43.56405,-23.08256],[-43.56185,-23.07892],[-43.56043,-23.08136]]],[[[-44.67777,-23.12625],[-44.68018,-23.12675],[-44.6781,-23.12478],[-44.67777,-23.12625]]],[[[-43.50838,-23.06532],[-43.51432,-23.06659],[-43.51166,-23.06066],[-43.50838,-23.06532]]],[[[-44.67775,-23.11177],[-44.67402,-23.11312],[-44.67809,-23.11493],[-44.67775,-23.11177]]],[[[-44.681,-23.10972],[-44.67961,-23.10873],[-44.67908,-23.11008],[-44.681,-23.10972]]],[[[-44.68608,-23.10746],[-44.689,-23.10919],[-44.68614,-23.10663],[-44.68608,-23.10746]]],[[[-43.50901,-23.05333],[-43.50969,-23.05532],[-43.51007,-23.05501],[-43.50901,-23.05333]]],[[[-44.23527,-23.08501],[-44.09543,-23.17534],[-44.25172,-23.20497],[-44.29444,-23.17691],[-44.34925,-23.22664],[-44.34185,-23.18188],[-44.37763,-23.17169],[-44.23527,-23.08501]]],[[[-43.19749,-23.0351],[-43.20696,-23.04099],[-43.20879,-23.04079],[-43.19749,-23.0351]]],[[[-43.30299,-23.03844],[-43.30779,-23.03929],[-43.30446,-23.03654],[-43.30299,-23.03844]]],[[[-44.67197,-23.09728],[-44.67166,-23.09911],[-44.67383,-23.09858],[-44.67197,-23.09728]]],[[[-43.19157,-23.03007],[-43.19288,-23.0309],[-43.1929,-23.02941],[-43.19157,-23.03007]]],[[[-43.2994,-23.03129],[-43.29854,-23.03416],[-43.30146,-23.03365],[-43.2994,-23.03129]]],[[[-43.19184,-23.02587],[-43.19174,-23.02824],[-43.19549,-23.02775],[-43.19184,-23.02587]]],[[[-43.20599,-23.02773],[-43.20296,-23.02612],[-43.20176,-23.02697],[-43.20599,-23.02773]]],[[[-43.27649,-23.02927],[-43.2772,-23.03121],[-43.2791,-23.03079],[-43.27649,-23.02927]]],[[[-44.59386,-23.08722],[-44.59693,-23.08968],[-44.59505,-23.08575],[-44.59386,-23.08722]]],[[[-44.68433,-23.08859],[-44.68394,-23.08677],[-44.68295,-23.08711],[-44.68433,-23.08859]]],[[[-44.60332,-23.06733],[-44.603,-23.06948],[-44.60517,-23.06839],[-44.60332,-23.06733]]],[[[-44.63127,-23.06829],[-44.63013,-23.07161],[-44.63322,-23.07043],[-44.63127,-23.06829]]],[[[-44.64208,-23.07501],[-44.64272,-23.06827],[-44.636,-23.06842],[-44.64208,-23.07501]]],[[[-44.59511,-23.06659],[-44.59727,-23.06747],[-44.5956,-23.06573],[-44.59511,-23.06659]]],[[[-44.13655,-23.04044],[-44.13763,-23.04396],[-44.13933,-23.04347],[-44.13655,-23.04044]]],[[[-44.63031,-23.06253],[-44.62655,-23.0615],[-44.62967,-23.06339],[-44.63031,-23.06253]]],[[[-44.5578,-23.06374],[-44.5648,-23.06542],[-44.55274,-23.05641],[-44.5578,-23.06374]]],[[[-44.60144,-23.05721],[-44.60908,-23.06148],[-44.60155,-23.05632],[-44.60144,-23.05721]]],[[[-44.62346,-23.05704],[-44.6238,-23.0606],[-44.62489,-23.05964],[-44.62346,-23.05704]]],[[[-44.12374,-23.04036],[-44.12985,-23.04224],[-44.12972,-23.03579],[-44.12374,-23.04036]]],[[[-44.15838,-23.04059],[-44.16152,-23.04094],[-44.15991,-23.03635],[-44.15838,-23.04059]]],[[[-44.56102,-23.05175],[-44.5606,-23.0539],[-44.56366,-23.0534],[-44.56102,-23.05175]]],[[[-43.95316,-23.02256],[-43.95116,-23.02534],[-43.95413,-23.0246],[-43.95316,-23.02256]]],[[[-44.54458,-23.04752],[-44.54597,-23.05011],[-44.54647,-23.05009],[-44.54458,-23.04752]]],[[[-43.9441,-23.02007],[-43.9441,-23.02376],[-43.9476,-23.02342],[-43.9441,-23.02007]]],[[[-44.10989,-23.02224],[-44.11019,-23.02426],[-44.111,-23.02389],[-44.10989,-23.02224]]],[[[-43.94961,-23.00611],[-43.95254,-23.00986],[-43.95507,-23.00663],[-43.94961,-23.00611]]],[[[-43.95235,-23.00125],[-43.95477,-23.00258],[-43.9531,-23.00061],[-43.95235,-23.00125]]],[[[-43.70303,-22.98702],[-43.70655,-22.98899],[-43.70684,-22.98858],[-43.70303,-22.98702]]],[[[-44.04971,-23.00117],[-44.02994,-23.00738],[-44.0526,-23.00517],[-44.04971,-23.00117]]],[[[-43.90829,-22.98913],[-43.91043,-22.99475],[-43.91252,-22.99338],[-43.90829,-22.98913]]],[[[-43.91832,-22.98813],[-43.93441,-23.01528],[-43.93718,-22.99828],[-43.91832,-22.98813]]],[[[-43.91885,-22.98564],[-43.92396,-22.98696],[-43.91974,-22.98432],[-43.91885,-22.98564]]],[[[-44.04191,-22.9864],[-44.0405,-22.98921],[-44.04462,-22.98824],[-44.04191,-22.9864]]],[[[-43.91402,-22.97942],[-43.91046,-22.98355],[-43.91449,-22.98321],[-43.91402,-22.97942]]],[[[-43.9516,-22.96086],[-43.94802,-22.9595],[-43.94746,-22.96148],[-43.9516,-22.96086]]],[[[-43.85732,-22.94503],[-43.8639,-22.95585],[-43.86299,-22.94696],[-43.85732,-22.94503]]],[[[-43.86416,-22.94497],[-43.86654,-22.9456],[-43.86593,-22.94384],[-43.86416,-22.94497]]],[[[-43.96657,-22.94306],[-43.96796,-22.94589],[-43.96799,-22.94318],[-43.96657,-22.94306]]],[[[-43.96227,-22.94393],[-43.96449,-22.94467],[-43.9623,-22.94273],[-43.96227,-22.94393]]],[[[-43.91652,-22.94264],[-43.92052,-22.94209],[-43.91502,-22.93925],[-43.91652,-22.94264]]],[[[-43.86541,-22.92465],[-43.86541,-22.92473],[-43.86561,-22.92467],[-43.86541,-22.92465]]],[[[-43.91216,-22.93463],[-43.8733,-22.94742],[-43.90988,-22.95587],[-43.91216,-22.93463]]],[[[-43.23746,-22.86193],[-43.2219,-22.84146],[-43.21607,-22.87049],[-43.23746,-22.86193]]],[[[-43.17254,-22.77598],[-43.1734,-22.83116],[-43.26365,-22.8093],[-43.1903,-22.79638],[-43.17254,-22.77598]]],[[[-43.10329,-22.75314],[-43.11396,-22.76741],[-43.10865,-22.74809],[-43.10329,-22.75314]]],[[[-41.69753,-22.40333],[-41.6953,-22.41713],[-41.70583,-22.40768],[-41.69753,-22.40333]]],[[[-41.69233,-22.39751],[-41.68717,-22.4022],[-41.69206,-22.40079],[-41.69233,-22.39751]]],[[[-41.86159,-20.76401],[-41.74092,-20.81828],[-41.74186,-20.8728],[-41.71241,-20.87155],[-41.71848,-21.12369],[-41.27708,-21.24023],[-41.09245,-21.21863],[-40.9568,-21.30381],[-41.07467,-21.52224],[-41.01042,-21.61197],[-41.02614,-21.72102],[-40.97455,-21.96858],[-41.2037,-22.13211],[-41.68903,-22.30007],[-41.96948,-22.5434],[-41.9882,-22.70984],[-41.93478,-22.76625],[-41.86442,-22.75401],[-42.02431,-22.89364],[-42.01326,-22.99788],[-42.07654,-22.95561],[-42.36729,-22.9359],[-43.0521,-22.98286],[-43.13576,-22.93919],[-43.0954,-22.92628],[-43.13374,-22.88046],[-43.02684,-22.74235],[-43.03204,-22.69255],[-43.08593,-22.67758],[-43.23602,-22.73558],[-43.28863,-22.80077],[-43.23799,-22.87716],[-43.15709,-22.9132],[-43.19113,-22.99088],[-43.29675,-23.01699],[-43.29446,-23.012],[-43.30629,-23.00101],[-43.34282,-22.98228],[-43.36438,-22.97389],[-43.38307,-22.97407],[-43.39435,-22.98367],[-43.40396,-22.97525],[-43.40766,-22.99241],[-43.29574,-23.01256],[-43.44372,-23.022],[-43.53616,-23.05184],[-43.55407,-23.07659],[-43.66852,-23.05405],[-43.98724,-23.10234],[-44.01255,-23.07828],[-43.97432,-23.04555],[-43.88519,-23.06755],[-43.91216,-23.02027],[-43.79399,-23.06122],[-43.61452,-23.03182],[-43.568,-23.06462],[-43.55719,-23.04913],[-43.85635,-22.90206],[-44.00341,-22.93862],[-44.04721,-22.98445],[-44.04646,-22.9389],[-44.1301,-23.03244],[-44.20024,-23.03879],[-44.23755,-23.05691],[-44.24838,-22.99611],[-44.35827,-23.02333],[-44.29975,-22.95869],[-44.35899,-22.92298],[-44.37766,-22.96973],[-44.43538,-22.96487],[-44.43105,-23.02074],[-44.64564,-23.04707],[-44.69144,-23.08394],[-44.72163,-23.20208],[-44.71264,-23.23373],[-44.64322,-23.18576],[-44.62453,-23.22153],[-44.66295,-23.21462],[-44.69272,-23.25322],[-44.61359,-23.23453],[-44.65581,-23.29579],[-44.56032,-23.22755],[-44.58424,-23.26688],[-44.50286,-23.29681],[-44.55663,-23.3058],[-44.58553,-23.35989],[-44.64082,-23.33319],[-44.75013,-23.35849],[-44.87505,-23.24913],[-44.79247,-22.98213],[-44.49472,-22.84643],[-44.46621,-22.88549],[-44.27343,-22.83246],[-44.25757,-22.76692],[-44.16094,-22.67829],[-44.2268,-22.60454],[-44.34329,-22.58823],[-44.3567,-22.61446],[-44.38441,-22.57375],[-44.50931,-22.64089],[-44.63463,-22.60954],[-44.73349,-22.43357],[-44.804,-22.39379],[-44.72397,-22.35992],[-44.66069,-22.38049],[-44.43226,-22.25141],[-44.23547,-22.26618],[-44.10007,-22.17328],[-43.76599,-22.06289],[-43.7346,-22.09723],[-43.58608,-22.052],[-43.56313,-22.08822],[-43.34953,-22.00337],[-43.24654,-22.00701],[-43.13113,-22.02966],[-43.13654,-22.10964],[-43.07413,-22.09332],[-42.88251,-21.96009],[-42.26736,-21.71419],[-42.36914,-21.61994],[-42.25298,-21.49367],[-42.29262,-21.46002],[-42.1897,-21.24988],[-42.2085,-21.1782],[-42.0805,-21.03609],[-42.15168,-20.97436],[-41.97667,-20.93549],[-41.92814,-20.79498],[-41.86159,-20.76401]]]]}},{"type":"Feature","id":"RN","properties":{"SIGLA":"RN","Estado":"Rio Grande do Norte"},"geometry":{"type":"Polygon","coordinates":[[[-34.96884,-6.49017],[-35.05779,-6.52784],[-35.0956,-6.49773],[-35.17035,-6.55844],[-35.22351,-6.51628],[-35.33082,-6.54439],[-35.4008,-6.49165],[-35.56484,-6.49183],[-35.65731,-6.44583],[-35.97769,-6.48895],[-36.08006,-6.40524],[-36.24981,-6.43741],[-36.29929,-6.3821],[-36.29502,-6.29216],[-36.3946,-6.29402],[-36.43035,-6.3623],[-36.49989,-6.35821],[-36.53101,-6.45212],[-36.4359,-6.62592],[-36.52195,-6.59713],[-36.55892,-6.70915],[-36.50479,-6.75398],[-36.50553,-6.80887],[-36.56983,-6.84805],[-36.60983,-6.93688],[-36.68725,-6.92601],[-36.71859,-6.98278],[-36.76658,-6.93591],[-36.73023,-6.83644],[-36.7695,-6.83879],[-36.83507,-6.73111],[-36.95717,-6.79052],[-37.00363,-6.70966],[-37.23463,-6.82468],[-37.28375,-6.69396],[-37.48473,-6.71008],[-37.48316,-6.55045],[-37.39726,-6.51464],[-37.36339,-6.32221],[-37.16572,-6.17042],[-37.17397,-6.04806],[-37.25065,-6.02596],[-37.45898,-6.14505],[-37.7424,-6.18902],[-37.77565,-6.23139],[-37.75769,-6.29082],[-37.81045,-6.2917],[-37.92815,-6.42164],[-37.95542,-6.39825],[-38.01372,-6.42747],[-38.01298,-6.4689],[-38.05148,-6.44245],[-38.11709,-6.52271],[-38.23926,-6.48034],[-38.28875,-6.50566],[-38.40177,-6.4096],[-38.48626,-6.39879],[-38.45841,-6.33026],[-38.56333,-6.35607],[-38.58132,-6.31625],[-38.44734,-6.08532],[-38.38198,-6.05283],[-38.36527,-6.0885],[-38.30483,-6.08701],[-38.16482,-5.94658],[-38.04704,-5.73031],[-38.0872,-5.73057],[-38.08291,-5.67259],[-37.90228,-5.50071],[-37.64037,-4.92646],[-37.25302,-4.83177],[-37.16717,-4.93219],[-37.03883,-4.95112],[-36.96259,-4.91977],[-36.69157,-5.09334],[-36.50181,-5.06162],[-36.13769,-5.09344],[-35.98051,-5.04162],[-35.48812,-5.15954],[-35.26115,-5.48085],[-35.17994,-5.86882],[-35.1071,-6.00635],[-35.09478,-6.18176],[-35.03735,-6.23607],[-34.96884,-6.49017]]]}},{"type":"Feature","id":"RO","properties":{"SIGLA":"RO","Estado":"Rondônia"},"geometry":{"type":"Polygon","coordinates":[[[-62.86652,-7.97588],[-62.69219,-8.09292],[-62.68657,-8.17379],[-62.56183,-8.28402],[-62.52605,-8.38307],[-62.46539,-8.33975],[-62.36499,-8.39104],[-62.341,-8.60284],[-62.28122,-8.63861],[-62.2688,-8.57577],[-62.1883,-8.59057],[-62.20044,-8.64431],[-62.12489,-8.80169],[-62.03098,-8.79973],[-61.98514,-8.87862],[-61.9058,-8.87453],[-61.85997,-8.8531],[-61.83642,-8.73274],[-61.77452,-8.74824],[-61.71319,-8.68787],[-61.63047,-8.72197],[-61.60594,-8.79473],[-61.52349,-8.81861],[-61.46841,-8.91739],[-61.55566,-9.09242],[-61.52477,-9.24287],[-61.63327,-9.27062],[-61.62585,-9.36303],[-61.55243,-9.38835],[-61.5769,-9.4745],[-61.52573,-9.50182],[-61.54947,-9.51524],[-61.47673,-9.62874],[-61.50519,-9.6914],[-61.57456,-9.71781],[-61.53142,-9.73993],[-61.5084,-9.87299],[-61.60151,-10.15667],[-61.5586,-10.17652],[-61.57666,-10.2486],[-61.4617,-10.41993],[-61.49874,-10.69996],[-61.47376,-10.79754],[-61.51101,-10.79484],[-61.55018,-10.98612],[-60.45998,-10.98988],[-60.44652,-11.04218],[-60.3479,-11.10906],[-60.30118,-11.05717],[-60.19893,-11.11411],[-59.97671,-11.12241],[-59.98196,-11.23352],[-59.92056,-11.39803],[-60.11481,-11.59148],[-60.1087,-11.8396],[-59.98573,-11.9121],[-59.97929,-12.02957],[-59.89974,-12.11725],[-59.89227,-12.24548],[-59.77429,-12.34098],[-59.85579,-12.47863],[-59.93516,-12.48711],[-60.06867,-12.61602],[-60.07905,-12.88117],[-60.1171,-12.95958],[-60.18921,-12.97117],[-60.28293,-13.08058],[-60.26844,-13.145],[-60.36491,-13.29997],[-60.38786,-13.45473],[-60.63214,-13.57173],[-60.70924,-13.69304],[-60.87978,-13.61756],[-60.92297,-13.54089],[-61.00556,-13.55031],[-61.01486,-13.48757],[-61.19422,-13.53648],[-61.29625,-13.47786],[-61.46986,-13.55529],[-61.57666,-13.48866],[-61.84103,-13.54895],[-61.89292,-13.43097],[-61.97056,-13.40621],[-62.11392,-13.26104],[-62.11514,-13.16372],[-62.17042,-13.11396],[-62.19087,-13.15387],[-62.21472,-13.11138],[-62.27212,-13.15589],[-62.41393,-13.12842],[-62.45395,-13.06483],[-62.50256,-13.0832],[-62.61251,-13.04134],[-62.65076,-12.96562],[-62.72909,-13.02081],[-62.77903,-13.00961],[-62.89209,-12.85888],[-63.00916,-12.83782],[-63.08237,-12.72059],[-63.06131,-12.68658],[-63.15832,-12.61418],[-63.24542,-12.66223],[-63.23895,-12.69105],[-63.29548,-12.6819],[-63.44032,-12.60625],[-63.43462,-12.56588],[-63.50167,-12.56736],[-63.62205,-12.47292],[-63.78623,-12.42799],[-63.8339,-12.46347],[-63.88509,-12.44642],[-63.90064,-12.50606],[-63.95763,-12.53127],[-64.13497,-12.47651],[-64.16454,-12.51441],[-64.17212,-12.46719],[-64.23156,-12.45513],[-64.28437,-12.50322],[-64.29267,-12.45928],[-64.41614,-12.44151],[-64.51293,-12.34785],[-64.51272,-12.22292],[-64.54703,-12.24275],[-64.69876,-12.18721],[-64.7129,-12.08643],[-64.74987,-12.15761],[-64.7884,-12.08583],[-64.83254,-12.12175],[-64.8094,-12.05914],[-64.83951,-12.01072],[-65.03045,-11.99665],[-65.01487,-11.90095],[-65.07062,-11.86915],[-65.08806,-11.70942],[-65.1829,-11.72174],[-65.18137,-11.75661],[-65.26056,-11.70691],[-65.21784,-11.65213],[-65.21167,-11.53288],[-65.30906,-11.49368],[-65.33533,-11.35498],[-65.29173,-11.3242],[-65.34594,-11.30506],[-65.36062,-11.22336],[-65.31861,-11.19946],[-65.36342,-11.1473],[-65.28625,-11.09224],[-65.30117,-11.03217],[-65.25078,-10.98458],[-65.27548,-10.8712],[-65.36688,-10.79629],[-65.34456,-10.69089],[-65.41667,-10.61872],[-65.42966,-10.48098],[-65.28874,-10.2199],[-65.33292,-9.94431],[-65.28596,-9.84552],[-65.36371,-9.71161],[-65.44671,-9.66972],[-65.56027,-9.84469],[-65.63049,-9.83441],[-65.6688,-9.78117],[-65.71046,-9.80735],[-65.68488,-9.74904],[-65.74416,-9.78211],[-65.76788,-9.73557],[-65.79444,-9.79182],[-65.79979,-9.7567],[-65.86057,-9.79517],[-65.88582,-9.75251],[-65.96707,-9.77068],[-65.98113,-9.80868],[-66.15193,-9.78565],[-66.20358,-9.83465],[-66.3504,-9.84075],[-66.42616,-9.89966],[-66.43571,-9.86604],[-66.58224,-9.9059],[-66.80619,-9.81446],[-66.77527,-9.75312],[-66.50052,-9.63359],[-66.39232,-9.50051],[-66.40876,-9.40697],[-66.3074,-9.43687],[-65.97021,-9.41319],[-65.79143,-9.58567],[-65.72346,-9.55908],[-65.67934,-9.45142],[-65.59668,-9.41389],[-65.43463,-9.46636],[-65.44693,-9.3164],[-65.38512,-9.33806],[-65.24657,-9.25769],[-65.20793,-9.28566],[-65.18453,-9.42702],[-65.09776,-9.43254],[-64.94059,-9.26243],[-64.91718,-9.04316],[-64.83923,-8.99413],[-64.61098,-9.01647],[-64.32331,-8.92888],[-64.31123,-8.99649],[-64.14334,-8.95355],[-64.14298,-8.74305],[-64.02778,-8.71568],[-63.98612,-8.58565],[-63.94586,-8.6095],[-63.92239,-8.56739],[-63.97441,-8.48726],[-63.94075,-8.45228],[-63.9778,-8.42015],[-63.94403,-8.3312],[-63.86739,-8.28933],[-63.7815,-8.32926],[-63.62069,-7.96931],[-62.86652,-7.97588]]]}},{"type":"Feature","id":"RR","properties":{"SIGLA":"RR","Estado":"Roraima"},"geometry":{"type":"Polygon","coordinates":[[[-58.91844,1.31653],[-58.89536,0.26347],[-60.03786,0.26352],[-60.05897,0.16297],[-60.11616,0.13321],[-60.13845,0.02875],[-60.20316,-0.03361],[-60.22428,-0.14288],[-60.29928,-0.21176],[-60.28983,-0.29316],[-60.39984,-0.5101],[-60.38317,-0.59009],[-60.31345,-0.64148],[-60.30956,-0.72482],[-60.47901,-0.77093],[-60.53095,-0.87509],[-60.59123,-0.85342],[-60.6679,-0.89453],[-60.75261,-0.8612],[-60.76095,-0.76148],[-60.90817,-0.62398],[-60.92067,-0.55537],[-61.21678,-0.50009],[-61.224,-0.55953],[-61.46455,-0.6648],[-61.54205,-0.76369],[-61.58511,-0.93702],[-61.544,-1.0623],[-61.62873,-1.30147],[-61.58205,-1.35341],[-61.619,-1.3948],[-61.53817,-1.43341],[-61.47456,-1.57897],[-61.63539,-1.43397],[-61.79009,-1.37897],[-61.8965,-1.39535],[-62.01734,-1.14174],[-62.51011,-0.75896],[-62.4865,-0.68146],[-62.40678,-0.72729],[-62.2963,-0.65221],[-62.30907,-0.51415],[-62.18824,-0.33082],[-62.24824,-0.30193],[-62.24491,-0.17471],[-62.3196,-0.10252],[-62.34651,0.02961],[-62.42426,0.09158],[-62.46815,0.27576],[-62.44623,0.37912],[-62.53287,0.50882],[-62.53526,0.69037],[-62.4576,0.7852],[-62.47148,1.08629],[-62.52953,1.08883],[-62.63982,1.43825],[-62.80513,1.59136],[-62.72358,1.71241],[-62.70505,1.93658],[-62.83868,2.01648],[-63.05341,2.0289],[-63.14136,2.17246],[-63.28263,2.1539],[-63.35963,2.19712],[-63.4254,2.36428],[-63.40669,2.43596],[-63.751,2.38679],[-64.05591,2.49771],[-63.98563,2.64549],[-63.99314,2.76771],[-64.07146,2.93771],[-64.23508,3.11438],[-64.20814,3.18882],[-64.25507,3.41104],[-64.18425,3.48938],[-64.18563,3.55965],[-64.28897,3.69965],[-64.48535,3.78743],[-64.73201,4.12021],[-64.81034,4.17466],[-64.82146,4.25855],[-64.78035,4.2866],[-64.69951,4.25021],[-64.56032,4.1016],[-64.28952,4.14382],[-64.16424,4.12632],[-63.96396,3.86798],[-63.92813,3.92492],[-63.85007,3.94993],[-63.68258,3.90742],[-63.67675,4.01909],[-63.59174,3.88548],[-63.4973,3.83992],[-63.48924,3.87409],[-63.43452,3.86492],[-63.42869,3.97714],[-63.2048,3.9513],[-63.22647,3.83547],[-63.10314,3.79324],[-63.05897,3.74741],[-63.08119,3.69325],[-62.96008,3.60769],[-62.83564,3.73852],[-62.74369,3.67324],[-62.72897,3.80463],[-62.78869,3.89324],[-62.75342,4.0313],[-62.55563,4.01824],[-62.55258,4.10879],[-62.46508,4.13851],[-62.46258,4.17795],[-62.14591,4.07406],[-62.07675,4.15378],[-61.98286,4.18044],[-61.93064,4.10322],[-61.92287,4.15571],[-61.82397,4.1635],[-61.7748,4.24961],[-61.56008,4.25182],[-61.50814,4.32183],[-61.51342,4.40543],[-61.28814,4.45737],[-61.32306,4.53487],[-61.21717,4.53524],[-61.14675,4.48154],[-61.09528,4.52195],[-60.99454,4.51848],[-60.93204,4.58626],[-60.94953,4.65264],[-60.89981,4.71626],[-60.75147,4.75542],[-60.59176,4.9268],[-60.66108,5.16369],[-60.72314,5.21986],[-60.43426,5.18124],[-60.21231,5.27179],[-60.17231,5.22651],[-60.1351,5.24818],[-60.09426,5.14012],[-59.9967,5.08444],[-59.98953,4.9704],[-60.03009,4.7004],[-60.07509,4.60845],[-60.1601,4.56262],[-60.16315,4.50623],[-59.79677,4.46512],[-59.67232,4.37317],[-59.73176,4.28429],[-59.72537,4.18373],[-59.61649,4.13234],[-59.64954,4.0704],[-59.57482,4.00151],[-59.58483,3.96735],[-59.51621,3.94401],[-59.59232,3.8879],[-59.59593,3.79568],[-59.6801,3.75929],[-59.66843,3.70291],[-59.86621,3.57763],[-59.80399,3.5068],[-59.84171,3.42496],[-59.81399,3.42791],[-59.80371,3.36153],[-59.90732,3.21207],[-59.98871,2.88153],[-59.99038,2.68514],[-59.89538,2.48153],[-59.8965,2.36292],[-59.72233,2.27681],[-59.7515,1.85849],[-59.67761,1.83848],[-59.66289,1.87043],[-59.69011,1.75626],[-59.539,1.72237],[-59.25317,1.38821],[-58.97872,1.30154],[-58.91844,1.31653]]]}},{"type":"Feature","id":"RS","properties":{"SIGLA":"RS","Estado":"Rio Grande do Sul"},"geometry":{"type":"Polygon","coordinates":[[[-49.7117,-29.32579],[-50.018,-29.77225],[-50.33348,-30.50073],[-50.77226,-31.11368],[-51.25035,-31.56723],[-51.8851,-31.9581],[-52.07649,-32.18246],[-52.09329,-32.12876],[-52.01386,-31.9393],[-52.11102,-31.79319],[-51.85226,-31.86747],[-51.78718,-31.80622],[-51.8137,-31.81672],[-51.86533,-31.79932],[-51.66468,-31.7709],[-51.49131,-31.56923],[-51.43632,-31.6248],[-51.44888,-31.52774],[-51.46655,-31.55462],[-51.42863,-31.48009],[-51.37086,-31.52957],[-51.26414,-31.48052],[-51.17248,-31.33985],[-51.18028,-31.13439],[-51.16549,-31.06187],[-51.11075,-31.09994],[-50.98055,-31.04098],[-50.96675,-30.89603],[-50.91128,-30.90569],[-50.75377,-30.81886],[-50.70198,-30.74625],[-50.67953,-30.6091],[-50.71753,-30.35191],[-50.69909,-30.34591],[-50.64787,-30.39539],[-50.62398,-30.39203],[-50.63791,-30.37886],[-50.62506,-30.32833],[-50.59309,-30.40027],[-50.6544,-30.44289],[-50.57441,-30.4814],[-50.53777,-30.27379],[-50.59648,-30.19487],[-50.65888,-30.19985],[-50.67323,-30.29657],[-50.782,-30.2847],[-50.77585,-30.3339],[-50.92211,-30.3321],[-50.9392,-30.39406],[-50.90193,-30.44288],[-50.93003,-30.4357],[-51.05958,-30.38551],[-51.03075,-30.2744],[-51.08645,-30.24146],[-51.13366,-30.26985],[-51.16955,-30.26457],[-51.19819,-30.19436],[-51.24867,-30.18627],[-51.2258,-30.14485],[-51.28861,-30.12039],[-51.23031,-30.04367],[-51.29234,-30.06479],[-51.26016,-30.03444],[-51.29576,-30.00038],[-51.32859,-30.22658],[-51.27319,-30.24969],[-51.29363,-30.30401],[-51.2472,-30.32845],[-51.21071,-30.29785],[-51.18099,-30.3859],[-51.09464,-30.36466],[-51.15258,-30.50019],[-51.13574,-30.43671],[-51.19787,-30.40796],[-51.2984,-30.55901],[-51.28242,-30.81246],[-51.3312,-30.77841],[-51.29394,-30.77082],[-51.31765,-30.64809],[-51.36408,-30.63317],[-51.40582,-30.75668],[-51.369,-30.87428],[-51.44505,-30.87105],[-51.49992,-30.92998],[-51.44058,-31.08773],[-51.61798,-31.13948],[-51.643,-31.20259],[-51.61671,-31.26761],[-51.79046,-31.27319],[-51.85524,-31.33628],[-51.86999,-31.29478],[-51.93182,-31.32101],[-51.99368,-31.41349],[-52.03064,-31.69227],[-52.14989,-31.69827],[-52.22576,-31.75182],[-52.25005,-31.87537],[-52.13301,-31.92366],[-52.11291,-31.94303],[-52.11309,-31.95586],[-52.15066,-31.9282],[-52.21862,-31.95992],[-52.24822,-32.05165],[-52.2015,-32.04016],[-52.21715,-32.08456],[-52.11263,-32.0175],[-52.09859,-32.02195],[-52.09882,-32.00428],[-52.20153,-32.02393],[-52.1282,-31.95865],[-52.09613,-31.99611],[-52.0898,-32.02262],[-52.10882,-32.02739],[-52.06843,-32.03009],[-52.06361,-32.04652],[-52.13868,-32.06187],[-52.15776,-32.11274],[-52.08494,-32.06171],[-52.10506,-32.10736],[-52.08348,-32.18383],[-52.15646,-32.19055],[-52.30694,-32.36146],[-52.62923,-33.11611],[-52.79573,-33.30248],[-53.39717,-33.75201],[-53.44089,-33.69451],[-53.53265,-33.68944],[-53.53619,-33.64949],[-53.50901,-33.52799],[-53.46133,-33.57524],[-53.43465,-33.50033],[-53.47083,-33.26079],[-53.42689,-33.14031],[-53.41112,-33.1083],[-53.40638,-33.10815],[-53.42432,-33.15613],[-53.39221,-33.15112],[-53.31976,-33.05477],[-53.26175,-33.10948],[-53.15998,-32.96961],[-53.12669,-32.79371],[-53.07277,-32.83297],[-52.98849,-32.74374],[-52.98561,-32.81675],[-52.86291,-32.909],[-52.75066,-32.86232],[-52.5862,-32.52667],[-52.69489,-32.31786],[-52.62241,-32.14391],[-52.77414,-32.20553],[-52.72909,-32.21301],[-52.77395,-32.23168],[-52.81988,-32.33983],[-52.72214,-32.38506],[-52.78958,-32.44425],[-52.94773,-32.48085],[-53.00079,-32.5978],[-53.08463,-32.65817],[-53.38862,-32.58751],[-53.46373,-32.48547],[-53.55756,-32.4717],[-53.64465,-32.38462],[-53.64269,-32.29716],[-53.74567,-32.07911],[-53.83327,-32.0556],[-53.84997,-32.00143],[-53.96063,-31.95614],[-53.97002,-31.91883],[-54.03094,-31.8963],[-54.10041,-31.92831],[-54.4545,-31.65334],[-54.4531,-31.60084],[-54.58631,-31.45711],[-54.83619,-31.44243],[-54.89153,-31.37555],[-54.94029,-31.38144],[-55.00657,-31.26735],[-55.07399,-31.3326],[-55.23964,-31.26096],[-55.29049,-31.14413],[-55.33824,-31.13405],[-55.35129,-31.03798],[-55.42449,-31.01693],[-55.57813,-30.83333],[-55.65766,-30.86405],[-55.6659,-30.95433],[-55.72384,-30.94322],[-55.72802,-30.97973],[-55.88214,-31.07738],[-56.0101,-31.0814],[-55.98988,-30.85911],[-56.02341,-30.78683],[-56.12938,-30.73812],[-56.17039,-30.6155],[-56.3836,-30.49867],[-56.46189,-30.38521],[-56.54519,-30.36018],[-56.54046,-30.31471],[-56.61707,-30.30127],[-56.64618,-30.20504],[-56.77692,-30.16446],[-56.80642,-30.10524],[-57.0699,-30.08702],[-57.21799,-30.29032],[-57.31451,-30.25872],[-57.39569,-30.30538],[-57.46139,-30.26719],[-57.52269,-30.28657],[-57.56625,-30.25901],[-57.56343,-30.20964],[-57.64332,-30.19364],[-57.46169,-30.10654],[-57.33827,-29.99307],[-57.29481,-29.83157],[-57.12153,-29.76538],[-56.79926,-29.46555],[-56.76843,-29.37771],[-56.70562,-29.36135],[-56.65504,-29.27789],[-56.59056,-29.12036],[-56.41855,-29.07549],[-56.41005,-28.97591],[-56.29834,-28.8906],[-56.2937,-28.79759],[-56.18941,-28.77242],[-56.00212,-28.59853],[-56.01127,-28.50705],[-55.8861,-28.48036],[-55.87709,-28.36167],[-55.75389,-28.36973],[-55.69688,-28.42577],[-55.66803,-28.34115],[-55.77221,-28.24311],[-55.63006,-28.17614],[-55.60819,-28.11613],[-55.55741,-28.16582],[-55.49421,-28.0773],[-55.44604,-28.098],[-55.36833,-28.02944],[-55.38451,-27.98414],[-55.19653,-27.85656],[-55.13372,-27.89708],[-55.10654,-27.8466],[-55.03515,-27.85873],[-55.08121,-27.77929],[-54.93641,-27.77263],[-54.90649,-27.64113],[-54.85043,-27.62405],[-54.81491,-27.53385],[-54.77513,-27.58597],[-54.68249,-27.5747],[-54.67179,-27.50518],[-54.63203,-27.5458],[-54.58532,-27.45461],[-54.53031,-27.50588],[-54.44411,-27.47274],[-54.46991,-27.42879],[-54.41069,-27.4057],[-54.34661,-27.46719],[-54.33881,-27.40463],[-54.28395,-27.44787],[-54.2166,-27.38563],[-54.17248,-27.2547],[-54.15491,-27.29663],[-54.07795,-27.29728],[-53.95297,-27.15375],[-53.90311,-27.17458],[-53.87658,-27.12617],[-53.73953,-27.19288],[-53.66674,-27.1621],[-53.64248,-27.22084],[-53.56458,-27.17342],[-53.49377,-27.20366],[-53.50184,-27.13463],[-53.41878,-27.14372],[-53.37398,-27.09056],[-53.29322,-27.13431],[-53.31093,-27.21839],[-53.24294,-27.17067],[-53.18867,-27.19149],[-53.1576,-27.13862],[-53.13401,-27.1782],[-53.07274,-27.15828],[-53.07655,-27.10359],[-53.02897,-27.08054],[-53.04309,-27.14596],[-52.99382,-27.13823],[-52.97767,-27.22114],[-52.95908,-27.16344],[-52.92462,-27.20248],[-52.8586,-27.16768],[-52.84067,-27.20921],[-52.76743,-27.20934],[-52.75394,-27.25833],[-52.71491,-27.23689],[-52.69156,-27.28416],[-52.66814,-27.23841],[-52.48606,-27.26691],[-52.44005,-27.21673],[-52.44046,-27.26267],[-52.39585,-27.29261],[-52.38207,-27.25765],[-52.3757,-27.30425],[-52.29913,-27.31864],[-52.30176,-27.26002],[-52.25391,-27.25747],[-52.27795,-27.29641],[-52.21016,-27.33086],[-52.16659,-27.27334],[-52.16727,-27.3116],[-52.11907,-27.30317],[-52.09799,-27.34837],[-52.01826,-27.33172],[-52.00963,-27.37189],[-51.95101,-27.3813],[-52.00857,-27.40052],[-51.93746,-27.427],[-51.96268,-27.47042],[-51.90098,-27.46317],[-51.88938,-27.52058],[-51.86713,-27.47669],[-51.84902,-27.5255],[-51.79712,-27.53055],[-51.79862,-27.491],[-51.72521,-27.51399],[-51.68449,-27.47945],[-51.65126,-27.51981],[-51.63222,-27.48886],[-51.62653,-27.5452],[-51.58209,-27.52307],[-51.56799,-27.58479],[-51.49006,-27.55944],[-51.43424,-27.65495],[-51.38186,-27.62517],[-51.24313,-27.77557],[-51.08773,-27.83184],[-51.01916,-27.95638],[-50.92883,-27.96905],[-50.87494,-28.04133],[-50.90359,-28.08687],[-50.87496,-28.13188],[-50.78962,-28.13924],[-50.78866,-28.19836],[-50.6251,-28.39142],[-50.37869,-28.44617],[-50.35716,-28.42074],[-50.34575,-28.46331],[-50.24758,-28.42886],[-50.15718,-28.49768],[-50.12672,-28.42904],[-50.09985,-28.48554],[-49.97155,-28.43912],[-49.94421,-28.4824],[-49.92873,-28.45109],[-49.85482,-28.44394],[-49.83981,-28.49619],[-49.76471,-28.45999],[-49.69358,-28.62496],[-49.78859,-28.6132],[-49.87566,-28.74483],[-49.90523,-28.70724],[-49.93538,-28.72806],[-49.97117,-28.92427],[-49.91989,-28.97835],[-50.00745,-29.07215],[-49.94006,-29.05713],[-49.96286,-29.11791],[-50.01405,-29.12822],[-50.01481,-29.18165],[-50.09941,-29.16387],[-50.08396,-29.22796],[-50.11199,-29.24026],[-50.0938,-29.21331],[-50.13925,-29.19263],[-50.17497,-29.24965],[-50.03728,-29.35137],[-50.11467,-29.25924],[-49.99685,-29.22789],[-49.96973,-29.20318],[-49.9625,-29.19913],[-49.95479,-29.19951],[-49.7117,-29.32579]]]}},{"type":"Feature","id":"SC","properties":{"SIGLA":"SC","Estado":"Santa Catarina"},"geometry":{"type":"Polygon","coordinates":[[[-48.96078,-25.98097],[-48.59456,-25.97704],[-48.58097,-26.16522],[-48.52636,-26.16517],[-48.49279,-26.2192],[-48.68684,-26.68802],[-48.67136,-26.76237],[-48.58535,-26.78323],[-48.64325,-26.90164],[-48.62123,-26.99971],[-48.56721,-27.0078],[-48.60262,-27.12506],[-48.5493,-27.15752],[-48.50725,-27.11126],[-48.46562,-27.14512],[-48.48705,-27.21139],[-48.51747,-27.22092],[-48.53017,-27.18133],[-48.61013,-27.22666],[-48.59623,-27.31653],[-48.54042,-27.29826],[-48.52412,-27.33407],[-48.55219,-27.43362],[-48.60855,-27.43143],[-48.5855,-27.45424],[-48.54123,-27.43609],[-48.48315,-27.37976],[-48.41651,-27.38176],[-48.35987,-27.44864],[-48.50119,-27.71462],[-48.48362,-27.78397],[-48.60226,-27.8649],[-48.57396,-27.89001],[-48.63109,-27.99464],[-48.59884,-28.03033],[-48.66358,-28.18186],[-48.64644,-28.23392],[-48.75935,-28.44393],[-48.75763,-28.53812],[-48.80977,-28.60546],[-49.29107,-28.88296],[-49.7117,-29.32579],[-49.95479,-29.19951],[-50.11467,-29.25924],[-50.03728,-29.35137],[-50.17497,-29.24965],[-50.13925,-29.19263],[-50.0938,-29.21331],[-50.11199,-29.24026],[-50.08396,-29.22796],[-50.09941,-29.16387],[-50.01481,-29.18165],[-50.01405,-29.12822],[-49.96286,-29.11791],[-49.94006,-29.05713],[-50.00745,-29.07215],[-49.91989,-28.97835],[-49.97117,-28.92427],[-49.93538,-28.72806],[-49.90523,-28.70724],[-49.87566,-28.74483],[-49.78859,-28.6132],[-49.69358,-28.62496],[-49.76471,-28.45999],[-49.83981,-28.49619],[-49.85482,-28.44394],[-49.94421,-28.4824],[-49.97155,-28.43912],[-50.09985,-28.48554],[-50.12672,-28.42904],[-50.15718,-28.49768],[-50.24758,-28.42886],[-50.34575,-28.46331],[-50.35716,-28.42074],[-50.37869,-28.44617],[-50.6251,-28.39142],[-50.78866,-28.19836],[-50.78962,-28.13924],[-50.87496,-28.13188],[-50.90359,-28.08687],[-50.87494,-28.04133],[-50.92883,-27.96905],[-51.01916,-27.95638],[-51.08773,-27.83184],[-51.24313,-27.77557],[-51.38186,-27.62517],[-51.43424,-27.65495],[-51.49006,-27.55944],[-51.56799,-27.58479],[-51.58209,-27.52307],[-51.62653,-27.5452],[-51.63222,-27.48886],[-51.65126,-27.51981],[-51.68449,-27.47945],[-51.72521,-27.51399],[-51.79862,-27.491],[-51.79712,-27.53055],[-51.84902,-27.5255],[-51.86713,-27.47669],[-51.88938,-27.52058],[-51.90098,-27.46317],[-51.96268,-27.47042],[-51.93746,-27.427],[-52.00857,-27.40052],[-51.95101,-27.3813],[-52.00963,-27.37189],[-52.01826,-27.33172],[-52.09799,-27.34837],[-52.11907,-27.30317],[-52.16727,-27.3116],[-52.16659,-27.27334],[-52.21016,-27.33086],[-52.27795,-27.29641],[-52.25391,-27.25747],[-52.30176,-27.26002],[-52.29913,-27.31864],[-52.3757,-27.30425],[-52.38207,-27.25765],[-52.39585,-27.29261],[-52.44046,-27.26267],[-52.44005,-27.21673],[-52.48606,-27.26691],[-52.66814,-27.23841],[-52.69156,-27.28416],[-52.71491,-27.23689],[-52.75394,-27.25833],[-52.76743,-27.20934],[-52.84067,-27.20921],[-52.8586,-27.16768],[-52.92462,-27.20248],[-52.95908,-27.16344],[-52.97767,-27.22114],[-52.99382,-27.13823],[-53.04309,-27.14596],[-53.02897,-27.08054],[-53.07655,-27.10359],[-53.07274,-27.15828],[-53.13401,-27.1782],[-53.1576,-27.13862],[-53.18867,-27.19149],[-53.24294,-27.17067],[-53.31093,-27.21839],[-53.29322,-27.13431],[-53.37398,-27.09056],[-53.41878,-27.14372],[-53.50184,-27.13463],[-53.49377,-27.20366],[-53.56458,-27.17342],[-53.64248,-27.22084],[-53.66674,-27.1621],[-53.73953,-27.19288],[-53.77639,-27.14771],[-53.83364,-27.16955],[-53.79805,-27.14488],[-53.81514,-27.0988],[-53.77541,-27.1021],[-53.80177,-27.04043],[-53.75834,-27.06189],[-53.78436,-27.02595],[-53.74686,-27.03075],[-53.70766,-26.93269],[-53.67079,-26.94223],[-53.69566,-26.85801],[-53.66056,-26.85672],[-53.71024,-26.81223],[-53.6993,-26.76781],[-53.72919,-26.77344],[-53.7435,-26.76263],[-53.71443,-26.74919],[-53.75733,-26.7193],[-53.71594,-26.68273],[-53.75886,-26.64107],[-53.70655,-26.55978],[-53.74093,-26.54535],[-53.69483,-26.49028],[-53.70901,-26.38778],[-53.64372,-26.25308],[-53.4972,-26.3028],[-53.37558,-26.242],[-53.28329,-26.24592],[-53.09067,-26.39047],[-52.98819,-26.34655],[-52.73775,-26.34213],[-52.60265,-26.41542],[-52.18565,-26.44537],[-52.01123,-26.56651],[-51.87436,-26.60036],[-51.66506,-26.56604],[-51.49965,-26.58569],[-51.41152,-26.71724],[-51.39124,-26.66385],[-51.22986,-26.61626],[-51.21596,-26.57008],[-51.29919,-26.41911],[-51.24307,-26.32467],[-51.10496,-26.27539],[-51.08002,-26.22821],[-51.00046,-26.23044],[-50.94468,-26.28247],[-50.93542,-26.23356],[-50.90196,-26.28943],[-50.87109,-26.24216],[-50.83936,-26.2715],[-50.79329,-26.22769],[-50.71849,-26.24474],[-50.73179,-26.20337],[-50.62847,-26.06308],[-50.55165,-26.05372],[-50.59214,-26.04253],[-50.57281,-26.00342],[-50.5476,-26.04015],[-50.44514,-26.03434],[-50.32359,-26.1349],[-50.33206,-26.08366],[-50.25151,-26.03019],[-50.18126,-26.07981],[-50.17185,-26.02512],[-50.12106,-26.06431],[-50.08481,-26.0235],[-50.00665,-26.04589],[-50.01153,-26.01396],[-49.9758,-26.01414],[-49.97277,-26.05829],[-49.94183,-26.00845],[-49.83142,-26.05342],[-49.72664,-26.17105],[-49.55507,-26.23729],[-49.45913,-26.16266],[-49.37028,-26.15591],[-49.17527,-26.0018],[-49.05521,-26.0205],[-48.96078,-25.98097]]]}},{"type":"Feature","id":"SE","properties":{"SIGLA":"SE","Estado":"Sergipe"},"geometry":{"type":"Polygon","coordinates":[[[-36.39392,-10.50037],[-36.62048,-10.58948],[-36.8541,-10.74363],[-37.26127,-11.28888],[-37.32999,-11.44514],[-37.51993,-11.54888],[-37.65174,-11.51832],[-37.67356,-11.56868],[-37.81234,-11.51607],[-37.86795,-11.42629],[-37.97796,-11.39359],[-38.01493,-11.2857],[-37.97414,-11.19536],[-38.06177,-11.17213],[-38.10176,-11.01509],[-38.22638,-10.91661],[-38.24508,-10.82301],[-38.19902,-10.71102],[-38.04712,-10.69354],[-37.98522,-10.75911],[-37.81397,-10.69109],[-37.84061,-10.5496],[-37.81309,-10.50852],[-37.85893,-10.43024],[-37.83902,-10.37925],[-37.73491,-10.33284],[-37.83325,-9.99838],[-37.9575,-9.97132],[-37.99769,-9.91675],[-37.9667,-9.81609],[-38.0316,-9.77107],[-38.04185,-9.69142],[-37.98791,-9.64743],[-38.0403,-9.57248],[-38.00332,-9.51507],[-37.89417,-9.53956],[-37.78512,-9.63866],[-37.7118,-9.63133],[-37.58173,-9.73412],[-37.41779,-9.76299],[-37.23558,-9.89607],[-37.153,-9.90089],[-37.04425,-9.98873],[-36.99492,-9.9754],[-36.91846,-10.13228],[-36.72043,-10.26498],[-36.62545,-10.25641],[-36.56594,-10.32955],[-36.56307,-10.41973],[-36.4569,-10.40738],[-36.39392,-10.50037]]]}},{"type":"Feature","id":"TO","properties":{"SIGLA":"TO","Estado":"Tocantins"},"geometry":{"type":"Polygon","coordinates":[[[-48.35616,-5.17187],[-48.17853,-5.26079],[-48.07925,-5.27508],[-48.00405,-5.23463],[-47.88556,-5.26071],[-47.84375,-5.37622],[-47.7443,-5.38015],[-47.55536,-5.46575],[-47.48325,-5.56491],[-47.49336,-5.73054],[-47.43105,-5.86922],[-47.4331,-6.11215],[-47.37722,-6.26678],[-47.49863,-6.67125],[-47.5283,-6.97342],[-47.64444,-7.14363],[-47.70472,-7.13587],[-47.75043,-7.18952],[-47.6477,-7.30414],[-47.54288,-7.2654],[-47.48532,-7.31765],[-47.49824,-7.37874],[-47.59092,-7.43953],[-47.50469,-7.43665],[-47.4752,-7.53338],[-47.42763,-7.52578],[-47.38189,-7.65407],[-47.3598,-7.61616],[-47.31419,-7.66064],[-47.31241,-7.73158],[-47.24621,-7.75853],[-47.24499,-7.81552],[-47.17453,-7.85701],[-47.04349,-8.0536],[-46.99741,-8.06724],[-46.87274,-7.95558],[-46.61474,-7.89682],[-46.47695,-8.01223],[-46.50758,-8.27079],[-46.54365,-8.31894],[-46.64129,-8.32152],[-46.80682,-8.39899],[-46.84601,-8.5336],[-46.91435,-8.58713],[-46.88693,-8.68651],[-46.93123,-8.7266],[-46.91343,-8.84784],[-46.99357,-8.88026],[-46.97027,-8.91451],[-47.06848,-9.01204],[-47.04346,-9.02657],[-47.06871,-9.06389],[-46.92332,-9.06568],[-46.93178,-9.12808],[-46.89354,-9.12232],[-46.82779,-9.19429],[-46.84704,-9.29244],[-46.76109,-9.41018],[-46.66596,-9.39194],[-46.63999,-9.45774],[-46.56096,-9.48405],[-46.53674,-9.55713],[-46.59237,-9.58722],[-46.64639,-9.73372],[-46.57507,-9.75859],[-46.57699,-9.8052],[-46.50943,-9.80139],[-46.4574,-10.0473],[-46.36787,-10.16883],[-46.21085,-10.16976],[-46.16568,-10.21316],[-46.02826,-10.17689],[-46.00403,-10.26168],[-45.87784,-10.23936],[-45.79373,-10.26769],[-45.73384,-10.22862],[-45.72662,-10.15688],[-45.69926,-10.16688],[-45.69751,-10.26359],[-45.74306,-10.34676],[-45.82737,-10.36826],[-45.83075,-10.43876],[-46.02501,-10.5428],[-46.06448,-10.60279],[-46.21095,-10.64969],[-46.19254,-10.71974],[-46.25141,-10.75709],[-46.22978,-10.7986],[-46.28392,-10.83208],[-46.2834,-10.90677],[-46.39892,-10.99441],[-46.47234,-11.1912],[-46.61707,-11.28947],[-46.47929,-11.51641],[-46.08665,-11.62216],[-46.15398,-11.657],[-46.31151,-11.62724],[-46.29404,-11.69278],[-46.37345,-11.75115],[-46.3227,-11.77076],[-46.37456,-11.86862],[-46.17231,-11.89792],[-46.32568,-11.95851],[-46.39784,-12.04036],[-46.37435,-12.28983],[-46.3234,-12.30356],[-46.35237,-12.33742],[-46.25638,-12.42434],[-46.25478,-12.49433],[-46.15386,-12.48338],[-46.16062,-12.52859],[-46.27992,-12.58492],[-46.26228,-12.83169],[-46.30476,-12.94989],[-46.11388,-12.91827],[-46.36425,-12.99138],[-46.36617,-12.86487],[-46.4176,-12.82348],[-46.45462,-12.97117],[-46.75058,-12.96917],[-46.97884,-13.13159],[-47.15381,-13.20991],[-47.22589,-13.19351],[-47.28212,-13.26492],[-47.37931,-13.23112],[-47.42566,-13.28975],[-47.47809,-13.18769],[-47.56304,-13.18424],[-47.56131,-13.12468],[-47.63441,-13.10321],[-47.66886,-13.20942],[-47.6227,-13.36768],[-47.67895,-13.46772],[-47.82396,-13.31188],[-47.96629,-13.31527],[-48.06242,-13.23541],[-48.08298,-13.28879],[-48.16537,-13.3058],[-48.14662,-13.15202],[-48.44167,-13.29238],[-48.4697,-13.14959],[-48.50873,-13.12892],[-48.58637,-13.31762],[-48.60137,-13.06089],[-48.73071,-12.9898],[-48.73654,-12.92103],[-48.85746,-12.80543],[-48.97559,-12.9572],[-49.0777,-12.90417],[-49.06748,-12.84734],[-49.12099,-12.7904],[-49.23737,-12.88408],[-49.35435,-13.10299],[-49.34029,-13.2539],[-49.36949,-13.27463],[-49.91143,-12.96677],[-50.29261,-12.83992],[-50.30038,-12.68226],[-50.19313,-12.56431],[-50.22935,-12.53391],[-50.21763,-12.4889],[-50.14263,-12.39593],[-50.3661,-12.54824],[-50.43252,-12.69],[-50.47802,-12.70995],[-50.51124,-12.86081],[-50.5978,-12.82678],[-50.58449,-12.79978],[-50.62268,-12.81974],[-50.64688,-12.65112],[-50.69964,-12.64778],[-50.7061,-12.61352],[-50.66793,-12.59601],[-50.68643,-12.56921],[-50.61821,-12.429],[-50.64366,-12.22297],[-50.68682,-12.20216],[-50.68219,-11.99089],[-50.63924,-11.88457],[-50.68626,-11.859],[-50.722,-11.7397],[-50.66366,-11.67753],[-50.65743,-11.59281],[-50.71594,-11.57896],[-50.74202,-11.46035],[-50.65599,-11.24024],[-50.65861,-11.12882],[-50.60941,-11.06745],[-50.62196,-10.83942],[-50.57054,-10.75251],[-50.60335,-10.66001],[-50.53738,-10.60522],[-50.47368,-10.40491],[-50.41824,-10.35629],[-50.39211,-10.13404],[-50.3026,-10.03542],[-50.10733,-9.59429],[-50.03775,-9.28941],[-49.90766,-9.17383],[-49.74505,-8.90617],[-49.59238,-8.8395],[-49.40986,-8.58001],[-49.37395,-8.45123],[-49.27621,-8.36369],[-49.17485,-8.05056],[-49.14841,-7.80732],[-49.32805,-7.6778],[-49.38393,-7.54617],[-49.18526,-7.23518],[-49.2095,-6.92543],[-49.01886,-6.78422],[-48.83219,-6.75404],[-48.68269,-6.67792],[-48.64532,-6.50702],[-48.51122,-6.35695],[-48.3813,-6.37564],[-48.43192,-6.17731],[-48.30384,-6.1169],[-48.28577,-6.07041],[-48.3347,-6.00429],[-48.22938,-5.93043],[-48.2908,-5.83944],[-48.2935,-5.75121],[-48.17358,-5.70996],[-48.13223,-5.61814],[-48.30069,-5.52302],[-48.38159,-5.396],[-48.59672,-5.42256],[-48.75675,-5.3529],[-48.67942,-5.30583],[-48.60611,-5.33651],[-48.52298,-5.19334],[-48.35616,-5.17187]]]}},{"type":"Feature","id":"PR","properties":{"SIGLA":"PR","Estado":"Paraná"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-48.02721,-25.22139],[-48.21187,-25.4642],[-48.31543,-25.48983],[-48.27738,-25.51454],[-48.29889,-25.568],[-48.43088,-25.64276],[-48.56548,-25.85879],[-48.59152,-25.97731],[-49.17543,-26.00055],[-49.30337,-26.11349],[-49.46141,-26.16279],[-49.55507,-26.23729],[-49.72664,-26.17105],[-49.83142,-26.05342],[-49.94183,-26.00845],[-49.97277,-26.05829],[-49.9758,-26.01414],[-50.01142,-26.01382],[-50.00665,-26.04589],[-50.08481,-26.0235],[-50.12068,-26.06253],[-50.17185,-26.02512],[-50.18112,-26.07951],[-50.25151,-26.03019],[-50.33206,-26.08366],[-50.32359,-26.1349],[-50.44427,-26.03479],[-50.54763,-26.04014],[-50.57071,-26.00244],[-50.59214,-26.04253],[-50.55165,-26.05372],[-50.63148,-26.06511],[-50.73179,-26.20337],[-50.71849,-26.24474],[-50.79329,-26.22769],[-50.83936,-26.2715],[-50.87109,-26.24216],[-50.90196,-26.28943],[-50.93542,-26.23356],[-50.94468,-26.28247],[-51.00046,-26.23044],[-51.07935,-26.22791],[-51.10501,-26.27535],[-51.24307,-26.32467],[-51.30098,-26.41836],[-51.25397,-26.46346],[-51.27428,-26.49705],[-51.21596,-26.57008],[-51.22986,-26.61626],[-51.39161,-26.66373],[-51.41152,-26.71724],[-51.50607,-26.58201],[-51.6667,-26.56423],[-51.87394,-26.59984],[-52.01,-26.56642],[-52.18565,-26.44537],[-52.60265,-26.41542],[-52.73775,-26.34213],[-52.98885,-26.34625],[-53.09067,-26.39047],[-53.28329,-26.24592],[-53.55054,-26.29331],[-53.64046,-26.253],[-53.65159,-26.19514],[-53.74196,-26.11878],[-53.73429,-26.04444],[-53.83651,-25.97242],[-53.82302,-25.79415],[-53.89095,-25.62366],[-53.94846,-25.61234],[-53.9561,-25.64747],[-54.01074,-25.56804],[-54.07767,-25.55906],[-54.101,-25.61892],[-54.10951,-25.49623],[-54.20635,-25.54055],[-54.17651,-25.58465],[-54.23116,-25.56339],[-54.25062,-25.59773],[-54.2876,-25.55634],[-54.38652,-25.59925],[-54.43374,-25.69606],[-54.49602,-25.6163],[-54.59422,-25.59283],[-54.6198,-25.45457],[-54.4308,-25.15994],[-54.46185,-25.09525],[-54.44243,-24.94962],[-54.3259,-24.66315],[-54.32919,-24.4714],[-54.25964,-24.3654],[-54.34157,-24.13138],[-54.10173,-23.95194],[-53.9815,-23.45684],[-53.7418,-23.33612],[-53.63655,-23.12501],[-53.60738,-22.95131],[-52.9729,-22.57096],[-52.70873,-22.62854],[-52.5854,-22.56641],[-52.50315,-22.63504],[-52.25089,-22.61591],[-52.22305,-22.67602],[-52.15715,-22.6452],[-52.14923,-22.54974],[-52.10968,-22.51678],[-51.94098,-22.56783],[-51.8691,-22.63093],[-51.73618,-22.62478],[-51.71864,-22.6695],[-51.62835,-22.65872],[-51.56004,-22.69766],[-51.44688,-22.65344],[-51.26567,-22.66818],[-51.15005,-22.75554],[-50.88919,-22.79702],[-50.79342,-22.89484],[-50.80446,-22.94622],[-50.73684,-22.96384],[-50.66205,-22.89596],[-50.50655,-22.94586],[-50.37954,-22.90589],[-50.33663,-22.95088],[-50.238,-22.95463],[-49.97568,-22.90498],[-49.9698,-22.96497],[-49.90005,-23.00249],[-49.91207,-23.05161],[-49.7407,-23.09966],[-49.67878,-23.16555],[-49.62693,-23.28341],[-49.63756,-23.35211],[-49.59087,-23.38328],[-49.62148,-23.39791],[-49.56709,-23.42691],[-49.62984,-23.51232],[-49.60067,-23.53611],[-49.61761,-23.64008],[-49.54967,-23.70451],[-49.56136,-23.8164],[-49.6112,-23.85218],[-49.51207,-23.92639],[-49.43048,-24.08903],[-49.33703,-24.13807],[-49.35643,-24.215],[-49.28428,-24.30855],[-49.20168,-24.34319],[-49.31646,-24.556],[-49.30587,-24.67414],[-49.06487,-24.68652],[-49.00515,-24.63373],[-48.95892,-24.68577],[-48.82778,-24.65495],[-48.77912,-24.69722],[-48.67919,-24.67138],[-48.66095,-24.70725],[-48.5825,-24.67119],[-48.49977,-24.73946],[-48.56494,-24.83477],[-48.58313,-25.05214],[-48.51347,-25.09114],[-48.4115,-24.98021],[-48.32383,-25.05435],[-48.24961,-24.97788],[-48.15606,-25.14425],[-48.18621,-25.20728],[-48.11427,-25.25086],[-48.02721,-25.22139]]],[[[-49.52971,-26.22379],[-49.5294,-26.22452],[-49.52988,-26.22362],[-49.52971,-26.22379]]]]}},{"type":"Feature","id":"SP","properties":{"SIGLA":"SP","Estado":"São Paulo"},"geometry":{"type":"MultiPolygon","coordinates":[[[[-50.46993,-19.78036],[-50.34612,-19.86805],[-50.10149,-19.87549],[-50.01574,-19.92685],[-49.85603,-19.94731],[-49.55194,-19.90633],[-49.44497,-19.981],[-49.26116,-19.96395],[-49.30899,-20.10438],[-49.24566,-20.28263],[-49.17302,-20.31303],[-49.0675,-20.15488],[-49.00088,-20.16124],[-48.97081,-20.39028],[-48.89866,-20.44236],[-48.86702,-20.39885],[-48.88507,-20.26644],[-48.8262,-20.16245],[-48.64258,-20.16699],[-48.40623,-20.114],[-48.25514,-20.14431],[-48.21771,-20.12528],[-48.23833,-20.02928],[-48.10594,-20.14685],[-48.02061,-20.12452],[-47.97642,-20.03587],[-47.89591,-20.12575],[-47.85389,-19.99041],[-47.70885,-19.9792],[-47.63521,-20.04974],[-47.46874,-19.96192],[-47.43916,-20.05158],[-47.25749,-20.1667],[-47.23152,-20.21965],[-47.29866,-20.34893],[-47.29321,-20.44823],[-47.15337,-20.52123],[-47.09734,-20.64502],[-47.11527,-20.70719],[-47.18671,-20.73163],[-47.24049,-20.88578],[-47.14826,-20.97299],[-47.11924,-21.18625],[-47.06229,-21.21035],[-47.07774,-21.24072],[-46.99821,-21.3508],[-47.01196,-21.42277],[-46.92123,-21.42763],[-46.76515,-21.36056],[-46.70641,-21.40534],[-46.66598,-21.36165],[-46.61571,-21.44196],[-46.5097,-21.4704],[-46.51861,-21.61302],[-46.56585,-21.6793],[-46.62204,-21.67604],[-46.62655,-21.76648],[-46.69142,-21.83746],[-46.61287,-22.00685],[-46.72387,-22.07739],[-46.59898,-22.13688],[-46.67429,-22.17611],[-46.72371,-22.30712],[-46.66137,-22.36654],[-46.66229,-22.41931],[-46.4074,-22.54043],[-46.43418,-22.57507],[-46.39355,-22.66333],[-46.47934,-22.69877],[-46.33499,-22.7608],[-46.38464,-22.87046],[-46.35676,-22.9008],[-46.14475,-22.85834],[-46.13954,-22.92325],[-46.0084,-22.89],[-45.91228,-22.81679],[-45.88921,-22.87647],[-45.84089,-22.83327],[-45.78962,-22.85862],[-45.75518,-22.79343],[-45.71356,-22.81475],[-45.71386,-22.75745],[-45.72813,-22.72365],[-45.81797,-22.71134],[-45.69397,-22.65168],[-45.73741,-22.60851],[-45.71688,-22.57835],[-45.66081,-22.57902],[-45.66663,-22.6517],[-45.57697,-22.6022],[-45.58482,-22.65219],[-45.54036,-22.65308],[-45.4743,-22.58939],[-45.39832,-22.65392],[-45.27469,-22.61674],[-45.14211,-22.50412],[-44.80759,-22.40544],[-44.73394,-22.4341],[-44.64069,-22.60856],[-44.55089,-22.60501],[-44.50903,-22.64145],[-44.38414,-22.57391],[-44.35647,-22.6154],[-44.34425,-22.58873],[-44.20795,-22.6135],[-44.16181,-22.67876],[-44.25755,-22.76679],[-44.27343,-22.83246],[-44.46621,-22.88549],[-44.49472,-22.84643],[-44.79247,-22.98213],[-44.82308,-23.15836],[-44.88977,-23.2271],[-44.72626,-23.37088],[-44.84409,-23.38921],[-44.90851,-23.3347],[-44.97662,-23.40382],[-45.05985,-23.42309],[-45.05956,-23.45794],[-45.01487,-23.46424],[-45.06257,-23.47225],[-45.08141,-23.52389],[-45.16749,-23.49469],[-45.16032,-23.54379],[-45.19365,-23.52166],[-45.22585,-23.53861],[-45.20975,-23.58397],[-45.2977,-23.5722],[-45.40883,-23.62546],[-45.43122,-23.68272],[-45.39657,-23.80939],[-45.42376,-23.82996],[-45.51471,-23.84224],[-45.56028,-23.7926],[-45.7664,-23.76273],[-45.92329,-23.77053],[-46.12305,-23.83888],[-46.18028,-23.92615],[-46.16132,-23.94575],[-46.18562,-23.96522],[-46.18703,-23.98549],[-46.18274,-23.99154],[-46.2491,-23.99382],[-46.28113,-24.02214],[-46.28338,-24.0437],[-46.29524,-24.01719],[-46.32483,-24.02288],[-46.31746,-23.97981],[-46.37909,-23.96985],[-46.40031,-24.03383],[-46.50639,-24.04364],[-46.82388,-24.20643],[-46.99482,-24.32535],[-47.0086,-24.41392],[-47.79444,-24.9321],[-47.9104,-25.05388],[-47.90932,-25.168],[-48.09765,-25.31011],[-48.02721,-25.22139],[-48.11427,-25.25086],[-48.18621,-25.20728],[-48.15606,-25.14425],[-48.24961,-24.97788],[-48.32383,-25.05435],[-48.4115,-24.98021],[-48.51347,-25.09114],[-48.58313,-25.05214],[-48.56494,-24.83477],[-48.49977,-24.73946],[-48.5825,-24.67119],[-48.66095,-24.70725],[-48.67919,-24.67138],[-48.77912,-24.69722],[-48.82778,-24.65495],[-48.95892,-24.68577],[-49.00515,-24.63373],[-49.06487,-24.68652],[-49.30587,-24.67414],[-49.31646,-24.556],[-49.20168,-24.34319],[-49.28428,-24.30855],[-49.35643,-24.215],[-49.33703,-24.13807],[-49.43048,-24.08903],[-49.51207,-23.92639],[-49.6112,-23.85218],[-49.56136,-23.8164],[-49.54967,-23.70451],[-49.61761,-23.64008],[-49.60067,-23.53611],[-49.62984,-23.51232],[-49.56709,-23.42691],[-49.62148,-23.39791],[-49.59087,-23.38328],[-49.63756,-23.35211],[-49.62693,-23.28341],[-49.67878,-23.16555],[-49.7407,-23.09966],[-49.91207,-23.05161],[-49.90005,-23.00249],[-49.9698,-22.96497],[-49.98663,-22.89799],[-49.99534,-22.93021],[-50.04832,-22.90245],[-50.238,-22.95463],[-50.33663,-22.95088],[-50.37954,-22.90589],[-50.50655,-22.94586],[-50.66205,-22.89596],[-50.73684,-22.96384],[-50.80446,-22.94622],[-50.79342,-22.89484],[-50.88919,-22.79702],[-51.15005,-22.75554],[-51.26567,-22.66818],[-51.44688,-22.65344],[-51.56004,-22.69766],[-51.62835,-22.65872],[-51.71864,-22.6695],[-51.73618,-22.62478],[-51.8691,-22.63093],[-51.94098,-22.56783],[-52.10968,-22.51678],[-52.14923,-22.54974],[-52.15715,-22.6452],[-52.22305,-22.67602],[-52.25089,-22.61591],[-52.50315,-22.63504],[-52.5854,-22.56641],[-52.70217,-22.62815],[-52.92522,-22.5661],[-53.08826,-22.65893],[-53.10558,-22.62249],[-52.99865,-22.49342],[-52.48819,-22.21769],[-52.37718,-22.10666],[-52.30149,-21.94223],[-52.05373,-21.67256],[-52.09592,-21.54205],[-52.06692,-21.50507],[-51.96837,-21.50225],[-51.86573,-21.34943],[-51.87589,-21.13639],[-51.78963,-21.1021],[-51.72278,-20.97733],[-51.62354,-20.94442],[-51.63486,-20.75484],[-51.5941,-20.64383],[-51.35082,-20.36317],[-51.11804,-20.28743],[-50.96606,-20.03398],[-50.66178,-19.90949],[-50.57496,-19.81498],[-50.46993,-19.78036]]],[[[-44.32371,-22.84938],[-44.32221,-22.84832],[-44.32377,-22.84954],[-44.32371,-22.84938]]],[[[-44.90322,-23.3499],[-44.90695,-23.35554],[-44.90807,-23.35307],[-44.90322,-23.3499]]],[[[-44.90312,-23.37797],[-44.90758,-23.37972],[-44.90746,-23.37805],[-44.90312,-23.37797]]],[[[-44.88921,-23.38213],[-44.88844,-23.38635],[-44.89192,-23.38634],[-44.88921,-23.38213]]],[[[-44.94396,-23.38576],[-44.94747,-23.38915],[-44.9484,-23.38676],[-44.94396,-23.38576]]],[[[-44.93827,-23.3883],[-44.93787,-23.38938],[-44.93913,-23.38941],[-44.93827,-23.3883]]],[[[-44.85261,-23.3987],[-44.85459,-23.40205],[-44.85535,-23.40004],[-44.85261,-23.3987]]],[[[-44.8523,-23.40499],[-44.85196,-23.40648],[-44.85379,-23.40613],[-44.8523,-23.40499]]],[[[-44.85626,-23.41601],[-44.85612,-23.4275],[-44.86237,-23.42638],[-44.85626,-23.41601]]],[[[-44.90679,-23.42308],[-44.9056,-23.42779],[-44.90829,-23.4278],[-44.90679,-23.42308]]],[[[-44.86448,-23.4232],[-44.86375,-23.42485],[-44.86545,-23.4241],[-44.86448,-23.4232]]],[[[-45.04167,-23.51671],[-45.04241,-23.52095],[-45.0434,-23.51804],[-45.04167,-23.51671]]],[[[-45.08009,-23.56502],[-45.06018,-23.52713],[-45.04026,-23.53537],[-45.08009,-23.56502]]],[[[-45.21284,-23.53229],[-45.2121,-23.5339],[-45.21399,-23.53311],[-45.21284,-23.53229]]],[[[-45.21769,-23.5497],[-45.21826,-23.55134],[-45.21862,-23.55079],[-45.21769,-23.5497]]],[[[-45.02443,-23.54527],[-45.02557,-23.54629],[-45.02495,-23.54491],[-45.02443,-23.54527]]],[[[-45.02937,-23.54761],[-45.03004,-23.54902],[-45.03114,-23.54898],[-45.02937,-23.54761]]],[[[-45.02573,-23.55015],[-45.02664,-23.55121],[-45.02709,-23.55064],[-45.02573,-23.55015]]],[[[-45.15721,-23.56028],[-45.15851,-23.57382],[-45.16435,-23.56769],[-45.15721,-23.56028]]],[[[-45.31231,-23.58229],[-45.31231,-23.58467],[-45.31434,-23.58384],[-45.31231,-23.58229]]],[[[-45.30989,-23.58662],[-45.31035,-23.58924],[-45.31189,-23.58746],[-45.30989,-23.58662]]],[[[-45.29323,-23.59425],[-45.28378,-23.60241],[-45.29642,-23.60297],[-45.29323,-23.59425]]],[[[-45.3302,-23.72288],[-45.23048,-23.77739],[-45.24915,-23.82546],[-45.22871,-23.83917],[-45.27321,-23.83929],[-45.29072,-23.86683],[-45.27274,-23.89253],[-45.2251,-23.90372],[-45.2515,-23.96657],[-45.29119,-23.91102],[-45.34864,-23.91764],[-45.42787,-23.94196],[-45.46087,-23.91465],[-45.36738,-23.81173],[-45.3302,-23.72288]]],[[[-45.02445,-23.7419],[-45.02587,-23.74291],[-45.02625,-23.74175],[-45.02445,-23.7419]]],[[[-45.02609,-23.75278],[-45.00898,-23.7576],[-45.01637,-23.7666],[-45.02609,-23.75278]]],[[[-45.79945,-23.77353],[-45.79959,-23.77438],[-45.80127,-23.77386],[-45.79945,-23.77353]]],[[[-45.70539,-23.78778],[-45.70635,-23.79071],[-45.7085,-23.79009],[-45.70539,-23.78778]]],[[[-45.71369,-23.7892],[-45.71485,-23.7949],[-45.71828,-23.79454],[-45.71369,-23.7892]]],[[[-45.72014,-23.79944],[-45.71629,-23.80725],[-45.72666,-23.80341],[-45.72014,-23.79944]]],[[[-45.66792,-23.80563],[-45.67185,-23.80821],[-45.66822,-23.80451],[-45.66792,-23.80563]]],[[[-45.15724,-23.81413],[-45.11603,-23.80739],[-45.11797,-23.81667],[-45.15724,-23.81413]]],[[[-46.02535,-23.82664],[-46.02708,-23.82773],[-46.02644,-23.82636],[-46.02535,-23.82664]]],[[[-45.22835,-23.81421],[-45.22824,-23.81709],[-45.2308,-23.81551],[-45.22835,-23.81421]]],[[[-45.39322,-23.82854],[-45.3939,-23.83016],[-45.3942,-23.82931],[-45.39322,-23.82854]]],[[[-45.41065,-23.82986],[-45.41198,-23.83137],[-45.41231,-23.83002],[-45.41065,-23.82986]]],[[[-45.44634,-23.835],[-45.44826,-23.8379],[-45.44906,-23.83569],[-45.44634,-23.835]]],[[[-45.15493,-23.82817],[-45.15429,-23.83751],[-45.16063,-23.83174],[-45.15493,-23.82817]]],[[[-45.52084,-23.84829],[-45.52704,-23.85046],[-45.52365,-23.84667],[-45.52084,-23.84829]]],[[[-46.13074,-23.86987],[-46.1356,-23.87028],[-46.13149,-23.86727],[-46.13074,-23.86987]]],[[[-45.78671,-23.86006],[-45.77873,-23.86868],[-45.78758,-23.86947],[-45.78671,-23.86006]]],[[[-46.12459,-23.87867],[-46.1224,-23.88306],[-46.12821,-23.88141],[-46.12459,-23.87867]]],[[[-45.27933,-23.85234],[-45.28041,-23.85507],[-45.28146,-23.85369],[-45.27933,-23.85234]]],[[[-46.13036,-23.89319],[-46.13677,-23.89287],[-46.12965,-23.88971],[-46.13036,-23.89319]]],[[[-46.16843,-23.91642],[-46.16781,-23.91529],[-46.16705,-23.91616],[-46.16843,-23.91642]]],[[[-45.26183,-23.88849],[-45.26217,-23.89053],[-45.26356,-23.88881],[-45.26183,-23.88849]]],[[[-46.17022,-23.92778],[-46.17072,-23.9283],[-46.17088,-23.92808],[-46.17022,-23.92778]]],[[[-45.32428,-23.91749],[-45.327,-23.9187],[-45.3269,-23.91783],[-45.32428,-23.91749]]],[[[-45.29706,-23.91798],[-45.29703,-23.92072],[-45.29848,-23.91921],[-45.29706,-23.91798]]],[[[-46.16827,-23.96706],[-46.16638,-23.9678],[-46.16835,-23.96829],[-46.16827,-23.96706]]],[[[-46.17663,-23.97071],[-46.17694,-23.97039],[-46.17631,-23.97039],[-46.17663,-23.97071]]],[[[-46.18399,-23.97275],[-46.18197,-23.97533],[-46.18464,-23.97537],[-46.18399,-23.97275]]],[[[-46.25543,-24.00253],[-46.25459,-24.00209],[-46.25398,-24.00308],[-46.25543,-24.00253]]],[[[-46.32574,-24.00898],[-46.32393,-24.00909],[-46.32591,-24.00989],[-46.32574,-24.00898]]],[[[-46.29245,-24.02561],[-46.29421,-24.02605],[-46.29256,-24.02516],[-46.29245,-24.02561]]],[[[-46.29513,-24.02828],[-46.29472,-24.02881],[-46.29545,-24.02904],[-46.29513,-24.02828]]],[[[-46.97375,-24.36878],[-46.98066,-24.37078],[-46.97536,-24.36671],[-46.97375,-24.36878]]],[[[-46.90466,-24.37645],[-46.91157,-24.37845],[-46.90628,-24.37438],[-46.90466,-24.37645]]],[[[-46.90291,-24.38908],[-46.90718,-24.39542],[-46.90624,-24.38759],[-46.90291,-24.38908]]],[[[-48.00059,-25.24882],[-48.00448,-25.24883],[-48.00285,-25.24672],[-48.00059,-25.24882]]]]}}]};
    const mapEl = document.getElementById('mapBR');
    const svgNS = 'http://www.w3.org/2000/svg';
    function volumeColor(v,max){ const t=Math.sqrt(Math.max(0,v)/Math.max(1,max)); const c=Math.round(225-165*t); return 'rgb('+c+','+c+','+c+')'; }
    function updateMap(ua){
      try {
        // SVG nativo: funciona no displayHTML do Databricks sem Leaflet, CDN ou tiles.
        const W=650,H=660,pad=22;
        const lonMin=-74.1,lonMax=-34.5,latMin=-33.9,latMax=5.4;
        const scale=Math.min((W-2*pad)/(lonMax-lonMin),(H-2*pad)/(latMax-latMin));
        const ox=(W-(lonMax-lonMin)*scale)/2, oy=(H-(latMax-latMin)*scale)/2;
        const xy=pt=>[(ox+(pt[0]-lonMin)*scale).toFixed(2),(oy+(latMax-pt[1])*scale).toFixed(2)];
        const rings=geom=>geom.type==='Polygon'?[geom.coordinates]:geom.coordinates;
        const pathFor=geom=>rings(geom).map(poly=>poly.map(ring=>ring.map((pt,i)=>{const [x,y]=xy(pt);return (i?'L':'M')+x+','+y;}).join(' ')+' Z').join(' ')).join(' ');
        const max=Math.max(1,...Object.values(ua).map(Number));
        const esc=t=>String(t).replace(/[&<>"']/g,ch=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[ch]));
        let shapes='', labels='';
        brGeo.features.forEach(f=>{
          const uf=getSigla(f), v=Number(ua[uf]||0), name=UN[uf]||f.properties.Estado||uf;
          shapes+='<path d="'+pathFor(f.geometry)+'" fill="'+(v>0?volumeColor(v,max):'#d5d8dc')+'" stroke="#333" stroke-width="0.75" fill-rule="evenodd"><title>'+esc(name+' ('+uf+'): '+ff(v)+' m³')+'</title></path>';
        });
        Object.entries(UL).forEach(([uf,latlng])=>{
          const [x,y]=xy([latlng[1],latlng[0]]),v=Number(ua[uf]||0);
          labels+='<g><rect x="'+(x-20)+'" y="'+(y-11)+'" width="40" height="22" rx="3" fill="#111" fill-opacity=".88" stroke="#ddd" stroke-width=".35"/><text x="'+x+'" y="'+(Number(y)-2)+'" text-anchor="middle" font-size="9" font-weight="bold" fill="white">'+uf+'</text><text x="'+x+'" y="'+(Number(y)+7)+'" text-anchor="middle" font-size="7" fill="white">'+esc(ff(v))+'</text></g>';
        });
        mapEl.innerHTML='<svg xmlns="'+svgNS+'" viewBox="0 0 '+W+' '+H+'" role="img" aria-label="Mapa do Brasil com volume faturado por estado" style="width:100%;height:auto;display:block;max-height:660px">'+shapes+labels+'</svg>';
        if(_ms) _ms.style.display='none';
      }catch(e){if(_ms){_ms.style.display='block';_ms.textContent='Erro ao desenhar mapa: '+e.message;}console.error('Mapa Brasil',e);}
    }

    function go(){
      const f = gf();
      let total = 0, mi = 0, me = 0;
      DV.forEach(r => {
        if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){
          if(r[3] === 'MI'){ mi += r[4]; } else { me += r[4]; }
        }
      });
      total = mi + me;
      document.getElementById('k1').textContent = fv(total) + ' m³';
      document.getElementById('k2').textContent = fv(mi) + ' m³';
      document.getElementById('k3').textContent = fv(me) + ' m³';

      const pmi = total > 0 ? (mi / total * 100).toFixed(1) : 0;
      const pme = total > 0 ? (me / total * 100).toFixed(1) : 0;
      document.getElementById('cC').innerHTML = '<div class="pc"><svg width="160" height="160" viewBox="0 0 36 36"><circle cx="18" cy="18" r="15.9" fill="none" stroke="#fff" stroke-width="3" stroke-dasharray="' + pmi + ' ' + (100 - pmi) + '" stroke-dashoffset="25"/><circle cx="18" cy="18" r="15.9" fill="none" stroke="#666" stroke-width="3" stroke-dasharray="' + pme + ' ' + (100 - pme) + '" stroke-dashoffset="' + (25 - pmi) + '"/></svg><div class="pl"><div class="pi"><div class="pp" style="background:#fff"></div>MI: ' + ff(mi) + ' (' + pmi + '%)</div><div class="pi"><div class="pp" style="background:#666"></div>ME: ' + ff(me) + ' (' + pme + '%)</div></div></div>';

      const incMap = {};
      DI.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){ incMap[r[3]] = (incMap[r[3]] || 0) + r[4]; } });
      bar('cI', Object.entries(incMap).sort((a,b) => b[1] - a[1]));

      const ufMap = {};
      DU.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){ ufMap[r[3]] = (ufMap[r[3]] || 0) + r[4]; } });
      const ufSorted = Object.entries(ufMap)
        .sort((a,b) => b[1] - a[1])
        .map(([uf, value]) => [UN[uf] || uf, value]);
      bar('cU', ufSorted.slice(0, 10));
      updateMap(ufMap);

      const clientVol = {};
      DC.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){ clientVol[r[3]] = (clientVol[r[3]] || 0) + r[4]; } });
      const clientTop = Object.entries(clientVol).sort((a,b) => b[1] - a[1]).slice(0, 10);
      let th = '<div class="tc"><table><thead><tr><th>#</th><th>Cliente</th><th style="text-align:right">Volume (m³)</th></tr></thead><tbody>';
      clientTop.forEach((r, i) => { th += '<tr><td>' + (i + 1) + '</td><td>' + r[0] + '</td><td style="text-align:right">' + ff(r[1]) + '</td></tr>'; });
      th += '</tbody></table></div>';
      document.getElementById('cK').innerHTML = th;

      const linMap = {};
      DL.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){ linMap[r[3]] = (linMap[r[3]] || 0) + r[4]; } });
      bar('cL', Object.entries(linMap).sort((a,b) => b[1] - a[1]).slice(0, 12));

      const monthVolMap = {};
      MO.forEach(m => monthVolMap[m] = 0);
      DV.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1])){ monthVolMap[r[2]] = (monthVolMap[r[2]] || 0) + r[4]; } });
      const monthLine = MO.map(m => ({label:m, value:monthVolMap[m] || 0}));
      line('cE', monthLine, {
        valueFormatter: ff,
        stroke: '#cfd6de',
        fill: 'rgba(207,214,222,.08)',
        showValues: true,
        height: 360,
        headerLabel: 'Volume consolidado',
        headerValue: ff(monthLine.reduce((acc, item) => acc + item.value, 0)),
        headerSub: `Distribuição mensal do volume faturado em ${f.a.join('/')}`
      });

      const monthRecMap = {};
      MO.forEach(m => monthRecMap[m] = 0);
      DRL.forEach(r => { if(f.c.includes(r[0]) && f.a.includes(r[1])){ monthRecMap[r[2]] = (monthRecMap[r[2]] || 0) + r[3]; } });
      const recLine = MO.map(m => ({label:m, value:monthRecMap[m] || 0}));
      line('cR', recLine, {
        valueFormatter: fm,
        stroke: '#5b9bd5',
        fill: 'rgba(91,155,213,.10)',
        showValues: true,
        height: 360,
        headerLabel: 'Receita líquida consolidada',
        headerValue: fm(recLine.reduce((acc, item) => acc + item.value, 0)),
        headerSub: `Somatória total da receita líquida por mês em ${f.a.join('/')}`
      });

      const cmpColorByYear = {'2025':'#5b9bd5','2026':'#ffffff'};
      const cmpYears=[...DATA.anos].sort();
      const cmpSeries=cmpYears.map((yr,idx)=>{
        const ym={};MO.forEach(m=>ym[m]=0);
        DV.forEach(r=>{if(f.c.includes(r[0])&&r[1]===yr){ym[r[2]]=(ym[r[2]]||0)+r[4];}});
        return {name:'Ano '+yr,color:cmpColorByYear[yr] || '#d0d7df',values:f.m.map(m=>({label:m,value:ym[m]||0}))};
      });
      if(cmpSeries.some(s=>s.values.some(v=>v.value>0))){
        multiLine('cY',cmpSeries,{height:360});
      } else {
        document.getElementById('cY').innerHTML = emptyState('Sem dados comparativos para os filtros aplicados');
      }

      const dayMap = {};
      DFD.forEach(r => {
        if(f.c.includes(r[0]) && f.a.includes(r[1]) && f.m.includes(r[2])){
          dayMap[r[3]] = (dayMap[r[3]] || 0) + r[4];
        }
      });
      const daySeries = Object.entries(dayMap)
        .sort((a,b) => a[0].localeCompare(b[0]))
        .map(([d, v]) => ({key:d, label:fd(d), value:v}));
      const dayWindow = daySeries.slice(-5);
      if(dayWindow.length){
        const latestDay = dayWindow[dayWindow.length - 1];
        line('cD', dayWindow, {
          valueFormatter: ff,
          stroke: '#ffffff',
          fill: 'rgba(255,255,255,.06)',
          headerLabel: 'Faturamento',
          headerValue: ff(latestDay.value),
          headerSub: `Data de referência: ${latestDay.label}`,
          showValues: true,
          height: 340
        });
      } else {
        document.getElementById('cD').innerHTML = emptyState('Sem faturamento diário para os filtros aplicados');
      }

      renderAnalytics(f);
    }

    go();
  </script>
</body>
</html>
""".replace("__PAYLOAD__", payload)

displayHTML(html)
